
# Amazon ML Challenge 2026 — 10K Blocking Laboratory

## Purpose

This notebook is **not** the final submission pipeline.

It is a controlled laboratory for one question:

> **Can we design a candidate-generation system that keeps nearly all true matches while reducing each Source 1 record to a tiny candidate set?**

Only after the blocking design is proven here will we implement the same key logic against the full ~12M Source 2 + Source 3 corpus.

### Design principle

```text
NO:
S1 × S2/S3 pairwise similarity

YES:
S1
 ↓
blocking keys
 ↓
inverted postings
 ↓
candidate union
 ↓
evidence aggregation
 ↓
cheap top-K pruning
 ↓
optional expensive matcher
```

The competition's `candidate_pairs.tsv` is the final candidate set fed into the matching model, and every final match must be contained in that set. fileciteturn1file0

The official evaluation is macro-averaged F0.5 over Source 1 entities, including singletons. fileciteturn2file0



# What this experiment will prove

We will measure four things separately:

### 1. Candidate recall
Did the blocker retrieve every true linked S2/S3 record?

### 2. Candidate reduction
How many candidates does each S1 record receive?

### 3. Runtime
How long does retrieval take?

### 4. Scaling behavior
Does runtime/candidate volume grow approximately with the number of postings and retrieved candidates rather than with `S1 × S2/S3`?

### Blocking families we will test

1. Exact native normalized name
2. Exact transliterated/ASCII name
3. Rare name-token blocks
4. Rare address-token blocks
5. Postal-code blocks
6. Rare character n-gram blocks
8. Prefix fallback
9. A combined adaptive blocker

The combined blocker is the candidate we expect to carry forward.


In [1]:

# 1. INSTALL / IMPORTS
%pip install -q rapidfuzz Unidecode pyarrow

import os
import re
import math
import time
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from rapidfuzz import fuzz
from unidecode import unidecode

print("Python:", os.sys.version)
print("pandas:", pd.__version__)


Note: you may need to restart the kernel to use updated packages.
Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
pandas: 2.2.3


In [2]:
# 2. PATHS
#
# Expected structure:
#
# project/
# ├── this_notebook.ipynb
# └── student_resource/
#     └── dataset/
#         ├── train/
#         └── test/
#
# We auto-detect student_resource from the current working directory
# or its parent so launching Jupyter from either level works.

CWD = Path.cwd()

if (CWD / "student_resource" / "dataset").exists():
    PROJECT_DIR = CWD
elif (CWD.parent / "student_resource" / "dataset").exists():
    PROJECT_DIR = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate student_resource/dataset from the current "
        f"working directory: {CWD}"
    )

RESOURCE_DIR = PROJECT_DIR / "student_resource"

TRAIN_DIR = RESOURCE_DIR / "dataset" / "train"
TEST_DIR = RESOURCE_DIR / "dataset" / "test"

TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
TRAIN_GT = TRAIN_DIR / "train_ground_truth.tsv"

WORK_DIR = PROJECT_DIR / "blocking_lab_10k"
WORK_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_DIR = WORK_DIR / "benchmark"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
S1_SAMPLE_N = 10_000

DISTRACTOR_N_PER_SOURCE = 10_000

MAX_NAME_POSTING = 150
MAX_ADDR_POSTING = 200
MAX_QGRAM_POSTING = 250
MAX_PREFIX_POSTING = 300

RARE_NAME_KEYS = 2
RARE_ADDR_KEYS = 2
RARE_QGRAM_KEYS = 3

K_VALUES = [5, 10, 20, 30, 50]

print("PROJECT_DIR :", PROJECT_DIR)
print("RESOURCE_DIR:", RESOURCE_DIR)
print("TRAIN_DIR   :", TRAIN_DIR)
print("WORK_DIR    :", WORK_DIR)


PROJECT_DIR : /Users/atharv/Documents/AMAZON_ML
RESOURCE_DIR: /Users/atharv/Documents/AMAZON_ML/student_resource
TRAIN_DIR   : /Users/atharv/Documents/AMAZON_ML/student_resource/dataset/train
WORK_DIR    : /Users/atharv/Documents/AMAZON_ML/blocking_lab_10k


In [3]:

# 3. VERIFY FILES

required = {
    "TRAIN_S1": TRAIN_S1,
    "TRAIN_S2": TRAIN_S2,
    "TRAIN_S3": TRAIN_S3,
    "TRAIN_GT": TRAIN_GT,
}

for name, path in required.items():
    print(f"{name:10s} exists={path.exists()}  path={path}")

if not all(p.exists() for p in required.values()):
    raise FileNotFoundError(
        "One or more training files were not found. "
        "Check RESOURCE_DIR / student_resource / dataset."
    )

print("All training files found.")


TRAIN_S1   exists=True  path=/Users/atharv/Documents/AMAZON_ML/student_resource/dataset/train/train_source1.tsv
TRAIN_S2   exists=True  path=/Users/atharv/Documents/AMAZON_ML/student_resource/dataset/train/train_source2.tsv
TRAIN_S3   exists=True  path=/Users/atharv/Documents/AMAZON_ML/student_resource/dataset/train/train_source3.tsv
TRAIN_GT   exists=True  path=/Users/atharv/Documents/AMAZON_ML/student_resource/dataset/train/train_ground_truth.tsv
All training files found.



# 4. Robust normalization

We keep two name representations:

```text
native_normalized
ASCII/transliterated_normalized
```

This is deliberate.

For example, a business may appear in a regional script in one record and Romanized form in another. We do not collapse the two retrieval paths into one representation.

We also keep addresses in native and ASCII forms.


In [4]:

# 4. NORMALIZATION

_ws_re = re.compile(r"\s+")
_nonword_re = re.compile(r"[^\w\s]+", flags=re.UNICODE)
_token_re = re.compile(r"\w+", flags=re.UNICODE)
_ascii_alnum_re = re.compile(r"[^a-z0-9]+")

GENERIC_NAME_TOKENS = {
    "the", "and", "co", "company", "corp", "corporation", "inc", "incorporated",
    "llc", "ltd", "limited", "plc", "pvt", "private", "enterprises", "enterprise",
    "group", "holdings", "services", "service", "solutions", "international",
    "india", "ind", "usa", "us", "store", "shop", "mart", "market", "business",
    "trading", "traders", "works", "industries", "industry", "center", "centre"
}

GENERIC_ADDR_TOKENS = {
    "road", "rd", "street", "st", "lane", "ln", "avenue", "ave", "sector",
    "near", "opposite", "opp", "behind", "beside", "building", "bldg",
    "floor", "fl", "shop", "plot", "house", "home", "block", "district",
    "city", "state", "county", "india", "usa", "us"
}

def safe_text(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except (TypeError, ValueError):
        pass
    return str(x)

def norm_native(x):
    x = safe_text(x).lower().strip()
    if not x:
        return ""
    x = _nonword_re.sub(" ", x)
    return _ws_re.sub(" ", x).strip()

def norm_ascii(x):
    x = safe_text(x)
    if not x:
        return ""
    x = unidecode(x).lower()
    x = _ascii_alnum_re.sub(" ", x)
    return _ws_re.sub(" ", x).strip()

def tokens(text, generic):
    text = safe_text(text)
    if not text:
        return []

    out = []
    seen = set()

    for t in _token_re.findall(text):
        t = t.lower()

        if len(t) < 3:
            continue
        if t in generic:
            continue
        if t in seen:
            continue

        seen.add(t)
        out.append(t)

    return out

def postal_codes(text):
    text = safe_text(text)
    # Captures common 5/6 digit postal patterns without assuming a country.
    return list(dict.fromkeys(re.findall(r"(?<!\d)(\d{5,6})(?!\d)", text)))

def numeric_tokens(text):
    text = safe_text(text)
    return list(dict.fromkeys(
        x for x in re.findall(r"(?<!\w)\d{2,}(?!\w)", text)
    ))

def char_ngrams(text, n=3):
    text = safe_text(text)
    if len(text) < n:
        return []
    x = re.sub(r"[^a-z0-9]", "", text)
    if len(x) < n:
        return []
    return list(dict.fromkeys(x[i:i+n] for i in range(len(x)-n+1)))

def prefix_key(text, n=4):
    x = re.sub(r"[^a-z0-9]", "", safe_text(text))
    return x[:n] if len(x) >= n else ""

print("Normalization helpers loaded.")


Normalization helpers loaded.



# 5. Build a controlled 10K benchmark

A naive 10K-vs-10K sample is dangerous because some true S2/S3 matches may fall outside the random sample.

So we construct the benchmark as:

```text
10,000 S1 entities
+
ALL known positive S2/S3 records for those entities
+
~10,000 random distractors from each source
```

This guarantees that the candidate pool contains every known positive for the selected S1 entities.

The resulting S2/S3 benchmark may be slightly larger than 10K because the positive closure is preserved.


In [5]:

# 5A. MEMORY-SAFE RESERVOIR SAMPLE OF ENTITY IDS

def reservoir_sample_ids(path, n, seed=42):
    rng = random.Random(seed)
    reservoir = []
    seen = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id"],
        dtype=str,
        keep_default_na=False,
        chunksize=100_000,
    ):
        for entity_id in chunk["entity_id"]:
            seen += 1

            if len(reservoir) < n:
                reservoir.append(entity_id)
            else:
                j = rng.randrange(seen)
                if j < n:
                    reservoir[j] = entity_id

    return reservoir

t0 = time.time()

sample_s1_ids = reservoir_sample_ids(
    TRAIN_S1,
    S1_SAMPLE_N,
    seed=SEED,
)

print("Sampled S1 IDs:", len(sample_s1_ids))
print("Time:", round(time.time() - t0, 1), "sec")


Sampled S1 IDs: 10000
Time: 1.5 sec


In [6]:
# 5B. LOAD ONLY GROUND TRUTH ROWS FOR THE SELECTED S1 SAMPLE

def load_ground_truth_for_ids(path, wanted_ids, chunksize=100_000):
    wanted = set(wanted_ids)
    gt_map = {}

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        usecols=["source1_entity_id", "matched_entity_ids"],
        chunksize=chunksize,
        low_memory=False,
    ):
        hit = chunk[chunk["source1_entity_id"].isin(wanted)]

        for _, row in hit.iterrows():
            raw = safe_text(row["matched_entity_ids"])
            gt_map[row["source1_entity_id"]] = {
                x.strip()
                for x in raw.split(",")
                if x.strip()
            }

    # Explicitly create empty entries for sampled S1 rows that have no GT row.
    for s1_id in wanted:
        gt_map.setdefault(s1_id, set())

    return gt_map

GT_MAP = load_ground_truth_for_ids(
    TRAIN_GT,
    sample_s1_ids,
)

sample_s1_set = set(sample_s1_ids)

sample_truth = {
    s1_id: GT_MAP.get(s1_id, set())
    for s1_id in sample_s1_ids
}

positive_s2_ids = set()
positive_s3_ids = set()

for ids in sample_truth.values():
    for entity_id in ids:
        if entity_id.startswith("S2-"):
            positive_s2_ids.add(entity_id)
        elif entity_id.startswith("S3-"):
            positive_s3_ids.add(entity_id)

print("Sample S1:", len(sample_s1_ids))
print("Positive S2:", len(positive_s2_ids))
print("Positive S3:", len(positive_s3_ids))

singleton_count = sum(not ids for ids in sample_truth.values())
print("Sample S1 singletons:", singleton_count)


Sample S1: 10000
Positive S2: 16850
Positive S3: 17901
Sample S1 singletons: 522


In [7]:

# 5C. RESERVOIR SAMPLE RANDOM DISTRACTOR IDS

def reservoir_sample_ids_excluding(path, n, excluded, seed=42):
    rng = random.Random(seed)
    reservoir = []
    seen = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id"],
        dtype=str,
        keep_default_na=False,
        chunksize=100_000,
    ):
        for entity_id in chunk["entity_id"]:
            if entity_id in excluded:
                continue

            seen += 1

            if len(reservoir) < n:
                reservoir.append(entity_id)
            else:
                j = rng.randrange(seen)
                if j < n:
                    reservoir[j] = entity_id

    return reservoir

t0 = time.time()

distractor_s2_ids = reservoir_sample_ids_excluding(
    TRAIN_S2,
    DISTRACTOR_N_PER_SOURCE,
    positive_s2_ids,
    seed=SEED + 1,
)

distractor_s3_ids = reservoir_sample_ids_excluding(
    TRAIN_S3,
    DISTRACTOR_N_PER_SOURCE,
    positive_s3_ids,
    seed=SEED + 2,
)

print("S2 distractors:", len(distractor_s2_ids))
print("S3 distractors:", len(distractor_s3_ids))
print("Time:", round(time.time() - t0, 1), "sec")


S2 distractors: 10000
S3 distractors: 10000
Time: 9.6 sec


In [8]:

# 5D. STREAMING ROW RETRIEVAL BY ENTITY IDS

def load_rows_by_ids(path, wanted_ids, chunksize=100_000):
    wanted = set(wanted_ids)
    parts = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize,
        low_memory=False,
    ):
        hit = chunk[chunk["entity_id"].isin(wanted)]

        if not hit.empty:
            parts.append(hit)

    if not parts:
        return pd.DataFrame(
            columns=["entity_id", "business_name", "business_address", "country"]
        )

    return pd.concat(parts, ignore_index=True)

t0 = time.time()

s1_raw = load_rows_by_ids(TRAIN_S1, sample_s1_ids)

s2_wanted = positive_s2_ids | set(distractor_s2_ids)
s3_wanted = positive_s3_ids | set(distractor_s3_ids)

s2_raw = load_rows_by_ids(TRAIN_S2, s2_wanted)
s3_raw = load_rows_by_ids(TRAIN_S3, s3_wanted)

print("Loaded S1:", len(s1_raw))
print("Loaded S2:", len(s2_raw))
print("Loaded S3:", len(s3_raw))
print("Time:", round(time.time() - t0, 1), "sec")

assert set(s1_raw["entity_id"]) == set(sample_s1_ids)
assert positive_s2_ids.issubset(set(s2_raw["entity_id"]))
assert positive_s3_ids.issubset(set(s3_raw["entity_id"]))


Loaded S1: 10000
Loaded S2: 26850
Loaded S3: 27901
Time: 15.4 sec


In [9]:

# 6. PREPARE BENCHMARK RECORDS

def prepare_records(df, rid_col):
    out = pd.DataFrame()

    out[rid_col] = np.arange(len(df), dtype=np.int32)
    out["entity_id"] = df["entity_id"].map(safe_text)
    out["country"] = df["country"].map(lambda x: safe_text(x).lower())

    out["name_native"] = df["business_name"].map(norm_native)
    out["name_ascii"] = df["business_name"].map(norm_ascii)

    out["addr_native"] = df["business_address"].map(norm_native)
    out["addr_ascii"] = df["business_address"].map(norm_ascii)

    out["name_tokens_native"] = out["name_native"].map(
        lambda x: tokens(x, GENERIC_NAME_TOKENS)
    )
    out["name_tokens_ascii"] = out["name_ascii"].map(
        lambda x: tokens(x, GENERIC_NAME_TOKENS)
    )

    out["addr_tokens_native"] = out["addr_native"].map(
        lambda x: tokens(x, GENERIC_ADDR_TOKENS)
    )
    out["addr_tokens_ascii"] = out["addr_ascii"].map(
        lambda x: tokens(x, GENERIC_ADDR_TOKENS)
    )

    out["postal_native"] = df["business_address"].map(postal_codes)

    out["numbers_native"] = out["addr_native"].map(numeric_tokens)

    out["name_prefix4"] = out["name_ascii"].map(prefix_key)
    out["name_prefix6"] = out["name_ascii"].map(lambda x: prefix_key(x, 6))

    out["name_qgrams"] = out["name_ascii"].map(lambda x: char_ngrams(x, 3))

    return out

S1 = prepare_records(s1_raw, "s1_rid")
S2 = prepare_records(s2_raw, "s2_rid")
S3 = prepare_records(s3_raw, "s3_rid")

CAND = pd.concat(
    [
        S2.assign(source="S2"),
        S3.assign(source="S3", s23_rid=lambda x: x["s3_rid"] + 10_000_000),
    ],
    ignore_index=True,
)

# Give the unified candidate table one stable integer row id.
CAND = CAND.reset_index(drop=True)
CAND["cand_rid"] = np.arange(len(CAND), dtype=np.int32)

# Remove any row-order advantage created by positive-closure construction.
CAND = CAND.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
CAND["cand_rid"] = np.arange(len(CAND), dtype=np.int32)

print("S1 rows:", len(S1))
print("Candidate rows:", len(CAND))
print(CAND["source"].value_counts())


S1 rows: 10000
Candidate rows: 54751
source
S3    27901
S2    26850
Name: count, dtype: int64



# 7. Learn posting frequencies

A blocking key is useful when:

```text
it retrieves true matches
AND
its posting list is not enormous
```

We therefore measure key frequency first.

The rarest key available for an S1 record should normally be preferred over a generic key such as `medical`, `store`, `india`, or `road`.


In [10]:
# 7. POSTING COUNTS
#
# Frequencies are kept separately for native and ASCII forms.
# This avoids one representation changing the rarity of another.

def count_postings(series):
    c = Counter()
    for values in series:
        c.update(set(values))
    return c

name_token_freq_native = count_postings(CAND["name_tokens_native"])
name_token_freq_ascii = count_postings(CAND["name_tokens_ascii"])

addr_token_freq_native = count_postings(CAND["addr_tokens_native"])
addr_token_freq_ascii = count_postings(CAND["addr_tokens_ascii"])

qgram_freq = count_postings(CAND["name_qgrams"])

name_native_freq = Counter(
    x for x in CAND["name_native"] if x
)
name_ascii_freq = Counter(
    x for x in CAND["name_ascii"] if x
)

prefix4_freq = Counter(
    CAND.loc[CAND["name_prefix4"] != "", "name_prefix4"]
)

postal_freq = count_postings(CAND["postal_native"])

print("Native name token types:", len(name_token_freq_native))
print("ASCII name token types :", len(name_token_freq_ascii))
print("Native addr token types:", len(addr_token_freq_native))
print("ASCII addr token types :", len(addr_token_freq_ascii))
print("Q-gram types            :", len(qgram_freq))
print("Exact native names      :", len(name_native_freq))
print("Exact ASCII names       :", len(name_ascii_freq))
print("Prefix4 types            :", len(prefix4_freq))
print("Postal codes             :", len(postal_freq))


Native name token types: 26472
ASCII name token types : 25883
Native addr token types: 37607
ASCII addr token types : 37624
Q-gram types            : 15435
Exact native names      : 49076
Exact ASCII names       : 48216
Prefix4 types            : 10116
Postal codes             : 2458


In [11]:
# 8. GENERATE RARE KEYS FOR ONE RECORD

def rare_keys(values, freq_map, max_posting, n):
    usable = [
        (v, freq_map.get(v, 10**12))
        for v in set(values)
        if 0 < freq_map.get(v, 10**12) <= max_posting
    ]

    usable.sort(key=lambda x: (x[1], -len(x[0])))
    return [v for v, _ in usable[:n]]

def record_keys(row):
    return {
        "exact_native_name": (
            ("N", row["name_native"]) if row["name_native"] else None
        ),
        "exact_ascii_name": (
            ("A", row["name_ascii"]) if row["name_ascii"] else None
        ),
        "rare_name_native": [
            ("NT", t)
            for t in rare_keys(
                row["name_tokens_native"],
                name_token_freq_native,
                MAX_NAME_POSTING,
                RARE_NAME_KEYS,
            )
        ],
        "rare_name_ascii": [
            ("AT", t)
            for t in rare_keys(
                row["name_tokens_ascii"],
                name_token_freq_ascii,
                MAX_NAME_POSTING,
                RARE_NAME_KEYS,
            )
        ],
        "rare_addr_native": [
            ("DN", t)
            for t in rare_keys(
                row["addr_tokens_native"],
                addr_token_freq_native,
                MAX_ADDR_POSTING,
                RARE_ADDR_KEYS,
            )
        ],
        "rare_addr_ascii": [
            ("DA", t)
            for t in rare_keys(
                row["addr_tokens_ascii"],
                addr_token_freq_ascii,
                MAX_ADDR_POSTING,
                RARE_ADDR_KEYS,
            )
        ],
        "postal": [
            ("P", p)
            for p in row["postal_native"]
            if postal_freq.get(p, 10**12) <= MAX_ADDR_POSTING
        ],
        "prefix4": (
            [("PX4", row["name_prefix4"])]
            if (
                row["name_prefix4"]
                and prefix4_freq.get(row["name_prefix4"], 10**12)
                <= MAX_PREFIX_POSTING
            )
            else []
        ),
        "qgram": [
            ("Q", q)
            for q in rare_keys(
                row["name_qgrams"],
                qgram_freq,
                MAX_QGRAM_POSTING,
                RARE_QGRAM_KEYS,
            )
        ],
    }

example_keys = record_keys(S1.iloc[0])
example_keys


{'exact_native_name': ('N', 'raheja estates limited'),
 'exact_ascii_name': ('A', 'raheja estates limited'),
 'rare_name_native': [('NT', 'raheja'), ('NT', 'estates')],
 'rare_name_ascii': [('AT', 'raheja'), ('AT', 'estates')],
 'rare_addr_native': [('DN', 'zahid'), ('DN', 'kkg')],
 'rare_addr_ascii': [('DA', 'zahid'), ('DA', 'kkg')],
 'postal': [],
 'prefix4': [('PX4', 'rahe')],
 'qgram': [('Q', 'jae'), ('Q', 'hej'), ('Q', 'aes')]}


# 9. Build inverted indexes

Each index is:

```text
blocking_key → candidate row IDs
```

There are **no pairwise comparisons** during index construction.

Common keys are omitted once their posting list exceeds the configured cap.

That is the critical guardrail that prevents a key such as `store` from exploding candidate generation.


In [12]:
# 9. INVERTED INDEXES
#
# IMPORTANT:
# We do NOT keep "the first 150 rows" of a common key.
# That would make the benchmark depend on row order and, because positives
# are deliberately included first, could artificially inflate recall.
#
# Instead, a key is either:
#   - fully indexed when its true benchmark posting size is <= threshold
#   - omitted when it is too common

def add_posting(index, key, rid):
    index[key].add(int(rid))

def build_indexes(cand):
    idx = {
        "exact_native_name": defaultdict(set),
        "exact_ascii_name": defaultdict(set),
        "rare_name_native": defaultdict(set),
        "rare_name_ascii": defaultdict(set),
        "rare_addr_native": defaultdict(set),
        "rare_addr_ascii": defaultdict(set),
        "postal": defaultdict(set),
        "prefix4": defaultdict(set),
        "qgram": defaultdict(set),
    }

    for _, row in cand.iterrows():
        rid = int(row["cand_rid"])

        if (
            row["name_native"]
            and name_native_freq.get(row["name_native"], 10**12)
            <= MAX_NAME_POSTING
        ):
            add_posting(
                idx["exact_native_name"],
                row["name_native"],
                rid,
            )

        if (
            row["name_ascii"]
            and name_ascii_freq.get(row["name_ascii"], 10**12)
            <= MAX_NAME_POSTING
        ):
            add_posting(
                idx["exact_ascii_name"],
                row["name_ascii"],
                rid,
            )

        for t in rare_keys(
            row["name_tokens_native"],
            name_token_freq_native,
            MAX_NAME_POSTING,
            RARE_NAME_KEYS,
        ):
            add_posting(idx["rare_name_native"], t, rid)

        for t in rare_keys(
            row["name_tokens_ascii"],
            name_token_freq_ascii,
            MAX_NAME_POSTING,
            RARE_NAME_KEYS,
        ):
            add_posting(idx["rare_name_ascii"], t, rid)

        for t in rare_keys(
            row["addr_tokens_native"],
            addr_token_freq_native,
            MAX_ADDR_POSTING,
            RARE_ADDR_KEYS,
        ):
            add_posting(idx["rare_addr_native"], t, rid)

        for t in rare_keys(
            row["addr_tokens_ascii"],
            addr_token_freq_ascii,
            MAX_ADDR_POSTING,
            RARE_ADDR_KEYS,
        ):
            add_posting(idx["rare_addr_ascii"], t, rid)

        for p in row["postal_native"]:
            if postal_freq.get(p, 10**12) <= MAX_ADDR_POSTING:
                add_posting(idx["postal"], p, rid)

        p4 = row["name_prefix4"]

        if (
            p4
            and prefix4_freq.get(p4, 10**12) <= MAX_PREFIX_POSTING
        ):
            add_posting(idx["prefix4"], p4, rid)

        for q in rare_keys(
            row["name_qgrams"],
            qgram_freq,
            MAX_QGRAM_POSTING,
            RARE_QGRAM_KEYS,
        ):
            add_posting(idx["qgram"], q, rid)

    return idx

t0 = time.time()
INDEX = build_indexes(CAND)
print("Index build time:", round(time.time() - t0, 2), "sec")

for name, d in INDEX.items():
    print(f"{name:20s} keys={len(d):7d}")


Index build time: 2.34 sec
exact_native_name    keys=  49076
exact_ascii_name     keys=  48216
rare_name_native     keys=  26245
rare_name_ascii      keys=  25652
rare_addr_native     keys=  35393
rare_addr_ascii      keys=  35402
postal               keys=   2458
prefix4              keys=  10112
qgram                keys=  14358



# 10. Retrieval evidence

We do not simply union everything and keep all results.

Each retrieval path contributes evidence:

```text
exact full name        = very strong
postal code            = strong
rare name token        = strong
rare address token     = medium
rare q-gram            = weak/diagnostic
prefix                 = fallback
```

Then the candidate set can be ranked before any fuzzy matcher is involved.


In [13]:

# 10. RETRIEVE CANDIDATES FOR ONE S1 ROW

BLOCK_WEIGHTS = {
    "exact_native_name": 10.0,
    "exact_ascii_name": 9.0,
    "postal": 6.0,
    "rare_name_native": 4.0,
    "rare_name_ascii": 4.0,
    "rare_addr_native": 2.5,
    "rare_addr_ascii": 2.5,
    "qgram": 1.0,
    "prefix4": 0.5,
}

def retrieve_one(row, indexes, retrieval_top_k=50):
    evidence = defaultdict(lambda: {
        "score": 0.0,
        "hits": defaultdict(int),
    })

    def add_hits(index_name, key, weight):
        if not key:
            return

        postings = indexes[index_name].get(key, ())
        for rid in postings:
            evidence[rid]["score"] += weight
            evidence[rid]["hits"][index_name] += 1

    if row["name_native"]:
        add_hits(
            "exact_native_name",
            row["name_native"],
            BLOCK_WEIGHTS["exact_native_name"],
        )

    if row["name_ascii"]:
        add_hits(
            "exact_ascii_name",
            row["name_ascii"],
            BLOCK_WEIGHTS["exact_ascii_name"],
        )

    for t in rare_keys(
        row["name_tokens_native"],
        name_token_freq_native,
        MAX_NAME_POSTING,
        RARE_NAME_KEYS,
    ):
        add_hits("rare_name_native", t, BLOCK_WEIGHTS["rare_name_native"])

    for t in rare_keys(
        row["name_tokens_ascii"],
        name_token_freq_native,
        MAX_NAME_POSTING,
        RARE_NAME_KEYS,
    ):
        add_hits("rare_name_ascii", t, BLOCK_WEIGHTS["rare_name_ascii"])

    for t in rare_keys(
        row["addr_tokens_native"],
        addr_token_freq_native,
        MAX_ADDR_POSTING,
        RARE_ADDR_KEYS,
    ):
        add_hits("rare_addr_native", t, BLOCK_WEIGHTS["rare_addr_native"])

    for t in rare_keys(
        row["addr_tokens_ascii"],
        addr_token_freq_ascii,
        MAX_ADDR_POSTING,
        RARE_ADDR_KEYS,
    ):
        add_hits("rare_addr_ascii", t, BLOCK_WEIGHTS["rare_addr_ascii"])

    for p in row["postal_native"]:
        if postal_freq.get(p, 10**12) <= MAX_ADDR_POSTING:
            add_hits("postal", p, BLOCK_WEIGHTS["postal"])

    p4 = row["name_prefix4"]
    if p4 and prefix4_freq.get(p4, 10**12) <= MAX_PREFIX_POSTING:
        add_hits("prefix4", p4, BLOCK_WEIGHTS["prefix4"])

    for q in rare_keys(
        row["name_qgrams"],
        qgram_freq,
        MAX_QGRAM_POSTING,
        RARE_QGRAM_KEYS,
    ):
        add_hits("qgram", q, BLOCK_WEIGHTS["qgram"])

    ranked = sorted(
        evidence.items(),
        key=lambda kv: (
            kv[1]["score"],
            sum(kv[1]["hits"].values()),
        ),
        reverse=True,
    )

    out = []

    for rid, info in ranked[:retrieval_top_k]:
        out.append({
            "cand_rid": rid,
            "block_score": info["score"],
            "block_hits": dict(info["hits"]),
        })

    return out


In [14]:

# 11. RUN THE COMBINED BLOCKER

def run_retrieval(s1_df, indexes, retrieval_top_k=50):
    rows = []

    t0 = time.time()

    for _, row in s1_df.iterrows():
        hits = retrieve_one(
            row,
            indexes,
            retrieval_top_k=retrieval_top_k,
        )

        for h in hits:
            h["s1_rid"] = int(row["s1_rid"])
            rows.append(h)

    elapsed = time.time() - t0

    result = pd.DataFrame(rows)

    if not result.empty:
        result = result[
            ["s1_rid", "cand_rid", "block_score", "block_hits"]
        ]

    return result, elapsed

retrieval_50, retrieval_time = run_retrieval(
    S1,
    INDEX,
    retrieval_top_k=50,
)

print("Candidate rows:", len(retrieval_50))
print("Retrieval time:", round(retrieval_time, 3), "sec")


Candidate rows: 485561
Retrieval time: 2.596 sec



# 12. Ground-truth-aware evaluation

We now measure recall **before** any matching model.

A correct blocker should put every true linked S2/S3 record into its candidate set.

This is the metric that determines whether a more sophisticated matcher can recover the answer.


In [15]:

# 12. EVALUATION HELPERS

cand_entity_by_rid = dict(
    zip(CAND["cand_rid"].astype(int), CAND["entity_id"])
)

s1_entity_by_rid = dict(
    zip(S1["s1_rid"].astype(int), S1["entity_id"])
)

def evaluate_candidate_recall(candidate_df, k_values=K_VALUES):
    if candidate_df.empty:
        return pd.DataFrame(columns=[
            "K", "macro_candidate_recall",
            "any_hit_rate", "avg_candidates",
            "median_candidates", "p95_candidates"
        ])

    grouped = {
        s1_rid: group.sort_values(
            "block_score", ascending=False
        )["cand_rid"].astype(int).tolist()
        for s1_rid, group in candidate_df.groupby("s1_rid")
    }

    result_rows = []

    for k in k_values:
        recalls = []
        any_hits = []

        for s1_rid, s1_id in s1_entity_by_rid.items():
            truth = sample_truth.get(s1_id, set())

            if not truth:
                # Candidate recall is not meaningful for a true singleton.
                continue

            retrieved = {
                cand_entity_by_rid[rid]
                for rid in grouped.get(s1_rid, [])[:k]
            }

            hit = truth.intersection(retrieved)

            recalls.append(len(hit) / len(truth))
            any_hits.append(bool(hit))

        # Candidate count statistics use all S1 entities.
        counts = np.array([
            min(k, len(grouped.get(rid, [])))
            for rid in s1_entity_by_rid
        ])

        result_rows.append({
            "K": k,
            "macro_candidate_recall": np.mean(recalls) if recalls else 0.0,
            "any_hit_rate": np.mean(any_hits) if any_hits else 0.0,
            "avg_candidates": float(counts.mean()),
            "median_candidates": float(np.median(counts)),
            "p95_candidates": float(np.quantile(counts, 0.95)),
        })

    return pd.DataFrame(result_rows)

recall_table = evaluate_candidate_recall(retrieval_50)
recall_table


,K,macro_candidate_recall,any_hit_rate,avg_candidates,median_candidates,p95_candidates
0,5,0.885028,0.988605,4.9981,5.0,5.0
1,10,0.925079,0.991137,9.9844,10.0,10.0
2,20,0.936913,0.992192,19.8965,20.0,20.0
3,30,0.945307,0.994408,29.6719,30.0,30.0
4,50,0.955685,0.995569,48.5561,50.0,50.0



# 13. Measure every blocking family independently

This is the most important experiment.

We want to know **which blocking channels actually contribute recall**, rather than assuming the combined system is good.

Run each family by itself, then compare:

```text
recall ↑
candidate count ↓
runtime ↓
```


In [16]:

# 13. SINGLE-BLOCK EXPERIMENTS

def retrieve_one_specific(row, index_name, indexes, top_k=50):
    evidence = defaultdict(float)

    def add(key):
        if not key:
            return
        for rid in indexes[index_name].get(key, ()):
            evidence[rid] += 1.0

    if index_name == "exact_native_name":
        add(row["name_native"])

    elif index_name == "exact_ascii_name":
        add(row["name_ascii"])

    elif index_name == "rare_name_native":
        for t in rare_keys(
            row["name_tokens_native"],
            name_token_freq_native,
            MAX_NAME_POSTING,
            RARE_NAME_KEYS,
        ):
            add(t)

    elif index_name == "rare_name_ascii":
        for t in rare_keys(
            row["name_tokens_ascii"],
            name_token_freq_ascii,
            MAX_NAME_POSTING,
            RARE_NAME_KEYS,
        ):
            add(t)

    elif index_name == "rare_addr_native":
        for t in rare_keys(
            row["addr_tokens_native"],
            addr_token_freq_native,
            MAX_ADDR_POSTING,
            RARE_ADDR_KEYS,
        ):
            add(t)

    elif index_name == "rare_addr_ascii":
        for t in rare_keys(
            row["addr_tokens_ascii"],
            addr_token_freq_ascii,
            MAX_ADDR_POSTING,
            RARE_ADDR_KEYS,
        ):
            add(t)

    elif index_name == "postal":
        for p in row["postal_native"]:
            add(p)

    elif index_name == "prefix4":
        add(row["name_prefix4"])

    elif index_name == "qgram":
        for q in rare_keys(
            row["name_qgrams"],
            qgram_freq,
            MAX_QGRAM_POSTING,
            RARE_QGRAM_KEYS,
        ):
            add(q)

    ranked = sorted(
        evidence.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    return [
        {"cand_rid": rid, "block_score": score, "block_hits": {}}
        for rid, score in ranked[:top_k]
    ]

block_families = [
    "exact_native_name",
    "exact_ascii_name",
    "rare_name_native",
    "rare_name_ascii",
    "rare_addr_native",
    "rare_addr_ascii",
    "postal",
    "qgram",
    "prefix4",
]

single_block_results = []

for family in block_families:
    rows = []
    t0 = time.time()

    for _, row in S1.iterrows():
        for hit in retrieve_one_specific(
            row,
            family,
            INDEX,
            top_k=50,
        ):
            hit["s1_rid"] = int(row["s1_rid"])
            rows.append(hit)

    tmp = pd.DataFrame(rows)
    elapsed = time.time() - t0

    metrics = evaluate_candidate_recall(tmp)

    if not metrics.empty:
        best = metrics.iloc[-1].to_dict()
        single_block_results.append({
            "family": family,
            "runtime_sec": elapsed,
            "recall@50": best["macro_candidate_recall"],
            "any_hit@50": best["any_hit_rate"],
            "avg_candidates@50": best["avg_candidates"],
        })

single_block_results = pd.DataFrame(single_block_results)
single_block_results.sort_values(
    ["recall@50", "avg_candidates@50"],
    ascending=[False, True],
)


,family,runtime_sec,recall@50,any_hit@50,avg_candidates@50
5,rare_addr_ascii,0.520260,0.880898,0.986601,12.1749
4,rare_addr_native,0.358057,0.880898,0.986601,12.1783
7,qgram,1.282301,0.740695,0.921080,44.2289
3,rare_name_ascii,0.861541,0.720909,0.876345,27.3587
2,rare_name_native,1.000530,0.714404,0.877506,27.2526
8,prefix4,0.832494,0.641133,0.874235,26.0653
1,exact_ascii_name,0.173737,0.261707,0.611310,0.9777
0,exact_native_name,0.225340,0.221436,0.545685,0.8313
6,postal,0.120518,0.051175,0.065098,0.1992



# 14. Adaptive fallback logic

We do not need every retrieval channel for every record.

A good cascade is:

```text
Stage 1:
exact name / exact transliteration

if enough candidates:
    stop

else:
Stage 2:
rare name + address anchors

if enough candidates:
    stop

else:
Stage 3:
rare q-grams

if still empty:
Stage 4:
bounded prefix fallback
```

This is where large runtime savings can come from.

An easy business name should not receive the same amount of retrieval work as a terrible/noisy business name.


In [17]:

# 14. ADAPTIVE RETRIEVER

EARLY_STOP = 12

def adaptive_retrieve_one(row, indexes, final_k=30):
    evidence = defaultdict(lambda: {
        "score": 0.0,
        "hits": defaultdict(int),
    })

    def add(index_name, key, weight):
        if not key:
            return

        postings = indexes[index_name].get(key, ())

        for rid in postings:
            evidence[rid]["score"] += weight
            evidence[rid]["hits"][index_name] += 1

    # Stage 1 — exact names.
    if row["name_native"]:
        add(
            "exact_native_name",
            row["name_native"],
            BLOCK_WEIGHTS["exact_native_name"],
        )

    if row["name_ascii"]:
        add(
            "exact_ascii_name",
            row["name_ascii"],
            BLOCK_WEIGHTS["exact_ascii_name"],
        )

    # If the strong stage already produced enough evidence-backed candidates,
    # don't activate all weaker retrieval paths.
    if len(evidence) < EARLY_STOP:

        # Stage 2 — rare names.
        for t in rare_keys(
            row["name_tokens_native"],
            name_token_freq_native,
            MAX_NAME_POSTING,
            RARE_NAME_KEYS,
        ):
            add("rare_name_native", t, BLOCK_WEIGHTS["rare_name_native"])

        for t in rare_keys(
            row["name_tokens_ascii"],
            name_token_freq_ascii,
            MAX_NAME_POSTING,
            RARE_NAME_KEYS,
        ):
            add("rare_name_ascii", t, BLOCK_WEIGHTS["rare_name_ascii"])

        # Stage 2 — address anchors.
        for p in row["postal_native"]:
            if postal_freq.get(p, 10**12) <= MAX_ADDR_POSTING:
                add("postal", p, BLOCK_WEIGHTS["postal"])

        for t in rare_keys(
            row["addr_tokens_native"],
            addr_token_freq_native,
            MAX_ADDR_POSTING,
            RARE_ADDR_KEYS,
        ):
            add("rare_addr_native", t, BLOCK_WEIGHTS["rare_addr_native"])

        for t in rare_keys(
            row["addr_tokens_ascii"],
            addr_token_freq_ascii,
            MAX_ADDR_POSTING,
            RARE_ADDR_KEYS,
        ):
            add("rare_addr_ascii", t, BLOCK_WEIGHTS["rare_addr_ascii"])

    # Stage 3 — q-grams only for weak/no-result rows.
    if len(evidence) < EARLY_STOP:
        for q in rare_keys(
            row["name_qgrams"],
            qgram_freq,
            MAX_QGRAM_POSTING,
            RARE_QGRAM_KEYS,
        ):
            add("qgram", q, BLOCK_WEIGHTS["qgram"])

    # Stage 4 — bounded prefix fallback.
    if len(evidence) < EARLY_STOP:
        p4 = row["name_prefix4"]
        if (
            p4
            and prefix4_freq.get(p4, 10**12) <= MAX_PREFIX_POSTING
        ):
            add("prefix4", p4, BLOCK_WEIGHTS["prefix4"])

    ranked = sorted(
        evidence.items(),
        key=lambda kv: (
            kv[1]["score"],
            sum(kv[1]["hits"].values()),
        ),
        reverse=True,
    )

    return [
        {
            "cand_rid": rid,
            "block_score": info["score"],
            "block_hits": dict(info["hits"]),
        }
        for rid, info in ranked[:final_k]
    ]

adaptive_rows = []
t0 = time.time()

for _, row in S1.iterrows():
    for hit in adaptive_retrieve_one(
        row,
        INDEX,
        final_k=50,
    ):
        hit["s1_rid"] = int(row["s1_rid"])
        adaptive_rows.append(hit)

adaptive_candidates = pd.DataFrame(adaptive_rows)
adaptive_time = time.time() - t0

print("Adaptive runtime:", round(adaptive_time, 3), "sec")
print("Candidate rows:", len(adaptive_candidates))

adaptive_recall = evaluate_candidate_recall(adaptive_candidates)
adaptive_recall


Adaptive runtime: 2.067 sec
Candidate rows: 403185


,K,macro_candidate_recall,any_hit_rate,avg_candidates,median_candidates,p95_candidates
0,5,0.880553,0.988394,4.9981,5.0,5.0
1,10,0.913937,0.989977,9.9844,10.0,10.0
2,20,0.927170,0.991981,19.3428,20.0,20.0
3,30,0.935901,0.993353,27.2718,30.0,30.0
4,50,0.948329,0.994830,40.3185,50.0,50.0



# 15. Candidate quality after cheap reranking

Retrieval evidence alone is not enough.

We now use **cheap** RapidFuzz features only on the already-retrieved candidates.

This is still nowhere near an all-pairs comparison.

```text
20,000–30,000 benchmark rows
    ↓
blocking
    ↓
~tens of candidates / S1
    ↓
RapidFuzz
    ↓
top-K final candidates
```

The same structure can later feed CatBoost.


In [18]:

# 15. CHEAP RERANKER

def candidate_features(s1_row, cand_row, block_score):
    name_a = s1_row["name_ascii"]
    name_b = cand_row["name_ascii"]

    addr_a = s1_row["addr_ascii"]
    addr_b = cand_row["addr_ascii"]

    name_ratio = (
        fuzz.ratio(name_a, name_b) / 100.0
        if name_a and name_b else 0.0
    )

    name_token_ratio = (
        fuzz.token_set_ratio(name_a, name_b) / 100.0
        if name_a and name_b else 0.0
    )

    addr_ratio = (
        fuzz.ratio(addr_a, addr_b) / 100.0
        if addr_a and addr_b else 0.0
    )

    shared_name = len(
        set(s1_row["name_tokens_ascii"])
        & set(cand_row["name_tokens_ascii"])
    )

    shared_addr = len(
        set(s1_row["addr_tokens_ascii"])
        & set(cand_row["addr_tokens_ascii"])
    )

    postal_match = bool(
        set(s1_row["postal_native"])
        & set(cand_row["postal_native"])
    )

    country_match = (
        s1_row["country"] == cand_row["country"]
        and bool(s1_row["country"])
    )

    score = (
        0.45 * name_ratio
        + 0.18 * name_token_ratio
        + 0.20 * addr_ratio
        + 0.07 * min(shared_name / 2.0, 1.0)
        + 0.05 * min(shared_addr / 2.0, 1.0)
        + 0.03 * float(postal_match)
        + 0.02 * float(country_match)
        + 0.05 * min(block_score / 10.0, 1.0)
    )

    return score

def cheap_rerank(candidate_df, s1_df, cand_df, final_k=20):
    if candidate_df.empty:
        return candidate_df.copy()

    s1_lookup = s1_df.set_index("s1_rid")
    cand_lookup = cand_df.set_index("cand_rid")

    rows = []

    for _, c in candidate_df.iterrows():
        s1_row = s1_lookup.loc[int(c["s1_rid"])]
        cand_row = cand_lookup.loc[int(c["cand_rid"])]

        score = candidate_features(
            s1_row,
            cand_row,
            float(c["block_score"]),
        )

        rows.append({
            "s1_rid": int(c["s1_rid"]),
            "cand_rid": int(c["cand_rid"]),
            "block_score": float(c["block_score"]),
            "cheap_score": score,
            "block_hits": c.get("block_hits", {}),
        })

    x = pd.DataFrame(rows)

    x["rank"] = (
        x.groupby("s1_rid")["cheap_score"]
        .rank(method="first", ascending=False)
    )

    x = x[x["rank"] <= final_k].copy()

    return x

final_candidates = cheap_rerank(
    adaptive_candidates,
    S1,
    CAND,
    final_k=20,
)

print("Final candidate rows:", len(final_candidates))
print(
    final_candidates.groupby("s1_rid").size().describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)


Final candidate rows: 193428
count    10000.000000
mean        19.342800
std          1.969131
min          2.000000
50%         20.000000
90%         20.000000
95%         20.000000
99%         20.000000
max         20.000000
dtype: float64


In [19]:
# 16. CANDIDATE RECALL AT EVERY STAGE

print("=== Adaptive retrieval ===")
print(evaluate_candidate_recall(adaptive_candidates))

print("\n=== After cheap reranking ===")

# evaluate_candidate_recall() expects a ranking column called block_score.
# Do not rename cheap_score onto the existing block_score column because
# that creates duplicate column labels.
final_for_recall = final_candidates[
    ["s1_rid", "cand_rid", "cheap_score"]
].copy()

final_for_recall["block_score"] = final_for_recall["cheap_score"]
final_for_recall.drop(columns=["cheap_score"], inplace=True)

print(evaluate_candidate_recall(final_for_recall))


=== Adaptive retrieval ===
    K  macro_candidate_recall  any_hit_rate  avg_candidates  \
0   5                0.880553      0.988394          4.9981   
1  10                0.913937      0.989977          9.9844   
2  20                0.927170      0.991981         19.3428   
3  30                0.935901      0.993353         27.2718   
4  50                0.948329      0.994830         40.3185   

   median_candidates  p95_candidates  
0                5.0             5.0  
1               10.0            10.0  
2               20.0            20.0  
3               30.0            30.0  
4               50.0            50.0  

=== After cheap reranking ===
    K  macro_candidate_recall  any_hit_rate  avg_candidates  \
0   5                0.890405      0.989238          4.9981   
1  10                0.928105      0.992087          9.9844   
2  20                0.938688      0.993564         19.3428   
3  30                0.938688      0.993564         19.3428   
4  50         


# 17. Stress-test the design

A 10K benchmark alone is not enough to claim scalability.

We therefore test how candidate generation changes as the benchmark corpus grows.

Because the underlying algorithm is an inverted index, the important behavior is:

```text
index build:
O(number of records × number of keys per record)

query:
O(number of query keys + postings actually traversed)

NOT:
O(number of S1 × number of S2/S3)
```

That is the scalability property we are trying to preserve.


In [20]:
# 17. SCALE-CHECK NOTE
#
# Do not treat a 2K/5K/10K subset curve as proof of 12M-row scalability.
# A real full-corpus benchmark requires the complete posting distribution.
#
# The useful quantities from this 10K lab are:
#   1. candidate recall at K
#   2. candidate-count distribution
#   3. posting-list distribution
#   4. query runtime
#
# We report the observed posting statistics here.

posting_lengths = []

for index_name, index in INDEX.items():
    for key, postings in index.items():
        posting_lengths.append({
            "index": index_name,
            "key": str(key),
            "posting_size": len(postings),
        })

posting_stats = pd.DataFrame(posting_lengths)

if not posting_stats.empty:
    print(posting_stats.groupby("index")["posting_size"].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ))
else:
    print("No posting statistics available.")


                     count       mean        std  min  50%   90%   95%   99%  \
index                                                                          
exact_ascii_name   48216.0   1.135536   0.472390  1.0  1.0   1.0   2.0   3.0   
exact_native_name  49076.0   1.115637   0.431344  1.0  1.0   1.0   2.0   3.0   
postal              2458.0   1.721318   1.272754  1.0  1.0   4.0   5.0   6.0   
prefix4            10112.0   5.253659  13.944584  1.0  1.0   9.0  19.0  76.0   
qgram              14358.0  10.417607  13.535061  1.0  4.0  31.0  40.0  58.0   
rare_addr_ascii    35402.0   2.859697   3.417926  1.0  1.0   6.0   9.0  17.0   
rare_addr_native   35393.0   2.857627   3.413461  1.0  1.0   6.0   9.0  17.0   
rare_name_ascii    25652.0   3.283136   8.535950  1.0  1.0   6.0  10.0  48.0   
rare_name_native   26245.0   3.003620   7.837272  1.0  1.0   5.0   9.0  42.0   

                     max  
index                     
exact_ascii_name     9.0  
exact_native_name    9.0  
postal     


# 18. Full-scale complexity estimate

The 10K lab is intended to validate the **algorithmic structure**, not to claim an exact 12M-row runtime.

The next full-corpus implementation must learn posting frequencies from the real S2/S3 population (or a controlled full-corpus frequency pass). Otherwise a key that appears rare in the 10K benchmark could be common in the actual 5M+ corpus.

The observed candidate budget is still useful for estimating downstream model volume:

```text
2.2M S1 × 5   = ~11M candidate rows
2.2M S1 × 10  = ~22M
2.2M S1 × 20  = ~44M
2.2M S1 × 30  = ~66M
2.2M S1 × 150 = ~331M
```

That is why the objective of this lab is to make `K=10–30` viable without destroying candidate recall.


In [21]:

# 18. SCALING ESTIMATE

TOTAL_S1 = 2_206_821
TOTAL_S2 = 5_034_616
TOTAL_S3 = 5_300_000  # approximate; replace with exact test/train count if known

observed_avg_candidates = (
    final_candidates.groupby("s1_rid").size().mean()
    if not final_candidates.empty
    else 0
)

for k in [5, 10, 20, 30, 50, 150]:
    candidate_rows = TOTAL_S1 * k

    print(
        f"K={k:3d} -> "
        f"~{candidate_rows:,.0f} candidate rows "
        f"for {TOTAL_S1:,} S1 records"
    )

print(
    "\nObserved V2 average final candidates:",
    round(float(observed_avg_candidates), 2)
)


K=  5 -> ~11,034,105 candidate rows for 2,206,821 S1 records
K= 10 -> ~22,068,210 candidate rows for 2,206,821 S1 records
K= 20 -> ~44,136,420 candidate rows for 2,206,821 S1 records
K= 30 -> ~66,204,630 candidate rows for 2,206,821 S1 records
K= 50 -> ~110,341,050 candidate rows for 2,206,821 S1 records
K=150 -> ~331,023,150 candidate rows for 2,206,821 S1 records

Observed V2 average final candidates: 19.34



# 19. What qualifies as a V2 blocker

Do not move this design to the 12M-row corpus until the benchmark gives us evidence for all of these:

```text
1. High candidate recall
2. Small median candidate count
3. Small p95 candidate count
4. No enormous posting-list explosions
5. Runtime dominated by postings/candidates, not pairwise scans
6. Cheap reranker sees only the final candidate budget
```

### The critical comparison

Bad:

```text
Every S1 → fixed 150 candidates
```

Good:

```text
easy S1   → 1–10 candidates
normal S1 → 10–20 candidates
hard S1   → 20–50 candidates
```

### Production architecture after this notebook

The exact key definitions from this notebook should be retained.

Only the **storage backend** needs to change:

```text
10K prototype
Python dict postings

        ↓ same keys / same retrieval logic

full corpus
disk-backed postings
DuckDB / SQLite / custom sharded index
```

That separation is important. We do not want to redesign the blocking algorithm while holding 12M records.


In [22]:

# 19. SAVE BENCHMARK ARTIFACTS

S1.to_parquet(BENCHMARK_DIR / "s1_10k.parquet", index=False)
S2.to_parquet(BENCHMARK_DIR / "s2_benchmark.parquet", index=False)
S3.to_parquet(BENCHMARK_DIR / "s3_benchmark.parquet", index=False)
CAND.to_parquet(BENCHMARK_DIR / "candidate_benchmark.parquet", index=False)

adaptive_candidates.to_pickle(
    BENCHMARK_DIR / "adaptive_candidates.pkl"
)

final_candidates.to_pickle(
    BENCHMARK_DIR / "final_candidates.pkl"
)

single_block_results.to_csv(
    BENCHMARK_DIR / "single_block_results.tsv",
    sep="\t",
    index=False,
)

recall_table.to_csv(
    BENCHMARK_DIR / "combined_recall_table.tsv",
    sep="\t",
    index=False,
)

print("Saved benchmark artifacts to:", BENCHMARK_DIR)


Saved benchmark artifacts to: /Users/atharv/Documents/AMAZON_ML/blocking_lab_10k/benchmark



# 20. Decision gate

### Do not build the 12M-row production index yet.

First inspect:

```python
single_block_results
```

and:

```python
adaptive_recall
```

and:

```python
final_candidates.groupby("s1_rid").size().describe(...)
```

The next design iteration should be driven by the observed failure mode:

```text
high recall + too many candidates
    → tighten key rarity / adaptive pruning

low recall
    → add another blocking channel

good candidate recall + slow query
    → reduce postings / key count

good blocking + too many fuzzy comparisons
    → stronger cheap reranker
```

Only after this 10K blocker passes the gate should we write the full-corpus implementation.

**This notebook intentionally does not generate the final submission.**


# 21. VERIFIED CORRECTION + NEXT EXPERIMENT

Before we add another blocking rule, one implementation issue in the original retriever must be corrected.

**Correction:** the `rare_name_ascii` query path was using `name_token_freq_native` instead of `name_token_freq_ascii`. The index itself was built with the correct ASCII frequency map, but the query-side `rare_keys(...)` call was not. That makes the previous combined/adaptive retrieval numbers non-canonical for this notebook.

Therefore:

```text
Previous reported retrieval numbers
    -> keep as historical observations
    -> do NOT treat them as the final V2 blocker benchmark

Corrected retriever
    -> rerun first
    -> then test the focused address/transliteration additions
```

The full-union observation from the separate experiment remains useful as an independent diagnostic, but the corrected notebook benchmark is the one we should use for the next decision gate.


In [23]:
# 21A. CORRECT THE ASCII NAME QUERY PATH

def retrieve_one_corrected(row, indexes, retrieval_top_k=None):
    """
    Original V2 blocker, corrected:
    rare_name_ascii uses name_token_freq_ascii.
    If retrieval_top_k is None, return the full union.
    """
    evidence = defaultdict(lambda: {
        "score": 0.0,
        "hits": defaultdict(int),
    })

    def add_hits(index_name, key, weight):
        if not key:
            return
        for rid in indexes[index_name].get(key, ()):
            evidence[rid]["score"] += weight
            evidence[rid]["hits"][index_name] += 1

    # Exact name
    if row["name_native"]:
        add_hits("exact_native_name", row["name_native"], BLOCK_WEIGHTS["exact_native_name"])

    if row["name_ascii"]:
        add_hits("exact_ascii_name", row["name_ascii"], BLOCK_WEIGHTS["exact_ascii_name"])

    # Rare name tokens
    for t in rare_keys(
        row["name_tokens_native"],
        name_token_freq_native,
        MAX_NAME_POSTING,
        RARE_NAME_KEYS,
    ):
        add_hits("rare_name_native", t, BLOCK_WEIGHTS["rare_name_native"])

    for t in rare_keys(
        row["name_tokens_ascii"],
        name_token_freq_ascii,   # CORRECTED
        MAX_NAME_POSTING,
        RARE_NAME_KEYS,
    ):
        add_hits("rare_name_ascii", t, BLOCK_WEIGHTS["rare_name_ascii"])

    # Address anchors
    for t in rare_keys(
        row["addr_tokens_native"],
        addr_token_freq_native,
        MAX_ADDR_POSTING,
        RARE_ADDR_KEYS,
    ):
        add_hits("rare_addr_native", t, BLOCK_WEIGHTS["rare_addr_native"])

    for t in rare_keys(
        row["addr_tokens_ascii"],
        addr_token_freq_ascii,
        MAX_ADDR_POSTING,
        RARE_ADDR_KEYS,
    ):
        add_hits("rare_addr_ascii", t, BLOCK_WEIGHTS["rare_addr_ascii"])

    # Postal
    for p in row["postal_native"]:
        if postal_freq.get(p, 10**12) <= MAX_ADDR_POSTING:
            add_hits("postal", p, BLOCK_WEIGHTS["postal"])

    # Prefix
    p4 = row["name_prefix4"]
    if p4 and prefix4_freq.get(p4, 10**12) <= MAX_PREFIX_POSTING:
        add_hits("prefix4", p4, BLOCK_WEIGHTS["prefix4"])

    # Name q-grams
    for q in rare_keys(
        row["name_qgrams"],
        qgram_freq,
        MAX_QGRAM_POSTING,
        RARE_QGRAM_KEYS,
    ):
        add_hits("qgram", q, BLOCK_WEIGHTS["qgram"])

    ranked = sorted(
        evidence.items(),
        key=lambda kv: (
            kv[1]["score"],
            sum(kv[1]["hits"].values()),
        ),
        reverse=True,
    )

    if retrieval_top_k is None:
        selected = ranked
    else:
        selected = ranked[:retrieval_top_k]

    return [
        {
            "cand_rid": rid,
            "block_score": info["score"],
            "block_hits": dict(info["hits"]),
        }
        for rid, info in selected
    ]


def run_retrieval_custom(s1_df, indexes, retrieval_top_k=None):
    rows = []
    t0 = time.time()

    for _, row in s1_df.iterrows():
        for hit in retrieve_one_corrected(
            row,
            indexes,
            retrieval_top_k=retrieval_top_k,
        ):
            hit["s1_rid"] = int(row["s1_rid"])
            rows.append(hit)

    elapsed = time.time() - t0
    out = pd.DataFrame(rows)

    if not out.empty:
        out = out[["s1_rid", "cand_rid", "block_score", "block_hits"]]

    return out, elapsed


corrected_union, corrected_union_time = run_retrieval_custom(
    S1,
    INDEX,
    retrieval_top_k=None,
)

print("Corrected original union rows:", len(corrected_union))
print("Corrected union runtime:", round(corrected_union_time, 3), "sec")
print(evaluate_candidate_recall(corrected_union, k_values=[10, 20, 50, 100, 150]))


Corrected original union rows: 1493372
Corrected union runtime: 6.631 sec
     K  macro_candidate_recall  any_hit_rate  avg_candidates  \
0   10                0.925227      0.991032          9.9844   
1   20                0.937269      0.992192         19.8965   
2   50                0.955081      0.995569         48.5563   
3  100                0.972070      0.997362         89.0367   
4  150                0.981178      0.998839        116.8659   

   median_candidates  p95_candidates  
0               10.0            10.0  
1               20.0            20.0  
2               50.0            50.0  
3              100.0           100.0  
4              136.0           150.0  


# 22. Squashed transliteration + address conjunction experiment

The failure analysis points to two concrete weaknesses:

1. Some Indic-script names become space-separated transliterations after `unidecode`, so normal tokenization breaks the name into unusably small pieces.
2. Several hard matches share a **number + meaningful address anchor** or **two meaningful address anchors**, even when the business names are very weak.

We test only these targeted additions:

```text
A. name_squashed
   "l i f p r d y..." -> "lifprdy..."
   used for exact/rare q-gram retrieval

B. address number + token
   "201|mumbai"
   "307|delhi"
   "176|pune"

C. address token + token
   "pune|maharashtra"
   "bangalore|karnataka"
   "new|madison"
```

No more generic name+address composites are added.


In [24]:
# 22A. DERIVE SQUASHED NAME REPRESENTATIONS

def squash_ascii(text):
    text = safe_text(text)
    if not text:
        return ""
    return re.sub(r"[^a-z0-9]", "", text)


def squashed_qgrams(text, n=3):
    x = squash_ascii(text)
    if len(x) < n:
        return []
    return list(dict.fromkeys(x[i:i+n] for i in range(len(x)-n+1)))


for df in (S1, CAND):
    df["name_squashed"] = df["name_ascii"].map(squash_ascii)
    df["name_squashed_qgrams"] = df["name_squashed"].map(squashed_qgrams)

squashed_name_freq = Counter(
    x for x in CAND["name_squashed"] if x
)
squashed_qgram_freq = count_postings(
    CAND["name_squashed_qgrams"]
)

print("Unique squashed names:", len(squashed_name_freq))
print("Unique squashed q-grams:", len(squashed_qgram_freq))


Unique squashed names: 47919
Unique squashed q-grams: 15435


In [25]:
# 22B. BUILD TARGETED ADDRESS-COMPOSITE INDEXES

MAX_COMPOSITE_POSTING = 100
MAX_ADDR_TOKENS_FOR_PAIRS = 6
MAX_ADDR_NUMS_FOR_PAIRS = 4

def meaningful_addr_tokens(row, token_freq):
    vals = set(row["addr_tokens_ascii"])
    vals = [
        t for t in vals
        if token_freq.get(t, 10**12) <= MAX_ADDR_POSTING
    ]
    vals.sort(key=lambda t: (token_freq.get(t, 10**12), -len(t), t))
    return vals[:MAX_ADDR_TOKENS_FOR_PAIRS]


def make_addr_number_keys(row):
    nums = row["numbers_native"][:MAX_ADDR_NUMS_FOR_PAIRS]
    toks = meaningful_addr_tokens(row, addr_token_freq_ascii)
    return [
        f"{n}|{t}"
        for n in nums
        for t in toks
        if n and t
    ]


def make_addr_pair_keys(row):
    toks = meaningful_addr_tokens(row, addr_token_freq_ascii)
    return [
        f"{a}|{b}"
        for i, a in enumerate(toks)
        for b in toks[i+1:]
    ]


CAND["addr_num_keys"] = CAND.apply(make_addr_number_keys, axis=1)
CAND["addr_pair_keys"] = CAND.apply(make_addr_pair_keys, axis=1)

S1["addr_num_keys"] = S1.apply(make_addr_number_keys, axis=1)
S1["addr_pair_keys"] = S1.apply(make_addr_pair_keys, axis=1)

addr_num_freq = count_postings(CAND["addr_num_keys"])
addr_pair_freq = count_postings(CAND["addr_pair_keys"])

TARGET_INDEX = {
    "squashed_exact": defaultdict(set),
    "squashed_qgram": defaultdict(set),
    "addr_num": defaultdict(set),
    "addr_pair": defaultdict(set),
}

for _, row in CAND.iterrows():
    rid = int(row["cand_rid"])

    if (
        row["name_squashed"]
        and squashed_name_freq.get(row["name_squashed"], 10**12)
        <= MAX_COMPOSITE_POSTING
    ):
        TARGET_INDEX["squashed_exact"][row["name_squashed"]].add(rid)

    for q in rare_keys(
        row["name_squashed_qgrams"],
        squashed_qgram_freq,
        MAX_COMPOSITE_POSTING,
        3,
    ):
        TARGET_INDEX["squashed_qgram"][q].add(rid)

    for key in row["addr_num_keys"]:
        if addr_num_freq.get(key, 10**12) <= MAX_COMPOSITE_POSTING:
            TARGET_INDEX["addr_num"][key].add(rid)

    for key in row["addr_pair_keys"]:
        if addr_pair_freq.get(key, 10**12) <= MAX_COMPOSITE_POSTING:
            TARGET_INDEX["addr_pair"][key].add(rid)

for name, d in TARGET_INDEX.items():
    print(f"{name:18s} keys={len(d):7d}")

print("addr_num total usable keys:", sum(
    addr_num_freq.get(k, 10**12) <= MAX_COMPOSITE_POSTING
    for k in addr_num_freq
))
print("addr_pair total usable keys:", sum(
    addr_pair_freq.get(k, 10**12) <= MAX_COMPOSITE_POSTING
    for k in addr_pair_freq
))


squashed_exact     keys=  47919
squashed_qgram     keys=  13426
addr_num           keys= 109930
addr_pair          keys= 157093
addr_num total usable keys: 109930
addr_pair total usable keys: 157093


In [26]:
# 22C. TARGETED RETRIEVAL OVER THE ORIGINAL UNION
#
# This measures marginal recall before any Top-K compression.

TARGET_WEIGHTS = {
    "squashed_exact": 8.0,
    "squashed_qgram": 1.25,
    "addr_num": 3.5,
    "addr_pair": 3.0,
}

def retrieve_target_additions(row):
    evidence = defaultdict(lambda: {
        "score": 0.0,
        "hits": defaultdict(int),
    })

    def add(index_name, key):
        if not key:
            return
        for rid in TARGET_INDEX[index_name].get(key, ()):
            evidence[rid]["score"] += TARGET_WEIGHTS[index_name]
            evidence[rid]["hits"][index_name] += 1

    add("squashed_exact", row["name_squashed"])

    for q in rare_keys(
        row["name_squashed_qgrams"],
        squashed_qgram_freq,
        MAX_COMPOSITE_POSTING,
        3,
    ):
        add("squashed_qgram", q)

    for key in row["addr_num_keys"]:
        if addr_num_freq.get(key, 10**12) <= MAX_COMPOSITE_POSTING:
            add("addr_num", key)

    for key in row["addr_pair_keys"]:
        if addr_pair_freq.get(key, 10**12) <= MAX_COMPOSITE_POSTING:
            add("addr_pair", key)

    return evidence


target_rows = []

for _, row in S1.iterrows():
    base = defaultdict(lambda: {
        "score": 0.0,
        "hits": defaultdict(int),
    })

    # Original corrected union.
    for hit in retrieve_one_corrected(
        row,
        INDEX,
        retrieval_top_k=None,
    ):
        base[hit["cand_rid"]]["score"] += float(hit["block_score"])
        base[hit["cand_rid"]]["hits"].update(hit["block_hits"])

    # Targeted additions.
    additions = retrieve_target_additions(row)

    for rid, info in additions.items():
        base[rid]["score"] += info["score"]
        for fam, n in info["hits"].items():
            base[rid]["hits"][fam] += n

    for rid, info in base.items():
        target_rows.append({
            "s1_rid": int(row["s1_rid"]),
            "cand_rid": int(rid),
            "block_score": float(info["score"]),
            "block_hits": dict(info["hits"]),
        })

target_union = pd.DataFrame(target_rows)

print("Targeted union rows:", len(target_union))
print(evaluate_candidate_recall(target_union, k_values=[10, 20, 50, 100, 150]))


Targeted union rows: 1641505
     K  macro_candidate_recall  any_hit_rate  avg_candidates  \
0   10                0.964423      0.996835          9.9930   
1   20                0.969734      0.997046         19.9419   
2   50                0.978938      0.997679         48.9897   
3  100                0.986409      0.998734         91.6742   
4  150                0.990023      0.999156        123.1311   

   median_candidates  p95_candidates  
0               10.0            10.0  
1               20.0            20.0  
2               50.0            50.0  
3              100.0           100.0  
4              150.0           150.0  


In [27]:
# 22D. MARGINAL CONTRIBUTION OF EACH NEW FAMILY

def run_one_addition_family(family_name):
    rows = []

    for _, row in S1.iterrows():
        ev = retrieve_target_additions(row)
        for rid, info in ev.items():
            if family_name not in info["hits"]:
                continue
            rows.append({
                "s1_rid": int(row["s1_rid"]),
                "cand_rid": int(rid),
                "block_score": float(info["score"]),
                "block_hits": {family_name: int(info["hits"][family_name])},
            })

    return pd.DataFrame(rows)


target_family_results = []

for family in ["squashed_exact", "squashed_qgram", "addr_num", "addr_pair"]:
    tmp = run_one_addition_family(family)
    metrics = evaluate_candidate_recall(tmp, k_values=[50])

    if metrics.empty:
        row = {
            "family": family,
            "macro_recall@50": 0.0,
            "any_hit@50": 0.0,
            "avg_candidates@50": 0.0,
        }
    else:
        m = metrics.iloc[0]
        row = {
            "family": family,
            "macro_recall@50": float(m["macro_candidate_recall"]),
            "any_hit@50": float(m["any_hit_rate"]),
            "avg_candidates@50": float(m["avg_candidates"]),
        }

    target_family_results.append(row)

target_family_results = pd.DataFrame(target_family_results)
target_family_results


,family,macro_recall@50,any_hit@50,avg_candidates@50
0,squashed_exact,0.272920,0.626081,1.0202
1,squashed_qgram,0.651113,0.789618,33.1864
2,addr_num,0.683979,0.845854,14.6940
3,addr_pair,0.834821,0.952100,4.9515


# 23. Decision gate after the corrected benchmark

Use the outputs above in this order:

```text
1. corrected_union
2. targeted_union
3. marginal family table
```

Interpretation:

```text
If corrected_union recall changes materially:
    the old retrieval benchmark was affected by the ASCII-frequency bug.

If targeted_union adds recall to corrected_union:
    keep only the contributing family/families.

If targeted_union adds no recall:
    stop adding blockers and move to candidate ranking/compression.

If address composites add recall:
    address-heavy hard cases are a real blocking signal.

If squashed transliteration adds recall:
    multilingual name representation is a real blocking signal.

Do NOT move to the 12M-row production implementation until this gate is passed.


In [28]:
# 23A. BUILD ADDITIONS FOR A SELECTED SET OF NEW BLOCKING FAMILIES

def build_family_additions(families):
    rows = []

    for _, row in S1.iterrows():
        additions = retrieve_target_additions(row)

        for rid, info in additions.items():
            selected_hits = {
                fam: int(info["hits"][fam])
                for fam in families
                if fam in info["hits"]
            }

            if not selected_hits:
                continue

            add_score = sum(
                TARGET_WEIGHTS[fam] * n
                for fam, n in selected_hits.items()
            )

            rows.append({
                "s1_rid": int(row["s1_rid"]),
                "cand_rid": int(rid),
                "add_score": float(add_score),
            })

    if not rows:
        return pd.DataFrame(
            columns=["s1_rid", "cand_rid", "add_score"]
        )

    out = pd.DataFrame(rows)

    # Multiple keys from the same family may hit the same candidate.
    out = (
        out.groupby(["s1_rid", "cand_rid"], as_index=False)["add_score"]
        .sum()
    )

    return out

In [29]:
# 23B. MERGE BASE UNION + SELECTED NEW FAMILIES

def make_ablation_union(families):
    base = corrected_union[
        ["s1_rid", "cand_rid", "block_score"]
    ].copy()

    additions = build_family_additions(families)

    if additions.empty:
        return base

    merged = base.merge(
        additions,
        on=["s1_rid", "cand_rid"],
        how="outer",
    )

    merged["block_score"] = (
        merged["block_score"].fillna(0.0)
        + merged["add_score"].fillna(0.0)
    )

    merged = merged[
        ["s1_rid", "cand_rid", "block_score"]
    ]

    return merged

In [30]:
# 23C. ABLATION EXPERIMENT

ABLATIONS = {
    "baseline": [],
    "baseline + addr_pair": ["addr_pair"],
    "baseline + addr_num": ["addr_num"],
    "baseline + squashed_qgram": ["squashed_qgram"],
    "baseline + squashed_exact": ["squashed_exact"],
    "addr_pair + addr_num": ["addr_pair", "addr_num"],
    "addr_pair + squashed_qgram": ["addr_pair", "squashed_qgram"],
    "all_new": [
        "squashed_exact",
        "squashed_qgram",
        "addr_num",
        "addr_pair",
    ],
}

ablation_results = []

for name, families in ABLATIONS.items():

    t0 = time.time()

    if name == "baseline":
        df = corrected_union[
            ["s1_rid", "cand_rid", "block_score"]
        ].copy()
    else:
        df = make_ablation_union(families)

    metrics = evaluate_candidate_recall(
        df,
        k_values=[20, 50, 100, 150],
    )

    elapsed = time.time() - t0

    for _, m in metrics.iterrows():
        ablation_results.append({
            "experiment": name,
            "K": int(m["K"]),
            "macro_recall": float(m["macro_candidate_recall"]),
            "any_hit": float(m["any_hit_rate"]),
            "avg_candidates": float(m["avg_candidates"]),
            "p95_candidates": float(m["p95_candidates"]),
            "runtime_sec": round(elapsed, 3),
        })

ablation_results = pd.DataFrame(ablation_results)

ablation_results

,experiment,K,macro_recall,any_hit,avg_candidates,p95_candidates,runtime_sec
0,baseline,20,0.937269,0.992192,19.8965,20.0,0.970
1,baseline,50,0.955081,0.995569,48.5563,50.0,0.970
2,baseline,100,0.972070,0.997362,89.0367,100.0,0.970
3,baseline,150,0.981178,0.998839,116.8659,150.0,0.970
4,baseline + addr_pair,20,0.960344,0.996202,19.9097,20.0,2.063
5,baseline + addr_pair,50,0.970597,0.997046,48.6461,50.0,2.063
6,baseline + addr_pair,100,0.981427,0.998312,89.4460,100.0,2.063
7,baseline + addr_pair,150,0.986365,0.999050,117.6917,150.0,2.063
8,baseline + addr_num,20,0.958323,0.995780,19.9337,20.0,2.541
9,baseline + addr_num,50,0.970950,0.997468,48.9287,50.0,2.541


In [31]:
# 23D. EASY-TO-READ COMPARISON

ablation_pivot = (
    ablation_results
    .pivot(
        index="experiment",
        columns="K",
        values="macro_recall"
    )
)

print("=== MACRO RECALL ===")
display(ablation_pivot)

print("\n=== K=50 DETAILS ===")

k50 = (
    ablation_results[
        ablation_results["K"] == 50
    ][
        [
            "experiment",
            "macro_recall",
            "any_hit",
            "avg_candidates",
            "p95_candidates",
        ]
    ]
    .sort_values("macro_recall", ascending=False)
)

display(k50)

=== MACRO RECALL ===


K,20,50,100,150
experiment,,,,
addr_pair + addr_num,0.968443,0.978232,0.986028,0.989622
addr_pair + squashed_qgram,0.962081,0.971767,0.982013,0.986470
all_new,0.969978,0.979005,0.986582,0.989800
baseline,0.937269,0.955081,0.972070,0.981178
baseline + addr_num,0.958323,0.970950,0.981586,0.987067
baseline + addr_pair,0.960344,0.970597,0.981427,0.986365
baseline + squashed_exact,0.937969,0.955394,0.972158,0.981079
baseline + squashed_qgram,0.935065,0.957178,0.973471,0.981349



=== K=50 DETAILS ===


,experiment,macro_recall,any_hit,avg_candidates,p95_candidates
29,all_new,0.979005,0.997784,48.9897,50.0
21,addr_pair + addr_num,0.978232,0.997995,48.9897,50.0
25,addr_pair + squashed_qgram,0.971767,0.997046,48.6461,50.0
9,baseline + addr_num,0.970950,0.997468,48.9287,50.0
5,baseline + addr_pair,0.970597,0.997046,48.6461,50.0
13,baseline + squashed_qgram,0.957178,0.995358,48.5563,50.0
17,baseline + squashed_exact,0.955394,0.995463,48.5563,50.0
1,baseline,0.955081,0.995569,48.5563,50.0


In [32]:
# 23E. HOW MANY TRUE PAIRS EACH FAMILY ADDS BEYOND THE BASE UNION?

base_pairs = {
    (int(r.s1_rid), int(r.cand_rid))
    for r in corrected_union.itertuples()
}

marginal_rows = []

for family in [
    "squashed_exact",
    "squashed_qgram",
    "addr_num",
    "addr_pair",
]:

    additions = build_family_additions([family])

    added = [
        (int(r.s1_rid), int(r.cand_rid))
        for r in additions.itertuples()
        if (int(r.s1_rid), int(r.cand_rid)) not in base_pairs
    ]

    added_set = set(added)

    # Ground-truth pairs represented by the new candidates
    added_truth = 0
    for s1_rid, cand_rid in added_set:
        s1_id = s1_entity_by_rid[s1_rid]
        cand_id = cand_entity_by_rid[cand_rid]

        if cand_id in sample_truth.get(s1_id, set()):
            added_truth += 1

    marginal_rows.append({
        "family": family,
        "new_candidate_pairs": len(added_set),
        "new_true_pairs": added_truth,
    })

marginal_df = pd.DataFrame(marginal_rows)

display(marginal_df)

,family,new_candidate_pairs,new_true_pairs
0,squashed_exact,0,0
1,squashed_qgram,0,0
2,addr_num,132952,185
3,addr_pair,16014,126


In [33]:
# 24A. BOUNDED RANKING POOL
#
# Scalability principle:
# never run expensive pairwise similarity over the entire candidate universe.
# First compress using cheap block evidence.

RANKING_POOL_K = 150

t0 = time.time()

ranking_pool = (
    target_union
    .groupby("s1_rid", group_keys=False)
    .apply(
        lambda g: g.nlargest(
            RANKING_POOL_K,
            "block_score"
        )
    )
    .reset_index(drop=True)
)

pool_time = time.time() - t0

print("Ranking-pool rows:", len(ranking_pool))
print("Ranking-pool runtime:", round(pool_time, 3), "sec")

print(
    ranking_pool
    .groupby("s1_rid")
    .size()
    .describe(percentiles=[0.50, 0.90, 0.95, 0.99])
)

print("\nBlock-score recall inside ranking pool:")
print(
    evaluate_candidate_recall(
        ranking_pool,
        k_values=[50, 100, 150]
    )
)

/var/folders/j0/djnjpdcd5w16zyp2jwbkbm6r0000gn/T/ipykernel_20587/1713195285.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Ranking-pool rows: 1231311
Ranking-pool runtime: 5.747 sec
count    10000.000000
mean       123.131100
std         36.729533
min          4.000000
50%        150.000000
90%        150.000000
95%        150.000000
99%        150.000000
max        150.000000
dtype: float64

Block-score recall inside ranking pool:
     K  macro_candidate_recall  any_hit_rate  avg_candidates  \
0   50                0.978627      0.997679         48.9897   
1  100                0.985973      0.998628         91.6742   
2  150                0.989711      0.999050        123.1311   

   median_candidates  p95_candidates  
0               50.0            50.0  
1              100.0           100.0  
2              150.0           150.0  


In [34]:
# 24B. CHEAP RERANK ONLY THE BOUNDED POOL

t0 = time.time()

ranked_pool = cheap_rerank(
    ranking_pool,
    S1,
    CAND,
    final_k=RANKING_POOL_K,
)

cheap_time = time.time() - t0

print("Cheap-ranking runtime:", round(cheap_time, 3), "sec")
print("Ranked-pool rows:", len(ranked_pool))

Cheap-ranking runtime: 68.636 sec
Ranked-pool rows: 1231311


In [35]:
# 24C. COMPRESSION SWEEP

K_SWEEP = [20, 30, 40, 50, 60, 75, 100, 125, 150]

ranked_for_eval = ranked_pool[
    ["s1_rid", "cand_rid", "cheap_score"]
].copy()

ranked_for_eval["block_score"] = ranked_for_eval["cheap_score"]
ranked_for_eval.drop(columns=["cheap_score"], inplace=True)

compression_table = evaluate_candidate_recall(
    ranked_for_eval,
    k_values=K_SWEEP
)

compression_table

,K,macro_candidate_recall,any_hit_rate,avg_candidates,median_candidates,p95_candidates
0,20,0.962550,0.995674,19.9419,20.0,20.0
1,30,0.969238,0.996729,29.7891,30.0,30.0
2,40,0.973957,0.997468,39.4920,40.0,40.0
3,50,0.977248,0.997679,48.9897,50.0,50.0
4,60,0.980135,0.998101,58.2403,60.0,60.0
5,75,0.982673,0.998312,71.5413,75.0,75.0
6,100,0.986057,0.998734,91.6742,100.0,100.0
7,125,0.988047,0.998839,108.9052,125.0,125.0
8,150,0.989711,0.999050,123.1311,150.0,150.0


In [36]:
# 24D. BLOCK SCORE vs CHEAP SCORE

block_pool_metrics = evaluate_candidate_recall(
    ranking_pool,
    k_values=K_SWEEP
).copy()

block_pool_metrics["ranking"] = "block_score"
block_pool_metrics["K"] = block_pool_metrics["K"].astype(int)

cheap_pool_metrics = compression_table.copy()
cheap_pool_metrics["ranking"] = "cheap_score"

comparison = pd.concat(
    [block_pool_metrics, cheap_pool_metrics],
    ignore_index=True
)

display(
    comparison[
        [
            "ranking",
            "K",
            "macro_candidate_recall",
            "any_hit_rate",
            "avg_candidates",
            "p95_candidates",
        ]
    ]
)

,ranking,K,macro_candidate_recall,any_hit_rate,avg_candidates,p95_candidates
0,block_score,20,0.969915,0.996940,19.9419,20.0
1,block_score,30,0.973542,0.997362,29.7891,30.0
2,block_score,40,0.976459,0.997362,39.4920,40.0
3,block_score,50,0.978627,0.997679,48.9897,50.0
4,block_score,60,0.980656,0.997890,58.2403,60.0
5,block_score,75,0.983039,0.998417,71.5413,75.0
6,block_score,100,0.985973,0.998628,91.6742,100.0
7,block_score,125,0.987918,0.998839,108.9052,125.0
8,block_score,150,0.989711,0.999050,123.1311,150.0
9,cheap_score,20,0.962550,0.995674,19.9419,20.0


In [37]:
# 24E. FOCUS ON THE PRACTICAL RANGE

decision_table = (
    comparison[
        comparison["K"].isin([20, 30, 40, 50, 60, 75, 100])
    ]
    .pivot(
        index="K",
        columns="ranking",
        values="macro_candidate_recall"
    )
)

print("Macro candidate recall:")
display(decision_table)

print("\nCheap-score ranking details:")
display(
    cheap_pool_metrics[
        cheap_pool_metrics["K"].isin([20, 30, 40, 50, 60, 75, 100])
    ][
        [
            "K",
            "macro_candidate_recall",
            "any_hit_rate",
            "avg_candidates",
            "p95_candidates",
        ]
    ]
)

Macro candidate recall:


ranking,block_score,cheap_score
K,,
20,0.969915,0.962550
30,0.973542,0.969238
40,0.976459,0.973957
50,0.978627,0.977248
60,0.980656,0.980135
75,0.983039,0.982673
100,0.985973,0.986057



Cheap-score ranking details:


,K,macro_candidate_recall,any_hit_rate,avg_candidates,p95_candidates
0,20,0.962550,0.995674,19.9419,20.0
1,30,0.969238,0.996729,29.7891,30.0
2,40,0.973957,0.997468,39.4920,40.0
3,50,0.977248,0.997679,48.9897,50.0
4,60,0.980135,0.998101,58.2403,60.0
5,75,0.982673,0.998312,71.5413,75.0
6,100,0.986057,0.998734,91.6742,100.0


In [38]:
# 24F. RECALL LOSS AS WE COMPRESS

cheap_only = cheap_pool_metrics.copy()

base_150 = float(
    cheap_only.loc[
        cheap_only["K"] == 150,
        "macro_candidate_recall"
    ].iloc[0]
)

cheap_only["loss_vs_K150"] = (
    base_150 - cheap_only["macro_candidate_recall"]
)

display(
    cheap_only[
        [
            "K",
            "macro_candidate_recall",
            "loss_vs_K150",
            "any_hit_rate",
            "avg_candidates",
        ]
    ]
)

,K,macro_candidate_recall,loss_vs_K150,any_hit_rate,avg_candidates
0,20,0.962550,0.027161,0.995674,19.9419
1,30,0.969238,0.020473,0.996729,29.7891
2,40,0.973957,0.015754,0.997468,39.4920
3,50,0.977248,0.012463,0.997679,48.9897
4,60,0.980135,0.009576,0.998101,58.2403
5,75,0.982673,0.007038,0.998312,71.5413
6,100,0.986057,0.003654,0.998734,91.6742
7,125,0.988047,0.001665,0.998839,108.9052
8,150,0.989711,0.000000,0.999050,123.1311


In [39]:
# 25A. DIVERSITY-AWARE CANDIDATE COMPRESSION
#
# Reserve a small number of candidates from important blocking families,
# then fill the remaining budget using global block_score.

def diverse_compress(
    candidate_df,
    final_k=50,
    reserve_addr_pair=2,
    reserve_addr_num=1,
):
    if candidate_df.empty:
        return candidate_df.copy()

    output = []

    for s1_rid, group in candidate_df.groupby("s1_rid", sort=False):

        g = group.sort_values(
            "block_score",
            ascending=False
        ).copy()

        selected_ids = set()

        # Protect strongest address-pair evidence.
        if reserve_addr_pair > 0:
            mask = g["block_hits"].map(
                lambda h: isinstance(h, dict) and "addr_pair" in h
            )

            selected_ids.update(
                g.loc[mask, "cand_rid"]
                .head(reserve_addr_pair)
                .astype(int)
                .tolist()
            )

        # Protect a small number of address-number candidates.
        if reserve_addr_num > 0:
            mask = g["block_hits"].map(
                lambda h: isinstance(h, dict) and "addr_num" in h
            )

            selected_ids.update(
                g.loc[mask, "cand_rid"]
                .head(reserve_addr_num)
                .astype(int)
                .tolist()
            )

        # Never exceed final_k.
        if len(selected_ids) > final_k:
            selected_ids = set(
                g[
                    g["cand_rid"].isin(selected_ids)
                ]
                .head(final_k)["cand_rid"]
                .astype(int)
            )

        remaining_slots = final_k - len(selected_ids)

        remaining = g[
            ~g["cand_rid"].isin(selected_ids)
        ]

        fill = remaining.head(remaining_slots)

        selected = g[
            g["cand_rid"].isin(selected_ids)
        ]

        result = pd.concat(
            [selected, fill],
            ignore_index=True
        )

        # Sort final candidates deterministically.
        result = result.sort_values(
            "block_score",
            ascending=False
        ).head(final_k)

        output.append(result)

    return pd.concat(
        output,
        ignore_index=True
    )

In [40]:
# 25B. TEST RESERVATION STRATEGIES

DIVERSITY_CONFIGS = {
    "baseline": (0, 0),

    "pair1": (1, 0),
    "pair2": (2, 0),
    "pair3": (3, 0),
    "pair5": (5, 0),

    "pair2_num1": (2, 1),
    "pair3_num1": (3, 1),
    "pair3_num2": (3, 2),
}

diversity_results = []

for name, (r_pair, r_num) in DIVERSITY_CONFIGS.items():

    t0 = time.time()

    compressed = diverse_compress(
        target_union,
        final_k=50,
        reserve_addr_pair=r_pair,
        reserve_addr_num=r_num,
    )

    elapsed = time.time() - t0

    metrics = evaluate_candidate_recall(
        compressed,
        k_values=[20, 30, 40, 50],
    )

    for _, m in metrics.iterrows():
        diversity_results.append({
            "config": name,
            "reserve_addr_pair": r_pair,
            "reserve_addr_num": r_num,
            "K": int(m["K"]),
            "macro_recall": float(
                m["macro_candidate_recall"]
            ),
            "any_hit": float(
                m["any_hit_rate"]
            ),
            "avg_candidates": float(
                m["avg_candidates"]
            ),
            "runtime_sec": round(elapsed, 3),
        })

diversity_results = pd.DataFrame(
    diversity_results
)

display(diversity_results)

,config,reserve_addr_pair,reserve_addr_num,K,macro_recall,any_hit,avg_candidates,runtime_sec
0,baseline,0,0,20,0.969741,0.997046,19.9419,4.241
1,baseline,0,0,30,0.973461,0.997257,29.7891,4.241
2,baseline,0,0,40,0.976516,0.997362,39.4920,4.241
3,baseline,0,0,50,0.978938,0.997679,48.9897,4.241
4,pair1,1,0,20,0.969741,0.997046,19.9419,8.280
5,pair1,1,0,30,0.973439,0.997257,29.7891,8.280
6,pair1,1,0,40,0.976516,0.997362,39.4920,8.280
7,pair1,1,0,50,0.979184,0.998101,48.9897,8.280
8,pair2,2,0,20,0.969749,0.997046,19.9419,5.550
9,pair2,2,0,30,0.973527,0.997257,29.7891,5.550


In [41]:
# 25C. COMPARE DIVERSITY STRATEGIES

comparison_25 = (
    diversity_results[
        diversity_results["K"].isin([20, 30, 40, 50])
    ]
    .sort_values(
        ["K", "macro_recall"],
        ascending=[True, False]
    )
)

display(
    comparison_25[
        [
            "config",
            "reserve_addr_pair",
            "reserve_addr_num",
            "K",
            "macro_recall",
            "any_hit",
            "avg_candidates",
        ]
    ]
)

,config,reserve_addr_pair,reserve_addr_num,K,macro_recall,any_hit,avg_candidates
16,pair5,5,0,20,0.969989,0.997046,19.9419
24,pair3_num1,3,1,20,0.969985,0.997046,19.9419
28,pair3_num2,3,2,20,0.969929,0.997046,19.9419
12,pair3,3,0,20,0.969902,0.997046,19.9419
20,pair2_num1,2,1,20,0.969797,0.997046,19.9419
8,pair2,2,0,20,0.969749,0.997046,19.9419
0,baseline,0,0,20,0.969741,0.997046,19.9419
4,pair1,1,0,20,0.969741,0.997046,19.9419
29,pair3_num2,3,2,30,0.973633,0.997257,29.7891
17,pair5,5,0,30,0.973628,0.997257,29.7891


In [42]:
# 25D. FOCUS ON K=50

k50_diversity = (
    diversity_results[
        diversity_results["K"] == 50
    ]
    .sort_values(
        "macro_recall",
        ascending=False
    )
)

display(k50_diversity)

,config,reserve_addr_pair,reserve_addr_num,K,macro_recall,any_hit,avg_candidates,runtime_sec
19,pair5,5,0,50,0.980948,0.998101,48.9897,5.460
31,pair3_num2,3,2,50,0.980740,0.998312,48.9897,6.578
27,pair3_num1,3,1,50,0.980468,0.998312,48.9897,8.960
15,pair3,3,0,50,0.980397,0.998101,48.9897,5.666
23,pair2_num1,2,1,50,0.979949,0.998312,48.9897,6.676
11,pair2,2,0,50,0.979879,0.998101,48.9897,5.550
7,pair1,1,0,50,0.979184,0.998101,48.9897,8.280
3,baseline,0,0,50,0.978938,0.997679,48.9897,4.241


In [43]:
# 25E. HOW MUCH DOES ADDRESS PROTECTION RECOVER?

baseline_50 = diverse_compress(
    target_union,
    final_k=50,
    reserve_addr_pair=0,
    reserve_addr_num=0,
)

pair3_50 = diverse_compress(
    target_union,
    final_k=50,
    reserve_addr_pair=3,
    reserve_addr_num=0,
)

base_pairs = {
    (int(r.s1_rid), int(r.cand_rid))
    for r in baseline_50.itertuples()
}

pair_pairs = {
    (int(r.s1_rid), int(r.cand_rid))
    for r in pair3_50.itertuples()
}

new_pairs = pair_pairs - base_pairs

new_true = 0

for s1_rid, cand_rid in new_pairs:
    s1_id = s1_entity_by_rid[s1_rid]
    cand_id = cand_entity_by_rid[cand_rid]

    if cand_id in sample_truth.get(s1_id, set()):
        new_true += 1

print("New candidates introduced:", len(new_pairs))
print("New true pairs recovered:", new_true)

New candidates introduced: 432
New true pairs recovered: 34


In [44]:
# 26A. FAST DIVERSITY-AWARE COMPRESSION
#
# Lab experiment only.
# At full scale, the same logic will be implemented in the
# streaming/DuckDB candidate pipeline rather than iterating
# over millions of pandas groups.

def diverse_compress_fast(
    candidate_df,
    final_k=50,
    reserve_addr_pair=5,
):
    if candidate_df.empty:
        return candidate_df.copy()

    pieces = []

    for s1_rid, g in candidate_df.groupby("s1_rid", sort=False):

        g = g.sort_values(
            "block_score",
            ascending=False
        )

        is_pair = g["block_hits"].map(
            lambda h: (
                isinstance(h, dict)
                and "addr_pair" in h
            )
        )

        protected = g[is_pair].head(reserve_addr_pair)

        selected_ids = set(
            protected["cand_rid"].astype(int)
        )

        slots = final_k - len(selected_ids)

        if slots > 0:
            filler = g[
                ~g["cand_rid"].isin(selected_ids)
            ].head(slots)

            result = pd.concat(
                [protected, filler],
                ignore_index=True
            )
        else:
            result = protected.head(final_k)

        pieces.append(result)

    return pd.concat(
        pieces,
        ignore_index=True
    )

In [45]:
# 26B. RESERVATION-SIZE SWEEP

RESERVATIONS = [0, 2, 3, 4, 5, 6, 8, 10]

reservation_results = []

for r in RESERVATIONS:

    t0 = time.time()

    compressed = diverse_compress_fast(
        target_union,
        final_k=50,
        reserve_addr_pair=r,
    )

    elapsed = time.time() - t0

    metrics = evaluate_candidate_recall(
        compressed,
        k_values=[50]
    ).iloc[0]

    reservation_results.append({
        "reserve_addr_pair": r,
        "macro_recall": float(
            metrics["macro_candidate_recall"]
        ),
        "any_hit": float(
            metrics["any_hit_rate"]
        ),
        "avg_candidates": float(
            metrics["avg_candidates"]
        ),
        "runtime_sec": round(elapsed, 3),
    })

reservation_results = pd.DataFrame(
    reservation_results
)

display(reservation_results)

,reserve_addr_pair,macro_recall,any_hit,avg_candidates,runtime_sec
0,0,0.978938,0.997679,48.9897,6.117
1,2,0.979879,0.998101,48.9897,4.283
2,3,0.980397,0.998101,48.9897,6.733
3,4,0.980758,0.998101,48.9897,4.249
4,5,0.980948,0.998101,48.9897,4.181
5,6,0.981080,0.998101,48.9897,6.906
6,8,0.981147,0.998101,48.9897,3.980
7,10,0.981152,0.998101,48.9897,3.514


In [46]:
# 26C. K-SWEEP WITH THE CURRENT BEST RESERVATION

BEST_RESERVE = 5

k_diverse_results = []

compressed_150 = diverse_compress_fast(
    target_union,
    final_k=150,
    reserve_addr_pair=BEST_RESERVE,
)

metrics = evaluate_candidate_recall(
    compressed_150,
    k_values=[20, 30, 40, 50, 60, 75, 100, 125, 150]
)

k_diverse_results = metrics.copy()

display(
    k_diverse_results[
        [
            "K",
            "macro_candidate_recall",
            "any_hit_rate",
            "avg_candidates",
            "p95_candidates",
        ]
    ]
)

,K,macro_candidate_recall,any_hit_rate,avg_candidates,p95_candidates
0,20,0.969434,0.997046,19.9419,20.0
1,30,0.973005,0.997362,29.7891,30.0
2,40,0.976118,0.997468,39.4920,40.0
3,50,0.978531,0.997679,48.9897,50.0
4,60,0.980840,0.997995,58.2403,60.0
5,75,0.983242,0.998523,71.5413,75.0
6,100,0.986224,0.998839,91.6742,100.0
7,125,0.988139,0.999050,108.9052,125.0
8,150,0.990023,0.999156,123.1311,150.0


In [47]:
# 26D. DIRECT COMPARISON

ordinary = evaluate_candidate_recall(
    target_union,
    k_values=[20, 30, 40, 50, 60, 75, 100]
).copy()

ordinary["method"] = "ordinary_block_score"

diverse = evaluate_candidate_recall(
    compressed_150,
    k_values=[20, 30, 40, 50, 60, 75, 100]
).copy()

diverse["method"] = "diversity_addr_pair"

comparison_26 = pd.concat(
    [ordinary, diverse],
    ignore_index=True
)

display(
    comparison_26[
        [
            "method",
            "K",
            "macro_candidate_recall",
            "any_hit_rate",
            "avg_candidates",
        ]
    ]
)

,method,K,macro_candidate_recall,any_hit_rate,avg_candidates
0,ordinary_block_score,20,0.969734,0.997046,19.9419
1,ordinary_block_score,30,0.973545,0.997362,29.7891
2,ordinary_block_score,40,0.976852,0.997362,39.4920
3,ordinary_block_score,50,0.978938,0.997679,48.9897
4,ordinary_block_score,60,0.981272,0.998101,58.2403
5,ordinary_block_score,75,0.983682,0.998628,71.5413
6,ordinary_block_score,100,0.986409,0.998734,91.6742
7,diversity_addr_pair,20,0.969434,0.997046,19.9419
8,diversity_addr_pair,30,0.973005,0.997362,29.7891
9,diversity_addr_pair,40,0.976118,0.997468,39.4920


In [48]:
# 26E. CANDIDATE-VOLUME SUMMARY

print("Targeted union rows:", len(target_union))
print(
    "Targeted union avg/S1:",
    len(target_union) / len(S1)
)

for k in [40, 50, 60, 75, 100]:
    row = k_diverse_results[
        k_diverse_results["K"] == k
    ].iloc[0]

    print(
        f"K={k:3d} | "
        f"recall={row['macro_candidate_recall']:.6f} | "
        f"avg_candidates={row['avg_candidates']:.2f}"
    )

Targeted union rows: 1641505
Targeted union avg/S1: 164.1505
K= 40 | recall=0.976118 | avg_candidates=39.49
K= 50 | recall=0.978531 | avg_candidates=48.99
K= 60 | recall=0.980840 | avg_candidates=58.24
K= 75 | recall=0.983242 | avg_candidates=71.54
K=100 | recall=0.986224 | avg_candidates=91.67


In [49]:
# 27A. TARGETED UNION RECALL CEILING
#
# This separates:
#   1. candidates missing from blocking entirely
#   2. candidates that are present but lost during compression

CEILING_K = [50, 100, 150, 200, 300, 500]

target_ceiling = evaluate_candidate_recall(
    target_union,
    k_values=CEILING_K,
)

display(
    target_ceiling[
        [
            "K",
            "macro_candidate_recall",
            "any_hit_rate",
            "avg_candidates",
            "p95_candidates",
        ]
    ]
)

,K,macro_candidate_recall,any_hit_rate,avg_candidates,p95_candidates
0,50,0.978938,0.997679,48.9897,50.0
1,100,0.986409,0.998734,91.6742,100.0
2,150,0.990023,0.999156,123.1311,150.0
3,200,0.991597,0.999261,143.5027,200.0
4,300,0.992620,0.999367,160.8637,300.0
5,500,0.992726,0.999367,164.1444,319.0


In [50]:
# 27B. EXTRACT FAMILY HIT COUNTS ONCE
#
# Avoid repeatedly parsing dictionaries during every ranking experiment.

target_rank_base = target_union[
    ["s1_rid", "cand_rid", "block_score", "block_hits"]
].copy()

target_rank_base["addr_pair_hits"] = np.fromiter(
    (
        int(h.get("addr_pair", 0))
        if isinstance(h, dict)
        else 0
        for h in target_rank_base["block_hits"]
    ),
    dtype=np.int16,
    count=len(target_rank_base),
)

target_rank_base["addr_num_hits"] = np.fromiter(
    (
        int(h.get("addr_num", 0))
        if isinstance(h, dict)
        else 0
        for h in target_rank_base["block_hits"]
    ),
    dtype=np.int16,
    count=len(target_rank_base),
)

target_rank_base.drop(
    columns=["block_hits"],
    inplace=True,
)

target_rank_base.head()

,s1_rid,cand_rid,block_score,addr_pair_hits,addr_num_hits
0,0,42474,78.25,15,0
1,0,11795,58.25,10,0
2,0,4147,67.50,15,0
3,0,7631,10.25,0,0
4,0,54663,8.00,0,0


In [51]:
# 27C. ADDR_PAIR SOFT-BOOST SWEEP

PAIR_BOOSTS = [
    0.0,
    0.5,
    1.0,
    1.5,
    2.0,
    3.0,
    4.0,
    5.0,
    7.0,
]

pair_boost_results = []

base_score = target_rank_base["block_score"].to_numpy(
    dtype=np.float32
)

pair_hits = target_rank_base["addr_pair_hits"].to_numpy(
    dtype=np.float32
)

for boost in PAIR_BOOSTS:

    ranked = pd.DataFrame({
        "s1_rid": target_rank_base["s1_rid"].to_numpy(),
        "cand_rid": target_rank_base["cand_rid"].to_numpy(),
        "block_score": base_score + boost * pair_hits,
    })

    metrics = evaluate_candidate_recall(
        ranked,
        k_values=[50],
    ).iloc[0]

    pair_boost_results.append({
        "pair_boost": boost,
        "macro_recall": float(
            metrics["macro_candidate_recall"]
        ),
        "any_hit": float(
            metrics["any_hit_rate"]
        ),
        "avg_candidates": float(
            metrics["avg_candidates"]
        ),
    })

pair_boost_results = pd.DataFrame(
    pair_boost_results
)

display(
    pair_boost_results.sort_values(
        "macro_recall",
        ascending=False
    )
)

,pair_boost,macro_recall,any_hit,avg_candidates
8,7.0,0.980861,0.998101,48.9897
7,5.0,0.980835,0.998101,48.9897
6,4.0,0.980654,0.998101,48.9897
5,3.0,0.980645,0.998101,48.9897
3,1.5,0.980576,0.998101,48.9897
4,2.0,0.980568,0.998101,48.9897
2,1.0,0.980566,0.998101,48.9897
1,0.5,0.980109,0.997995,48.9897
0,0.0,0.978938,0.997679,48.9897


In [52]:
# 27D. BEST PAIR BOOST ACROSS MULTIPLE K VALUES

best_pair_boost = float(
    pair_boost_results.loc[
        pair_boost_results["macro_recall"].idxmax(),
        "pair_boost"
    ]
)

print("Best addr_pair boost:", best_pair_boost)

ranked_best = pd.DataFrame({
    "s1_rid": target_rank_base["s1_rid"].to_numpy(),
    "cand_rid": target_rank_base["cand_rid"].to_numpy(),
    "block_score": (
        base_score
        + best_pair_boost * pair_hits
    ),
})

best_pair_metrics = evaluate_candidate_recall(
    ranked_best,
    k_values=[20, 30, 40, 50, 60, 75, 100, 125, 150],
)

display(
    best_pair_metrics[
        [
            "K",
            "macro_candidate_recall",
            "any_hit_rate",
            "avg_candidates",
            "p95_candidates",
        ]
    ]
)

Best addr_pair boost: 7.0


,K,macro_candidate_recall,any_hit_rate,avg_candidates,p95_candidates
0,20,0.973195,0.997362,19.9419,20.0
1,30,0.976587,0.997784,29.7891,30.0
2,40,0.979095,0.997784,39.4920,40.0
3,50,0.980861,0.998101,48.9897,50.0
4,60,0.982690,0.998312,58.2403,60.0
5,75,0.984394,0.998523,71.5413,75.0
6,100,0.986878,0.998734,91.6742,100.0
7,125,0.988465,0.999156,108.9052,125.0
8,150,0.990110,0.999156,123.1311,150.0


In [53]:
# 27E. SMALL ADDR_NUM BOOST SWEEP
#
# addr_num generated many more new candidates than addr_pair,
# so only test modest additional weighting.

NUM_BOOSTS = [
    0.0,
    0.25,
    0.5,
    0.75,
    1.0,
    1.5,
]

num_hits = target_rank_base["addr_num_hits"].to_numpy(
    dtype=np.float32
)

num_boost_results = []

for boost in NUM_BOOSTS:

    ranked = pd.DataFrame({
        "s1_rid": target_rank_base["s1_rid"].to_numpy(),
        "cand_rid": target_rank_base["cand_rid"].to_numpy(),
        "block_score": (
            base_score
            + best_pair_boost * pair_hits
            + boost * num_hits
        ),
    })

    metrics = evaluate_candidate_recall(
        ranked,
        k_values=[50],
    ).iloc[0]

    num_boost_results.append({
        "pair_boost": best_pair_boost,
        "num_boost": boost,
        "macro_recall": float(
            metrics["macro_candidate_recall"]
        ),
        "any_hit": float(
            metrics["any_hit_rate"]
        ),
        "avg_candidates": float(
            metrics["avg_candidates"]
        ),
    })

num_boost_results = pd.DataFrame(
    num_boost_results
)

display(
    num_boost_results.sort_values(
        "macro_recall",
        ascending=False
    )
)

,pair_boost,num_boost,macro_recall,any_hit,avg_candidates
3,7.0,0.75,0.981164,0.998206,48.9897
4,7.0,1.00,0.981150,0.998206,48.9897
5,7.0,1.50,0.981132,0.998206,48.9897
2,7.0,0.50,0.981116,0.998206,48.9897
1,7.0,0.25,0.980971,0.998206,48.9897
0,7.0,0.00,0.980861,0.998101,48.9897


In [54]:
# 28A. S1 DIFFICULTY SIGNALS

# Start from the full targeted union.

difficulty = (
    target_union
    .groupby("s1_rid")
    .agg(
        top_score=("block_score", "max"),
        candidate_count=("cand_rid", "count"),
    )
    .reset_index()
)

# Number of strong address-pair candidates.
pair_counts = (
    target_rank_base
    .groupby("s1_rid")["addr_pair_hits"]
    .apply(lambda x: int((x > 0).sum()))
    .rename("addr_pair_candidate_count")
    .reset_index()
)

difficulty = difficulty.merge(
    pair_counts,
    on="s1_rid",
    how="left",
)

difficulty["addr_pair_candidate_count"] = (
    difficulty["addr_pair_candidate_count"].fillna(0)
)

display(difficulty.describe())

,s1_rid,top_score,candidate_count,addr_pair_candidate_count
count,10000.00000,10000.000000,10000.000000,10000.000000
mean,4999.50000,62.157375,164.150500,4.982600
std,2886.89568,26.480580,84.395442,6.587582
min,0.00000,3.500000,4.000000,0.000000
25%,2499.75000,45.750000,100.000000,2.000000
50%,4999.50000,62.250000,152.000000,3.000000
75%,7499.25000,78.500000,219.000000,5.000000
max,9999.00000,189.250000,548.000000,70.000000


In [55]:
# 28B. ADAPTIVE K RULES
#
# Default = 50
# Hard = 100
# Very hard = 150
#
# We tune only simple thresholds.

def adaptive_budget(
    row,
    score_threshold,
    pair_threshold,
):
    if row["top_score"] < score_threshold:
        return 150

    if row["addr_pair_candidate_count"] >= pair_threshold:
        return 100

    return 50


def evaluate_adaptive_candidates(
    candidate_df,
    difficulty_df,
    score_threshold,
    pair_threshold,
):
    diff_map = difficulty_df.set_index("s1_rid")

    rows = []

    for s1_rid, g in candidate_df.groupby("s1_rid", sort=False):

        d = diff_map.loc[s1_rid]

        budget = adaptive_budget(
            d,
            score_threshold,
            pair_threshold,
        )

        g = g.sort_values(
            "block_score",
            ascending=False
        ).head(budget)

        rows.append(g)

    out = pd.concat(
        rows,
        ignore_index=True
    )

    metrics = evaluate_candidate_recall(
        out,
        k_values=[50, 100, 150],
    )

    return out, metrics

In [56]:
# 28C. ADAPTIVE BUDGET SWEEP

SCORE_THRESHOLDS = [
    0.5,
    0.75,
    1.0,
    1.25,
    1.5,
    2.0,
]

PAIR_THRESHOLDS = [
    1,
    2,
    3,
    5,
]

adaptive_results = []

for score_t in SCORE_THRESHOLDS:
    for pair_t in PAIR_THRESHOLDS:

        adaptive_df, _ = evaluate_adaptive_candidates(
            target_union,
            difficulty,
            score_t,
            pair_t,
        )

        # Calculate actual per-S1 budget statistics.
        counts = (
            adaptive_df
            .groupby("s1_rid")
            .size()
        )

        # Evaluate the actual adaptive set directly.
        metrics = evaluate_candidate_recall(
            adaptive_df,
            k_values=[150],
        ).iloc[0]

        adaptive_results.append({
            "score_threshold": score_t,
            "pair_threshold": pair_t,
            "macro_recall": float(
                metrics["macro_candidate_recall"]
            ),
            "any_hit": float(
                metrics["any_hit_rate"]
            ),
            "avg_candidates": float(
                counts.mean()
            ),
            "p95_candidates": float(
                counts.quantile(0.95)
            ),
            "max_candidates": int(
                counts.max()
            ),
        })

adaptive_results = pd.DataFrame(
    adaptive_results
)

display(
    adaptive_results.sort_values(
        ["macro_recall", "avg_candidates"],
        ascending=[False, True]
    )
)

,score_threshold,pair_threshold,macro_recall,any_hit,avg_candidates,p95_candidates,max_candidates
0,0.50,1,0.985439,0.998206,88.2746,100.0,100
4,0.75,1,0.985439,0.998206,88.2746,100.0,100
8,1.00,1,0.985439,0.998206,88.2746,100.0,100
12,1.25,1,0.985439,0.998206,88.2746,100.0,100
16,1.50,1,0.985439,0.998206,88.2746,100.0,100
20,2.00,1,0.985439,0.998206,88.2746,100.0,100
1,0.50,2,0.984187,0.998101,84.3290,100.0,100
5,0.75,2,0.984187,0.998101,84.3290,100.0,100
9,1.00,2,0.984187,0.998101,84.3290,100.0,100
13,1.25,2,0.984187,0.998101,84.3290,100.0,100


In [57]:
# 28D. FIXED VS ADAPTIVE

fixed_results = []

for k in [50, 75, 100, 150]:

    m = evaluate_candidate_recall(
        target_union,
        k_values=[k],
    ).iloc[0]

    fixed_results.append({
        "strategy": f"fixed_{k}",
        "macro_recall": float(
            m["macro_candidate_recall"]
        ),
        "any_hit": float(
            m["any_hit_rate"]
        ),
        "avg_candidates": float(
            m["avg_candidates"]
        ),
        "p95_candidates": float(
            m["p95_candidates"]
        ),
    })

best_adaptive = (
    adaptive_results
    .sort_values(
        ["macro_recall", "avg_candidates"],
        ascending=[False, True]
    )
    .iloc[0]
)

fixed_results.append({
    "strategy": "best_adaptive",
    "macro_recall": float(
        best_adaptive["macro_recall"]
    ),
    "any_hit": float(
        best_adaptive["any_hit"]
    ),
    "avg_candidates": float(
        best_adaptive["avg_candidates"]
    ),
    "p95_candidates": float(
        best_adaptive["p95_candidates"]
    ),
})

display(pd.DataFrame(fixed_results))

,strategy,macro_recall,any_hit,avg_candidates,p95_candidates
0,fixed_50,0.978938,0.997679,48.9897,50.0
1,fixed_75,0.983682,0.998628,71.5413,75.0
2,fixed_100,0.986409,0.998734,91.6742,100.0
3,fixed_150,0.990023,0.999156,123.1311,150.0
4,best_adaptive,0.985439,0.998206,88.2746,100.0


In [58]:
# 28E. BEST RULE

print(
    "score_threshold =",
    best_adaptive["score_threshold"]
)

print(
    "pair_threshold =",
    best_adaptive["pair_threshold"]
)

print(
    "macro_recall =",
    best_adaptive["macro_recall"]
)

print(
    "any_hit =",
    best_adaptive["any_hit"]
)

print(
    "avg_candidates =",
    best_adaptive["avg_candidates"]
)

print(
    "p95_candidates =",
    best_adaptive["p95_candidates"]
)

score_threshold = 0.5
pair_threshold = 1.0
macro_recall = 0.9854390870085108
any_hit = 0.9982063726524584
avg_candidates = 88.2746
p95_candidates = 100.0


In [59]:
# 29A. CANDIDATE-COUNT BASED ADAPTIVE BUDGET

def adaptive_budget_v2(row, count_t1, count_t2, pair_t1):
    """
    Small candidate universe:
        -> K=50

    Medium candidate universe:
        -> K=75

    Large candidate universe OR strong address-pair evidence:
        -> K=100
    """

    if (
        row["candidate_count"] > count_t2
        or row["addr_pair_candidate_count"] >= pair_t1
    ):
        return 100

    if row["candidate_count"] > count_t1:
        return 75

    return 50


def evaluate_adaptive_v2(
    candidate_df,
    difficulty_df,
    count_t1,
    count_t2,
    pair_t1,
):
    diff_map = difficulty_df.set_index("s1_rid")

    rows = []

    for s1_rid, g in candidate_df.groupby("s1_rid", sort=False):

        d = diff_map.loc[s1_rid]

        budget = adaptive_budget_v2(
            d,
            count_t1=count_t1,
            count_t2=count_t2,
            pair_t1=pair_t1,
        )

        g = (
            g.sort_values("block_score", ascending=False)
             .head(budget)
        )

        rows.append(g)

    out = pd.concat(rows, ignore_index=True)

    return out

In [60]:
# 29B. ADAPTIVE BUDGET SWEEP

COUNT_T1_VALUES = [50, 60, 70, 80]
COUNT_T2_VALUES = [90, 110, 130]
PAIR_T1_VALUES = [3, 5, 8, 10]

adaptive_v2_results = []

for t1 in COUNT_T1_VALUES:
    for t2 in COUNT_T2_VALUES:

        if t2 <= t1:
            continue

        for pair_t in PAIR_T1_VALUES:

            adaptive_df = evaluate_adaptive_v2(
                target_union,
                difficulty,
                count_t1=t1,
                count_t2=t2,
                pair_t1=pair_t,
            )

            counts = (
                adaptive_df
                .groupby("s1_rid")
                .size()
            )

            # Actual adaptive set: evaluate with a K larger
            # than any possible selected budget.
            m = evaluate_candidate_recall(
                adaptive_df,
                k_values=[150]
            ).iloc[0]

            adaptive_v2_results.append({
                "count_t1": t1,
                "count_t2": t2,
                "pair_t1": pair_t,
                "macro_recall": float(
                    m["macro_candidate_recall"]
                ),
                "any_hit": float(
                    m["any_hit_rate"]
                ),
                "avg_candidates": float(
                    counts.mean()
                ),
                "p95_candidates": float(
                    counts.quantile(0.95)
                ),
                "max_candidates": int(
                    counts.max()
                ),
            })

adaptive_v2_results = pd.DataFrame(
    adaptive_v2_results
)

display(
    adaptive_v2_results.sort_values(
        ["macro_recall", "avg_candidates"],
        ascending=[False, True]
    ).head(20)
)

,count_t1,count_t2,pair_t1,macro_recall,any_hit,avg_candidates,p95_candidates,max_candidates
36,80,90,3,0.986409,0.998734,90.9935,100.0,100
15,60,90,10,0.986409,0.998734,91.0216,100.0,100
14,60,90,8,0.986409,0.998734,91.0564,100.0,100
3,50,90,10,0.986409,0.998734,91.1666,100.0,100
13,60,90,5,0.986409,0.998734,91.1913,100.0,100
2,50,90,8,0.986409,0.998734,91.1947,100.0,100
24,70,90,3,0.986409,0.998734,91.2889,100.0,100
1,50,90,5,0.986409,0.998734,91.3023,100.0,100
12,60,90,3,0.986409,0.998734,91.4652,100.0,100
0,50,90,3,0.986409,0.998734,91.5167,100.0,100


In [61]:
# 29C. FIXED VS ADAPTIVE V2

fixed = []

for k in [50, 75, 100, 125, 150]:

    m = evaluate_candidate_recall(
        target_union,
        k_values=[k]
    ).iloc[0]

    fixed.append({
        "strategy": f"fixed_{k}",
        "macro_recall": float(
            m["macro_candidate_recall"]
        ),
        "any_hit": float(
            m["any_hit_rate"]
        ),
        "avg_candidates": float(
            m["avg_candidates"]
        ),
        "p95_candidates": float(
            m["p95_candidates"]
        ),
    })

best_v2 = (
    adaptive_v2_results
    .sort_values(
        ["macro_recall", "avg_candidates"],
        ascending=[False, True]
    )
    .iloc[0]
)

fixed.append({
    "strategy": "best_adaptive_v2",
    "macro_recall": float(
        best_v2["macro_recall"]
    ),
    "any_hit": float(
        best_v2["any_hit"]
    ),
    "avg_candidates": float(
        best_v2["avg_candidates"]
    ),
    "p95_candidates": float(
        best_v2["p95_candidates"]
    ),
})

display(pd.DataFrame(fixed))

,strategy,macro_recall,any_hit,avg_candidates,p95_candidates
0,fixed_50,0.978938,0.997679,48.9897,50.0
1,fixed_75,0.983682,0.998628,71.5413,75.0
2,fixed_100,0.986409,0.998734,91.6742,100.0
3,fixed_125,0.988432,0.999156,108.9052,125.0
4,fixed_150,0.990023,0.999156,123.1311,150.0
5,best_adaptive_v2,0.986409,0.998734,90.9935,100.0


In [62]:
# 29D. BEST ADAPTIVE RULE

print("count_t1:", best_v2["count_t1"])
print("count_t2:", best_v2["count_t2"])
print("pair_t1:", best_v2["pair_t1"])
print("macro_recall:", best_v2["macro_recall"])
print("any_hit:", best_v2["any_hit"])
print("avg_candidates:", best_v2["avg_candidates"])
print("p95_candidates:", best_v2["p95_candidates"])

count_t1: 80.0
count_t2: 90.0
pair_t1: 3.0
macro_recall: 0.9864090023010681
any_hit: 0.9987339101076177
avg_candidates: 90.9935
p95_candidates: 100.0


In [63]:
# 29E. BUDGET DISTRIBUTION FOR THE BEST RULE

best_adaptive_df = evaluate_adaptive_v2(
    target_union,
    difficulty,
    count_t1=int(best_v2["count_t1"]),
    count_t2=int(best_v2["count_t2"]),
    pair_t1=int(best_v2["pair_t1"]),
)

diff_map = difficulty.set_index("s1_rid")

budget_rows = []

for s1_rid in diff_map.index:
    d = diff_map.loc[s1_rid]

    budget = adaptive_budget_v2(
        d,
        count_t1=int(best_v2["count_t1"]),
        count_t2=int(best_v2["count_t2"]),
        pair_t1=int(best_v2["pair_t1"]),
    )

    budget_rows.append({
        "s1_rid": s1_rid,
        "budget": budget,
    })

budget_dist = (
    pd.DataFrame(budget_rows)
    ["budget"]
    .value_counts()
    .sort_index()
)

print(budget_dist)
print("\nBudget percentages:")
print(
    (budget_dist / budget_dist.sum() * 100).round(2)
)

budget
50      557
75      131
100    9312
Name: count, dtype: int64

Budget percentages:
budget
50      5.57
75      1.31
100    93.12
Name: count, dtype: float64


In [64]:
# ============================================================
# 23A. FEATURE LAB PAIR POOL
# ============================================================

import numpy as np
import pandas as pd
import time

LAB_TOP_K = 75

# Only keep the columns needed for this stage.
tu = target_union[
    ["s1_rid", "cand_rid", "block_score"]
].drop_duplicates(
    ["s1_rid", "cand_rid"]
).copy()

# ------------------------------------------------------------
# 1. Top-K hard candidates by blocker score
# ------------------------------------------------------------

tu_ranked = tu.sort_values(
    ["s1_rid", "block_score"],
    ascending=[True, False],
    kind="mergesort",
)

top_candidates = (
    tu_ranked
    .groupby("s1_rid", sort=False)
    .head(LAB_TOP_K)
    .copy()
)

# ------------------------------------------------------------
# 2. Recover all GT positives that are actually present
#    inside target_union
# ------------------------------------------------------------

# Candidate entity_id -> cand_rid
cand_id_to_rid = dict(
    zip(
        CAND["entity_id"].astype(str),
        CAND["cand_rid"].astype(np.int32),
    )
)

# S1 rid -> entity_id
s1_rid_to_entity = dict(
    zip(
        S1["s1_rid"].astype(np.int32),
        S1["entity_id"].astype(str),
    )
)

# Build all GT pairs that can be mapped into CAND.
positive_rows = []

for s1_rid, s1_entity_id in s1_rid_to_entity.items():
    gt_ids = GT_MAP.get(s1_entity_id, set())

    for cand_entity_id in gt_ids:
        cand_rid = cand_id_to_rid.get(str(cand_entity_id))

        if cand_rid is not None:
            positive_rows.append(
                {
                    "s1_rid": int(s1_rid),
                    "cand_rid": int(cand_rid),
                }
            )

positive_pairs10k = pd.DataFrame(
    positive_rows,
    columns=["s1_rid", "cand_rid"],
)

# Keep only positives that survived our blocking union.
positive_in_union10k = positive_pairs10k.merge(
    tu,
    on=["s1_rid", "cand_rid"],
    how="inner",
)

# ------------------------------------------------------------
# 3. Combine Top-K + all available positives
# ------------------------------------------------------------

feature_lab_pairs10k = pd.concat(
    [
        top_candidates,
        positive_in_union10k[
            ["s1_rid", "cand_rid", "block_score"]
        ],
    ],
    ignore_index=True,
).drop_duplicates(
    ["s1_rid", "cand_rid"]
)

# ------------------------------------------------------------
# 4. Add labels
# ------------------------------------------------------------

positive_key_set = set(
    zip(
        positive_in_union10k["s1_rid"].astype(int),
        positive_in_union10k["cand_rid"].astype(int),
    )
)

feature_lab_pairs10k["label"] = [
    int((int(s1_rid), int(cand_rid)) in positive_key_set)
    for s1_rid, cand_rid
    in zip(
        feature_lab_pairs10k["s1_rid"],
        feature_lab_pairs10k["cand_rid"],
    )
]

feature_lab_pairs10k = feature_lab_pairs10k.sort_values(
    ["s1_rid", "block_score"],
    ascending=[True, False],
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Diagnostics
# ------------------------------------------------------------

n_gt_total = sum(len(v) for v in GT_MAP.values())
n_gt_mapped_to_cand = len(positive_pairs10k)
n_gt_in_union = len(positive_in_union10k)

print("=== FEATURE LAB PAIR POOL ===")
print("S1 entities:", len(S1))
print("Ground-truth positive pairs:", n_gt_total)
print("GT positives found in CAND:", n_gt_mapped_to_cand)
print("GT positives inside target_union:", n_gt_in_union)

print()
print("Feature-lab candidate rows:", len(feature_lab_pairs10k))
print("Positive rows:", int(feature_lab_pairs10k["label"].sum()))
print("Negative rows:", int((feature_lab_pairs10k["label"] == 0).sum()))

print()
print(
    "Average candidates/S1:",
    feature_lab_pairs10k.groupby("s1_rid").size().mean()
)

print(
    "Positive rate:",
    feature_lab_pairs10k["label"].mean()
)

print()
print("Positive count distribution/S1:")
print(
    feature_lab_pairs10k.loc[
        feature_lab_pairs10k["label"] == 1
    ]
    .groupby("s1_rid")
    .size()
    .describe()
)

=== FEATURE LAB PAIR POOL ===
S1 entities: 10000
Ground-truth positive pairs: 34751
GT positives found in CAND: 34751
GT positives inside target_union: 34507

Feature-lab candidate rows: 715739
Positive rows: 34507
Negative rows: 681232

Average candidates/S1: 71.5739
Positive rate: 0.04821170845797141

Positive count distribution/S1:
count    9472.000000
mean        3.643053
std         1.531086
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max        10.000000
dtype: float64


In [65]:
# ============================================================
# 23B. CHEAP FEATURE MATRIX
# ============================================================

from rapidfuzz import fuzz
import numpy as np
import pandas as pd
import time

# ------------------------------------------------------------
# Fast helpers
# ------------------------------------------------------------

def safe_jaccard(a, b):
    a = a if isinstance(a, (set, frozenset)) else set(a or [])
    b = b if isinstance(b, (set, frozenset)) else set(b or [])

    if not a and not b:
        return 0.0

    union = a | b
    if not union:
        return 0.0

    return len(a & b) / len(union)


def overlap_count(a, b):
    a = a if isinstance(a, (set, frozenset)) else set(a or [])
    b = b if isinstance(b, (set, frozenset)) else set(b or [])
    return len(a & b)


def safe_ratio(a, b):
    if not a or not b:
        return 0.0
    return fuzz.ratio(a, b) / 100.0


def safe_token_sort_ratio(a, b):
    if not a or not b:
        return 0.0
    return fuzz.token_sort_ratio(a, b) / 100.0


def safe_token_set_ratio(a, b):
    if not a or not b:
        return 0.0
    return fuzz.token_set_ratio(a, b) / 100.0


# ------------------------------------------------------------
# Lookups
# ------------------------------------------------------------

s1_lookup10k = S1.set_index("s1_rid")
cand_lookup10k = CAND.set_index("cand_rid")

# Cache frequently accessed columns as dictionaries.
s1_name_native = S1.set_index("s1_rid")["name_native"].to_dict()
s1_name_ascii = S1.set_index("s1_rid")["name_ascii"].to_dict()
s1_addr_native = S1.set_index("s1_rid")["addr_native"].to_dict()
s1_addr_ascii = S1.set_index("s1_rid")["addr_ascii"].to_dict()
s1_country = S1.set_index("s1_rid")["country"].to_dict()

s1_name_tok_native = S1.set_index("s1_rid")["name_tokens_native"].to_dict()
s1_name_tok_ascii = S1.set_index("s1_rid")["name_tokens_ascii"].to_dict()
s1_addr_tok_native = S1.set_index("s1_rid")["addr_tokens_native"].to_dict()
s1_addr_tok_ascii = S1.set_index("s1_rid")["addr_tokens_ascii"].to_dict()
s1_numbers = S1.set_index("s1_rid")["numbers_native"].to_dict()
s1_postal = S1.set_index("s1_rid")["postal_native"].to_dict()

cand_name_native = CAND.set_index("cand_rid")["name_native"].to_dict()
cand_name_ascii = CAND.set_index("cand_rid")["name_ascii"].to_dict()
cand_addr_native = CAND.set_index("cand_rid")["addr_native"].to_dict()
cand_addr_ascii = CAND.set_index("cand_rid")["addr_ascii"].to_dict()
cand_country = CAND.set_index("cand_rid")["country"].to_dict()

cand_name_tok_native = CAND.set_index("cand_rid")["name_tokens_native"].to_dict()
cand_name_tok_ascii = CAND.set_index("cand_rid")["name_tokens_ascii"].to_dict()
cand_addr_tok_native = CAND.set_index("cand_rid")["addr_tokens_native"].to_dict()
cand_addr_tok_ascii = CAND.set_index("cand_rid")["addr_tokens_ascii"].to_dict()
cand_numbers = CAND.set_index("cand_rid")["numbers_native"].to_dict()
cand_postal = CAND.set_index("cand_rid")["postal_native"].to_dict()


# ------------------------------------------------------------
# Compute features
# ------------------------------------------------------------

t0 = time.time()

feature_rows = []

for i, r in enumerate(
    feature_lab_pairs10k.itertuples(index=False),
    start=1
):
    s1_rid = int(r.s1_rid)
    cand_rid = int(r.cand_rid)

    # -------- Name --------

    nn = s1_name_native[s1_rid]
    cn = cand_name_native[cand_rid]

    na = s1_name_ascii[s1_rid]
    ca = cand_name_ascii[cand_rid]

    name_ratio_native = safe_ratio(nn, cn)
    name_ratio_ascii = safe_ratio(na, ca)

    name_token_sort = safe_token_sort_ratio(na, ca)
    name_token_set = safe_token_set_ratio(na, ca)

    name_jaccard_native = safe_jaccard(
        s1_name_tok_native[s1_rid],
        cand_name_tok_native[cand_rid],
    )

    name_jaccard_ascii = safe_jaccard(
        s1_name_tok_ascii[s1_rid],
        cand_name_tok_ascii[cand_rid],
    )

    name_shared_tokens = overlap_count(
        s1_name_tok_ascii[s1_rid],
        cand_name_tok_ascii[cand_rid],
    )

    # -------- Address --------

    an = s1_addr_native[s1_rid]
    ac = cand_addr_native[cand_rid]

    aa = s1_addr_ascii[s1_rid]
    ad = cand_addr_ascii[cand_rid]

    address_ratio_native = safe_ratio(an, ac)
    address_ratio_ascii = safe_ratio(aa, ad)

    address_token_sort = safe_token_sort_ratio(aa, ad)
    address_token_set = safe_token_set_ratio(aa, ad)

    address_jaccard_native = safe_jaccard(
        s1_addr_tok_native[s1_rid],
        cand_addr_tok_native[cand_rid],
    )

    address_jaccard_ascii = safe_jaccard(
        s1_addr_tok_ascii[s1_rid],
        cand_addr_tok_ascii[cand_rid],
    )

    address_shared_tokens = overlap_count(
        s1_addr_tok_ascii[s1_rid],
        cand_addr_tok_ascii[cand_rid],
    )

    # -------- Numbers / postal --------

    number_overlap = safe_jaccard(
        s1_numbers[s1_rid],
        cand_numbers[cand_rid],
    )

    number_shared_count = overlap_count(
        s1_numbers[s1_rid],
        cand_numbers[cand_rid],
    )

    postal_match = int(
        bool(
            set(s1_postal[s1_rid] or [])
            & set(cand_postal[cand_rid] or [])
        )
    )

    # -------- Exact / missing --------

    name_exact_native = int(
        bool(nn and cn and nn == cn)
    )

    name_exact_ascii = int(
        bool(na and ca and na == ca)
    )

    address_exact_native = int(
        bool(an and ac and an == ac)
    )

    address_exact_ascii = int(
        bool(aa and ad and aa == ad)
    )

    country_match = int(
        bool(
            s1_country[s1_rid]
            and s1_country[s1_rid] == cand_country[cand_rid]
        )
    )

    name_missing = int(not na)
    candidate_name_missing = int(not ca)

    address_missing = int(not aa)
    candidate_address_missing = int(not ad)

    # -------- Length / structure --------

    name_len_s1 = len(na)
    name_len_cand = len(ca)

    address_len_s1 = len(aa)
    address_len_cand = len(ad)

    name_len_ratio = (
        min(name_len_s1, name_len_cand)
        / max(name_len_s1, name_len_cand)
        if max(name_len_s1, name_len_cand) > 0
        else 0.0
    )

    address_len_ratio = (
        min(address_len_s1, address_len_cand)
        / max(address_len_s1, address_len_cand)
        if max(address_len_s1, address_len_cand) > 0
        else 0.0
    )

    # -------- Feature interactions --------

    name_addr_ratio_mean = (
        name_ratio_ascii + address_ratio_ascii
    ) / 2.0

    max_field_ratio = max(
        name_ratio_ascii,
        address_ratio_ascii,
    )

    # -------- Save --------

    feature_rows.append(
        {
            "s1_rid": s1_rid,
            "cand_rid": cand_rid,

            "label": int(r.label),

            "block_score": float(r.block_score),

            # Name
            "name_ratio_native": name_ratio_native,
            "name_ratio_ascii": name_ratio_ascii,
            "name_token_sort": name_token_sort,
            "name_token_set": name_token_set,
            "name_jaccard_native": name_jaccard_native,
            "name_jaccard_ascii": name_jaccard_ascii,
            "name_shared_tokens": name_shared_tokens,
            "name_exact_native": name_exact_native,
            "name_exact_ascii": name_exact_ascii,

            # Address
            "address_ratio_native": address_ratio_native,
            "address_ratio_ascii": address_ratio_ascii,
            "address_token_sort": address_token_sort,
            "address_token_set": address_token_set,
            "address_jaccard_native": address_jaccard_native,
            "address_jaccard_ascii": address_jaccard_ascii,
            "address_shared_tokens": address_shared_tokens,
            "address_exact_native": address_exact_native,
            "address_exact_ascii": address_exact_ascii,

            # Numeric / postal
            "number_overlap": number_overlap,
            "number_shared_count": number_shared_count,
            "postal_match": postal_match,

            # Country
            "country_match": country_match,

            # Missingness
            "name_missing": name_missing,
            "candidate_name_missing": candidate_name_missing,
            "address_missing": address_missing,
            "candidate_address_missing": candidate_address_missing,

            # Length
            "name_len_ratio": name_len_ratio,
            "address_len_ratio": address_len_ratio,

            # Interactions
            "name_addr_ratio_mean": name_addr_ratio_mean,
            "max_field_ratio": max_field_ratio,
        }
    )

    if i % 100_000 == 0:
        print(
            f"{i:,}/{len(feature_lab_pairs10k):,} "
            f"pairs processed | "
            f"{time.time() - t0:.1f}s"
        )

features10k = pd.DataFrame(feature_rows)

elapsed = time.time() - t0

print()
print("=== FEATURE MATRIX ===")
print("Rows:", len(features10k))
print("Columns:", len(features10k.columns))
print("Runtime:", round(elapsed, 2), "sec")
print("Positive:", int(features10k["label"].sum()))
print("Negative:", int((features10k["label"] == 0).sum()))

features10k.head()

100,000/715,739 pairs processed | 2.6s
200,000/715,739 pairs processed | 4.6s
300,000/715,739 pairs processed | 6.5s
400,000/715,739 pairs processed | 8.6s
500,000/715,739 pairs processed | 10.6s
600,000/715,739 pairs processed | 12.4s
700,000/715,739 pairs processed | 14.4s

=== FEATURE MATRIX ===
Rows: 715739
Columns: 34
Runtime: 18.82 sec
Positive: 34507
Negative: 681232


,s1_rid,cand_rid,label,block_score,name_ratio_native,name_ratio_ascii,name_token_sort,name_token_set,name_jaccard_native,name_jaccard_ascii,...,postal_match,country_match,name_missing,candidate_name_missing,address_missing,candidate_address_missing,name_len_ratio,address_len_ratio,name_addr_ratio_mean,max_field_ratio
0,0,42474,1,78.25,0.777778,0.777778,0.777778,1.000000,1.000000,1.000000,...,0,1,0,0,0,0,0.636364,0.892157,0.756765,0.777778
1,0,4147,1,67.50,0.698413,0.698413,0.698413,1.000000,0.666667,0.666667,...,0,1,0,0,0,0,0.536585,1.000000,0.697246,0.698413
2,0,11795,1,58.25,0.900000,0.900000,0.900000,0.900000,1.000000,1.000000,...,0,1,0,0,0,0,0.818182,0.980392,0.801485,0.900000
3,0,7631,0,10.25,0.652174,0.652174,0.652174,0.652174,0.333333,0.333333,...,0,1,0,0,0,0,0.916667,0.892157,0.502253,0.652174
4,0,54663,0,8.00,0.600000,0.600000,0.600000,0.600000,0.500000,0.500000,...,0,1,0,0,0,0,0.818182,0.823529,0.472043,0.600000


In [66]:
# ============================================================
# 23C. UNIVARIATE FEATURE QUALITY
# ============================================================

from sklearn.metrics import roc_auc_score

exclude_cols = {
    "s1_rid",
    "cand_rid",
    "label",
}

feature_quality = []

y = features10k["label"].to_numpy()

for col in features10k.columns:
    if col in exclude_cols:
        continue

    x = features10k[col].to_numpy(dtype=float)

    # Constant feature -> no discrimination.
    if np.nanstd(x) == 0:
        auc = 0.5
    else:
        auc = roc_auc_score(y, x)

    # AUC below 0.5 simply means the inverse ordering is informative.
    effective_auc = max(auc, 1.0 - auc)

    feature_quality.append(
        {
            "feature": col,
            "auc": auc,
            "effective_auc": effective_auc,
            "positive_mean": features10k.loc[
                features10k["label"] == 1, col
            ].mean(),
            "negative_mean": features10k.loc[
                features10k["label"] == 0, col
            ].mean(),
        }
    )

feature_quality10k = (
    pd.DataFrame(feature_quality)
    .sort_values("effective_auc", ascending=False)
    .reset_index(drop=True)
)

print("=== INDIVIDUAL FEATURE QUALITY ===")
display(feature_quality10k)

=== INDIVIDUAL FEATURE QUALITY ===


,feature,auc,effective_auc,positive_mean,negative_mean
0,max_field_ratio,0.984999,0.984999,0.909244,0.497512
1,block_score,0.980960,0.980960,47.737662,5.853956
2,name_addr_ratio_mean,0.977453,0.977453,0.801970,0.407869
3,address_jaccard_native,0.965799,0.965799,0.654374,0.036842
4,address_jaccard_ascii,0.965719,0.965719,0.647477,0.036507
5,address_shared_tokens,0.954670,0.954670,4.033240,0.329350
6,address_token_set,0.953521,0.953521,0.872818,0.371104
7,address_token_sort,0.946368,0.946368,0.818100,0.359919
8,name_ratio_ascii,0.943470,0.943470,0.842043,0.470905
9,name_token_sort,0.940819,0.940819,0.849588,0.458667


In [67]:
# ============================================================
# 23D. SPARSE TF-IDF COSINE FEATURES
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
from scipy import sparse
import numpy as np
import pandas as pd
import time


# ------------------------------------------------------------
# Row-wise sparse cosine for candidate pairs
# ------------------------------------------------------------

def pairwise_sparse_cosine(
    left_matrix,
    right_matrix,
    left_indices,
    right_indices,
    batch_size=50_000,
):
    """
    Compute cosine similarity for paired sparse rows:

        left_matrix[left_indices[i]]
        vs
        right_matrix[right_indices[i]]

    without constructing a giant pairwise matrix.
    """

    left_indices = np.asarray(left_indices, dtype=np.int32)
    right_indices = np.asarray(right_indices, dtype=np.int32)

    result = np.empty(
        len(left_indices),
        dtype=np.float32,
    )

    for start in range(0, len(left_indices), batch_size):
        end = min(
            start + batch_size,
            len(left_indices),
        )

        A = left_matrix[left_indices[start:end]]
        B = right_matrix[right_indices[start:end]]

        # TF-IDF rows are L2 normalized.
        # Therefore cosine = sparse element-wise dot product.
        result[start:end] = np.asarray(
            A.multiply(B).sum(axis=1)
        ).ravel()

    return result


# ------------------------------------------------------------
# Prepare unique strings
# ------------------------------------------------------------

pair_s1_ids = features10k["s1_rid"].to_numpy(dtype=np.int32)
pair_cand_ids = features10k["cand_rid"].to_numpy(dtype=np.int32)


s1_index = S1["s1_rid"].to_numpy(dtype=np.int32)
cand_index = CAND["cand_rid"].to_numpy(dtype=np.int32)

s1_pos = {
    int(rid): i
    for i, rid in enumerate(s1_index)
}

cand_pos = {
    int(rid): i
    for i, rid in enumerate(cand_index)
}


left_idx = np.array(
    [s1_pos[int(x)] for x in pair_s1_ids],
    dtype=np.int32,
)

right_idx = np.array(
    [cand_pos[int(x)] for x in pair_cand_ids],
    dtype=np.int32,
)


# ------------------------------------------------------------
# Generic experiment function
# ------------------------------------------------------------

def run_tfidf_experiment(
    s1_text,
    cand_text,
    analyzer,
    ngram_range,
    min_df=2,
    max_features=None,
    batch_size=50_000,
):
    t0 = time.time()

    vectorizer = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        min_df=min_df,
        max_features=max_features,
        lowercase=False,
        norm="l2",
        dtype=np.float32,
    )

    # Fit on the combined vocabulary.
    # This is a feature-audit experiment.
    vectorizer.fit(
        pd.concat(
            [s1_text, cand_text],
            ignore_index=True,
        )
    )

    fit_time = time.time() - t0

    A = vectorizer.transform(s1_text)
    B = vectorizer.transform(cand_text)

    transform_time = time.time() - t0 - fit_time

    cosine = pairwise_sparse_cosine(
        A,
        B,
        left_idx,
        right_idx,
        batch_size=batch_size,
    )

    total_time = time.time() - t0

    return {
        "vectorizer": vectorizer,
        "cosine": cosine,
        "vocab_size": len(vectorizer.vocabulary_),
        "nnz_left": A.nnz,
        "nnz_right": B.nnz,
        "fit_time": fit_time,
        "transform_time": transform_time,
        "total_time": total_time,
    }


# ============================================================
# NAME — CHARACTER TF-IDF
# ============================================================

print("=== NAME CHAR TF-IDF ===")

name_char_result = run_tfidf_experiment(
    S1["name_ascii"].fillna(""),
    CAND["name_ascii"].fillna(""),
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
)

features10k["name_tfidf_char_cosine"] = (
    name_char_result["cosine"]
)

print(
    "Vocabulary:",
    name_char_result["vocab_size"]
)
print(
    "Left NNZ:",
    name_char_result["nnz_left"]
)
print(
    "Right NNZ:",
    name_char_result["nnz_right"]
)
print(
    "Runtime:",
    round(name_char_result["total_time"], 2),
    "sec"
)


# ============================================================
# NAME — WORD TF-IDF
# ============================================================

print("\n=== NAME WORD TF-IDF ===")

name_word_result = run_tfidf_experiment(
    S1["name_ascii"].fillna(""),
    CAND["name_ascii"].fillna(""),
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
)

features10k["name_tfidf_word_cosine"] = (
    name_word_result["cosine"]
)

print(
    "Vocabulary:",
    name_word_result["vocab_size"]
)
print(
    "Runtime:",
    round(name_word_result["total_time"], 2),
    "sec"
)


# ============================================================
# ADDRESS — CHARACTER TF-IDF
# ============================================================

print("\n=== ADDRESS CHAR TF-IDF ===")

addr_char_result = run_tfidf_experiment(
    S1["addr_ascii"].fillna(""),
    CAND["addr_ascii"].fillna(""),
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
)

features10k["address_tfidf_char_cosine"] = (
    addr_char_result["cosine"]
)

print(
    "Vocabulary:",
    addr_char_result["vocab_size"]
)
print(
    "Runtime:",
    round(addr_char_result["total_time"], 2),
    "sec"
)


# ============================================================
# ADDRESS — WORD TF-IDF
# ============================================================

print("\n=== ADDRESS WORD TF-IDF ===")

addr_word_result = run_tfidf_experiment(
    S1["addr_ascii"].fillna(""),
    CAND["addr_ascii"].fillna(""),
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
)

features10k["address_tfidf_word_cosine"] = (
    addr_word_result["cosine"]
)

print(
    "Vocabulary:",
    addr_word_result["vocab_size"]
)
print(
    "Runtime:",
    round(addr_word_result["total_time"], 2),
    "sec"
)


print("\n=== TF-IDF FEATURES ADDED ===")

print(
    features10k[
        [
            "name_tfidf_char_cosine",
            "name_tfidf_word_cosine",
            "address_tfidf_char_cosine",
            "address_tfidf_word_cosine",
        ]
    ].describe()
)

=== NAME CHAR TF-IDF ===
Vocabulary: 113238
Left NNZ: 828324
Right NNZ: 4447984
Runtime: 3.32 sec

=== NAME WORD TF-IDF ===
Vocabulary: 96702
Runtime: 0.49 sec

=== ADDRESS CHAR TF-IDF ===
Vocabulary: 219741
Runtime: 6.65 sec

=== ADDRESS WORD TF-IDF ===
Vocabulary: 188431
Runtime: 0.98 sec

=== TF-IDF FEATURES ADDED ===
       name_tfidf_char_cosine  name_tfidf_word_cosine  \
count           715739.000000           715739.000000   
mean                 0.228197                0.132821   
std                  0.221133                0.187358   
min                  0.000000                0.000000   
25%                  0.032423                0.000000   
50%                  0.190219                0.107699   
75%                  0.348190                0.179665   
max                  1.000000                1.000000   

       address_tfidf_char_cosine  address_tfidf_word_cosine  
count              715739.000000              715739.000000  
mean                    0.081439       

In [68]:
# ============================================================
# 23E. TF-IDF FEATURE QUALITY
# ============================================================

tfidf_cols = [
    "name_tfidf_char_cosine",
    "name_tfidf_word_cosine",
    "address_tfidf_char_cosine",
    "address_tfidf_word_cosine",
]

tfidf_quality = []

y = features10k["label"].to_numpy()

for col in tfidf_cols:

    x = features10k[col].to_numpy(
        dtype=float
    )

    auc = roc_auc_score(y, x)

    tfidf_quality.append(
        {
            "feature": col,
            "auc": auc,
            "positive_mean": features10k.loc[
                features10k["label"] == 1,
                col,
            ].mean(),
            "negative_mean": features10k.loc[
                features10k["label"] == 0,
                col,
            ].mean(),
            "positive_median": features10k.loc[
                features10k["label"] == 1,
                col,
            ].median(),
            "negative_median": features10k.loc[
                features10k["label"] == 0,
                col,
            ].median(),
        }
    )

tfidf_quality10k = (
    pd.DataFrame(tfidf_quality)
    .sort_values(
        "auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("=== TF-IDF QUALITY ===")
display(tfidf_quality10k)

=== TF-IDF QUALITY ===


,feature,auc,positive_mean,negative_mean,positive_median,negative_median
0,address_tfidf_word_cosine,0.966138,0.616083,0.026252,0.648296,0.000000
1,address_tfidf_char_cosine,0.953843,0.736185,0.048274,0.797783,0.015401
2,name_tfidf_char_cosine,0.929594,0.756979,0.201412,0.848514,0.174499
3,name_tfidf_word_cosine,0.874611,0.645975,0.106828,0.786157,0.099783


In [69]:
# ============================================================
# 24A. S1-LEVEL TRAIN / VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd
import time

# ------------------------------------------------------------
# Feature columns
# ------------------------------------------------------------

BASE_FEATURES = [
    "block_score",

    # Name
    "name_ratio_native",
    "name_ratio_ascii",
    "name_token_sort",
    "name_token_set",
    "name_jaccard_native",
    "name_jaccard_ascii",
    "name_shared_tokens",
    "name_exact_native",
    "name_exact_ascii",

    # Address
    "address_ratio_native",
    "address_ratio_ascii",
    "address_token_sort",
    "address_token_set",
    "address_jaccard_native",
    "address_jaccard_ascii",
    "address_shared_tokens",
    "address_exact_native",
    "address_exact_ascii",

    # Numeric / postal
    "number_overlap",
    "number_shared_count",
    "postal_match",

    # Country
    "country_match",

    # Missingness
    "name_missing",
    "candidate_name_missing",
    "address_missing",
    "candidate_address_missing",

    # Length
    "name_len_ratio",
    "address_len_ratio",

    # Interactions
    "name_addr_ratio_mean",
    "max_field_ratio",
]

TFIDF_FEATURES = [
    "name_tfidf_char_cosine",
    "name_tfidf_word_cosine",
    "address_tfidf_char_cosine",
    "address_tfidf_word_cosine",
]

ALL_FEATURES = BASE_FEATURES + TFIDF_FEATURES


# ------------------------------------------------------------
# S1-level split
# ------------------------------------------------------------

unique_s1 = features10k["s1_rid"].unique()

train_s1, valid_s1 = train_test_split(
    unique_s1,
    test_size=0.20,
    random_state=42,
)

train_mask = features10k["s1_rid"].isin(train_s1)
valid_mask = features10k["s1_rid"].isin(valid_s1)

train_df = features10k.loc[train_mask].copy()
valid_df = features10k.loc[valid_mask].copy()

print("=== SPLIT ===")
print("Train S1:", train_df["s1_rid"].nunique())
print("Valid S1:", valid_df["s1_rid"].nunique())

print()
print("Train rows:", len(train_df))
print("Valid rows:", len(valid_df))

print()
print("Train positives:", int(train_df["label"].sum()))
print("Valid positives:", int(valid_df["label"].sum()))

print()
print("Train positive rate:", train_df["label"].mean())
print("Valid positive rate:", valid_df["label"].mean())

=== SPLIT ===
Train S1: 8000
Valid S1: 2000

Train rows: 572681
Valid rows: 143058

Train positives: 27556
Valid positives: 6951

Train positive rate: 0.048117538385244144
Valid positive rate: 0.04858868430986034


In [70]:
# ============================================================
# 24B. ENTITY-LEVEL F0.5 EVALUATION
# ============================================================

def entity_f05(
    y_true,
    y_prob,
    s1_ids,
    threshold,
):
    """
    Macro F0.5 over S1 entities represented in the candidate pool.
    """

    tmp = pd.DataFrame(
        {
            "s1_rid": s1_ids,
            "y_true": y_true,
            "y_prob": y_prob,
        }
    )

    scores = []

    beta2 = 0.25

    for s1_rid, g in tmp.groupby("s1_rid"):
        true_set = set(
            g.loc[g["y_true"] == 1, "s1_rid"].index
        )

        # Candidate row indices with positive prediction.
        pred_idx = set(
            g.index[g["y_prob"] >= threshold]
        )

        true_idx = set(
            g.index[g["y_true"] == 1]
        )

        tp = len(pred_idx & true_idx)
        fp = len(pred_idx - true_idx)
        fn = len(true_idx - pred_idx)

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 1.0
        )

        if precision == 0.0 and recall == 0.0:
            f05 = 0.0
        else:
            f05 = (
                (1.0 + beta2)
                * precision
                * recall
                / (
                    beta2 * precision
                    + recall
                )
            )

        scores.append(f05)

    return float(np.mean(scores))


def threshold_sweep(
    y_true,
    y_prob,
    s1_ids,
    thresholds=None,
):
    if thresholds is None:
        thresholds = np.arange(
            0.10,
            0.991,
            0.01,
        )

    rows = []

    for threshold in thresholds:
        f = entity_f05(
            y_true,
            y_prob,
            s1_ids,
            threshold,
        )

        rows.append(
            {
                "threshold": threshold,
                "macro_f05": f,
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            "macro_f05",
            ascending=False,
        )
        .reset_index(drop=True)
    )

In [71]:
# ============================================================
# 24C. MODEL A — BASE FEATURES ONLY
# ============================================================

X_train_base = train_df[BASE_FEATURES]
y_train = train_df["label"]

X_valid_base = valid_df[BASE_FEATURES]
y_valid = valid_df["label"]

print("Training Model A...")
t0 = time.time()

model_base10k = CatBoostClassifier(
    iterations=600,
    depth=8,
    learning_rate=0.08,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False,
)

model_base10k.fit(
    X_train_base,
    y_train,
    eval_set=(X_valid_base, y_valid),
    early_stopping_rounds=75,
)

base_time = time.time() - t0

valid_prob_base = model_base10k.predict_proba(
    X_valid_base
)[:, 1]

base_auc = roc_auc_score(
    y_valid,
    valid_prob_base,
)

base_thresholds = threshold_sweep(
    y_valid.to_numpy(),
    valid_prob_base,
    valid_df["s1_rid"].to_numpy(),
)

print()
print("=== MODEL A ===")
print("Training time:", round(base_time, 2), "sec")
print("Validation pair AUC:", round(base_auc, 6))
print()
display(base_thresholds.head(10))

Training Model A...
0:	test: 0.9988722	best: 0.9988722 (0)	total: 110ms	remaining: 1m 5s
100:	test: 0.9999240	best: 0.9999240 (100)	total: 4.16s	remaining: 20.6s
200:	test: 0.9999383	best: 0.9999383 (200)	total: 8.13s	remaining: 16.1s
300:	test: 0.9999425	best: 0.9999427 (286)	total: 12.1s	remaining: 12s
400:	test: 0.9999443	best: 0.9999446 (360)	total: 16s	remaining: 7.93s
Stopped by overfitting detector  (75 iterations wait)

bestTest = 0.9999446014
bestIteration = 360

Shrink model to first 361 iterations.

=== MODEL A ===
Training time: 17.41 sec
Validation pair AUC: 0.999945



,threshold,macro_f05
0,0.60,0.934170
1,0.59,0.934130
2,0.65,0.934116
3,0.58,0.934114
4,0.61,0.934046
5,0.63,0.934004
6,0.66,0.933949
7,0.62,0.933939
8,0.64,0.933925
9,0.76,0.933853


In [72]:
# ============================================================
# 24D. MODEL B — BASE + TF-IDF
# ============================================================

X_train_all = train_df[ALL_FEATURES]
X_valid_all = valid_df[ALL_FEATURES]

print("Training Model B...")
t0 = time.time()

model_all10k = CatBoostClassifier(
    iterations=600,
    depth=8,
    learning_rate=0.08,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False,
)

model_all10k.fit(
    X_train_all,
    y_train,
    eval_set=(X_valid_all, y_valid),
    early_stopping_rounds=75,
)

all_time = time.time() - t0

valid_prob_all = model_all10k.predict_proba(
    X_valid_all
)[:, 1]

all_auc = roc_auc_score(
    y_valid,
    valid_prob_all,
)

all_thresholds = threshold_sweep(
    y_valid.to_numpy(),
    valid_prob_all,
    valid_df["s1_rid"].to_numpy(),
)

print()
print("=== MODEL B ===")
print("Training time:", round(all_time, 2), "sec")
print("Validation pair AUC:", round(all_auc, 6))
print()
display(all_thresholds.head(10))

Training Model B...
0:	test: 0.9950193	best: 0.9950193 (0)	total: 52.3ms	remaining: 31.3s
100:	test: 0.9999434	best: 0.9999434 (100)	total: 4.27s	remaining: 21.1s
200:	test: 0.9999544	best: 0.9999544 (200)	total: 8.64s	remaining: 17.1s
300:	test: 0.9999562	best: 0.9999563 (247)	total: 13.2s	remaining: 13.1s
Stopped by overfitting detector  (75 iterations wait)

bestTest = 0.999956296
bestIteration = 247

Shrink model to first 248 iterations.

=== MODEL B ===
Training time: 14.13 sec
Validation pair AUC: 0.999956



,threshold,macro_f05
0,0.65,0.935450
1,0.60,0.935440
2,0.61,0.935432
3,0.59,0.935418
4,0.62,0.935390
5,0.63,0.935374
6,0.58,0.935365
7,0.64,0.935357
8,0.66,0.935350
9,0.67,0.935318


In [73]:
# ============================================================
# 24E. MODEL COMPARISON
# ============================================================

best_base = base_thresholds.iloc[0]
best_all = all_thresholds.iloc[0]

comparison10k = pd.DataFrame(
    [
        {
            "model": "Base",
            "features": len(BASE_FEATURES),
            "pair_auc": base_auc,
            "best_macro_f05": best_base["macro_f05"],
            "best_threshold": best_base["threshold"],
            "train_time_sec": base_time,
        },
        {
            "model": "Base + TFIDF",
            "features": len(ALL_FEATURES),
            "pair_auc": all_auc,
            "best_macro_f05": best_all["macro_f05"],
            "best_threshold": best_all["threshold"],
            "train_time_sec": all_time,
        },
    ]
)

display(comparison10k)

,model,features,pair_auc,best_macro_f05,best_threshold,train_time_sec
0,Base,31,0.999945,0.93417,0.60,17.406584
1,Base + TFIDF,35,0.999956,0.93545,0.65,14.128059


In [74]:
# ============================================================
# 24F. ENTITY-LEVEL ERROR DIAGNOSTICS
# ============================================================

BEST_THRESHOLD = float(best_all["threshold"])

valid_diag = valid_df[
    [
        "s1_rid",
        "cand_rid",
        "label",
    ]
].copy()

valid_diag["prob"] = valid_prob_all

# ------------------------------------------------------------
# Per-S1 metrics
# ------------------------------------------------------------

diag_rows = []

for s1_rid, g in valid_diag.groupby("s1_rid"):

    true_set = set(
        g.loc[g["label"] == 1, "cand_rid"]
    )

    pred_set = set(
        g.loc[
            g["prob"] >= BEST_THRESHOLD,
            "cand_rid",
        ]
    )

    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 1.0
    )

    f05 = (
        1.25 * precision * recall
        / (0.25 * precision + recall)
        if precision > 0 and recall > 0
        else 0.0
    )

    diag_rows.append(
        {
            "s1_rid": s1_rid,
            "true_count": len(true_set),
            "pred_count": len(pred_set),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f05": f05,
        }
    )

entity_diag10k = pd.DataFrame(diag_rows)

print("=== ENTITY-LEVEL DIAGNOSTICS ===")

print(
    "Macro F0.5:",
    entity_diag10k["f05"].mean()
)

print(
    "Mean predicted matches/S1:",
    entity_diag10k["pred_count"].mean()
)

print(
    "Median predicted matches/S1:",
    entity_diag10k["pred_count"].median()
)

print(
    "95th percentile predicted matches/S1:",
    entity_diag10k["pred_count"].quantile(0.95)
)

print(
    "Mean true matches/S1:",
    entity_diag10k["true_count"].mean()
)

print(
    "Total TP:",
    entity_diag10k["tp"].sum()
)

print(
    "Total FP:",
    entity_diag10k["fp"].sum()
)

print(
    "Total FN:",
    entity_diag10k["fn"].sum()
)

print()
print("Singleton S1 count:")

print(
    (
        entity_diag10k["true_count"] == 0
    ).sum()
)

print(
    "Singletons with >=1 prediction:"
)

print(
    (
        (entity_diag10k["true_count"] == 0)
        &
        (entity_diag10k["pred_count"] > 0)
    ).sum()
)

print()
print("Worst 20 S1s by F0.5:")

display(
    entity_diag10k
    .sort_values(
        ["f05", "fp"],
        ascending=[True, False]
    )
    .head(20)
)

=== ENTITY-LEVEL DIAGNOSTICS ===
Macro F0.5: 0.9354501269958295
Mean predicted matches/S1: 3.443
Median predicted matches/S1: 3.0
95th percentile predicted matches/S1: 6.0
Mean true matches/S1: 3.4755
Total TP: 6841
Total FP: 45
Total FN: 110

Singleton S1 count:
113
Singletons with >=1 prediction:
5

Worst 20 S1s by F0.5:


,s1_rid,true_count,pred_count,tp,fp,fn,precision,recall,f05
243,1155,0,1,0,1,0,0.0,1.0,0.0
321,1536,0,1,0,1,0,0.0,1.0,0.0
380,1793,0,1,0,1,0,0.0,1.0,0.0
1093,5467,0,1,0,1,0,0.0,1.0,0.0
1220,6101,0,1,0,1,0,0.0,1.0,0.0
25,93,0,0,0,0,0,0.0,1.0,0.0
26,95,0,0,0,0,0,0.0,1.0,0.0
39,157,0,0,0,0,0,0.0,1.0,0.0
86,400,0,0,0,0,0,0.0,1.0,0.0
109,513,0,0,0,0,0,0.0,1.0,0.0


In [75]:
# ============================================================
# 24G. MODEL FEATURE IMPORTANCE
# ============================================================

importance10k = pd.DataFrame(
    {
        "feature": ALL_FEATURES,
        "importance": model_all10k.get_feature_importance(),
    }
).sort_values(
    "importance",
    ascending=False,
).reset_index(drop=True)

display(importance10k)

,feature,importance
0,address_len_ratio,18.176811
1,name_tfidf_char_cosine,12.477004
2,address_token_set,12.272266
3,name_tfidf_word_cosine,6.404538
4,address_tfidf_char_cosine,5.121608
5,name_ratio_ascii,5.097677
6,block_score,4.228484
7,name_ratio_native,4.039615
8,country_match,3.317382
9,number_overlap,3.148241


In [76]:
# ============================================================
# 24H. PROBABILITY DISTRIBUTION
# ============================================================

print("=== PROBABILITY DISTRIBUTION ===")

display(
    valid_diag.groupby("label")["prob"].describe()
)

print()
print("Positive probabilities below threshold:")

print(
    (
        valid_diag.loc[
            valid_diag["label"] == 1,
            "prob"
        ] < BEST_THRESHOLD
    ).mean()
)

print()
print("Negative probabilities above threshold:")

print(
    (
        valid_diag.loc[
            valid_diag["label"] == 0,
            "prob"
        ] >= BEST_THRESHOLD
    ).mean()
)

=== PROBABILITY DISTRIBUTION ===


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,136107.0,0.000940,0.019652,3.084215e-09,3.409391e-07,0.000001,0.000007,0.999541
1,6951.0,0.982772,0.092694,1.771944e-03,9.991964e-01,0.999862,0.999973,0.999999



Positive probabilities below threshold:
0.015825061142281688

Negative probabilities above threshold:
0.00033062223103881507


In [77]:
# ============================================================
# 25A. CORRECT COMPETITION-STYLE ENTITY F0.5
# ============================================================

BEST_THRESHOLD = float(best_all["threshold"])

# ------------------------------------------------------------
# Map every GT candidate entity ID -> candidate row ID
# ------------------------------------------------------------

cand_entity_to_rid10k = dict(
    zip(
        CAND["entity_id"].astype(str),
        CAND["cand_rid"].astype(int),
    )
)

# Full ground-truth sets expressed as CAND row IDs.
FULL_GT_RID10K = {}

for s1_rid in valid_s1:

    s1_entity_id = s1_rid_to_entity[int(s1_rid)]

    gt_entity_ids = GT_MAP.get(
        s1_entity_id,
        set(),
    )

    FULL_GT_RID10K[int(s1_rid)] = {
        cand_entity_to_rid10k[x]
        for x in gt_entity_ids
        if x in cand_entity_to_rid10k
    }


# ------------------------------------------------------------
# Predictions from Model B
# ------------------------------------------------------------

valid_predictions10k = valid_df[
    [
        "s1_rid",
        "cand_rid",
    ]
].copy()

valid_predictions10k["prob"] = valid_prob_all


# ------------------------------------------------------------
# Correct entity-level F0.5
# ------------------------------------------------------------

def correct_entity_f05(
    valid_s1_ids,
    prediction_df,
    full_gt_map,
    threshold,
):
    pred_map = {
        int(s1_rid): set(
            g.loc[
                g["prob"] >= threshold,
                "cand_rid",
            ].astype(int)
        )
        for s1_rid, g
        in prediction_df.groupby("s1_rid")
    }

    rows = []

    beta2 = 0.25

    for s1_rid in valid_s1_ids:

        s1_rid = int(s1_rid)

        true_set = full_gt_map.get(
            s1_rid,
            set(),
        )

        pred_set = pred_map.get(
            s1_rid,
            set(),
        )

        # ----------------------------------------------------
        # Correct singleton handling
        # ----------------------------------------------------

        if len(true_set) == 0:

            f05 = (
                1.0
                if len(pred_set) == 0
                else 0.0
            )

            precision = (
                1.0
                if len(pred_set) == 0
                else 0.0
            )

            recall = 1.0

            tp = 0
            fp = len(pred_set)
            fn = 0

        else:

            tp = len(
                true_set & pred_set
            )

            fp = len(
                pred_set - true_set
            )

            fn = len(
                true_set - pred_set
            )

            precision = (
                tp / (tp + fp)
                if tp + fp > 0
                else 0.0
            )

            recall = (
                tp / (tp + fn)
                if tp + fn > 0
                else 0.0
            )

            if precision == 0 or recall == 0:
                f05 = 0.0
            else:
                f05 = (
                    1.25
                    * precision
                    * recall
                    / (
                        0.25 * precision
                        + recall
                    )
                )

        rows.append(
            {
                "s1_rid": s1_rid,
                "true_count": len(true_set),
                "pred_count": len(pred_set),
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": precision,
                "recall": recall,
                "f05": f05,
            }
        )

    result = pd.DataFrame(rows)

    return result


correct_diag10k = correct_entity_f05(
    valid_s1,
    valid_predictions10k,
    FULL_GT_RID10K,
    BEST_THRESHOLD,
)

correct_f05 = correct_diag10k["f05"].mean()

print("=== CORRECT ENTITY-LEVEL SCORE ===")
print("Threshold:", BEST_THRESHOLD)
print("Macro F0.5:", correct_f05)

print()
print("S1 entities evaluated:", len(correct_diag10k))

print(
    "True singleton S1s:",
    (correct_diag10k["true_count"] == 0).sum()
)

print(
    "Correctly predicted singletons:",
    (
        (correct_diag10k["true_count"] == 0)
        &
        (correct_diag10k["pred_count"] == 0)
    ).sum()
)

print(
    "False-positive singleton S1s:",
    (
        (correct_diag10k["true_count"] == 0)
        &
        (correct_diag10k["pred_count"] > 0)
    ).sum()
)

print()
print("Total TP:", correct_diag10k["tp"].sum())
print("Total FP:", correct_diag10k["fp"].sum())
print("Total FN:", correct_diag10k["fn"].sum())

print()
print("Mean predicted matches/S1:")
print(correct_diag10k["pred_count"].mean())

print()
print("Worst 20 entities:")
display(
    correct_diag10k
    .sort_values(
        ["f05", "fp", "fn"],
        ascending=[True, False, False],
    )
    .head(20)
)

=== CORRECT ENTITY-LEVEL SCORE ===
Threshold: 0.6499999999999997
Macro F0.5: 0.98675636909547

S1 entities evaluated: 2000
True singleton S1s: 112
Correctly predicted singletons: 107
False-positive singleton S1s: 5

Total TP: 6841
Total FP: 45
Total FN: 164

Mean predicted matches/S1:
3.443

Worst 20 entities:


,s1_rid,true_count,pred_count,tp,fp,fn,precision,recall,f05
696,5467,0,1,0,1,0,0.000000,1.000000,0.000000
699,1155,0,1,0,1,0,0.000000,1.000000,0.000000
1048,1793,0,1,0,1,0,0.000000,1.000000,0.000000
1522,6101,0,1,0,1,0,0.000000,1.000000,0.000000
1752,1536,0,1,0,1,0,0.000000,1.000000,0.000000
118,2344,1,0,0,0,1,0.000000,0.000000,0.000000
1318,6117,1,0,0,0,1,0.000000,0.000000,0.000000
294,7968,1,3,1,2,0,0.333333,1.000000,0.384615
1090,6800,2,4,2,2,0,0.500000,1.000000,0.555556
1239,6449,2,4,2,2,0,0.500000,1.000000,0.555556


In [78]:
# ============================================================
# 25B. CORRECT THRESHOLD SWEEP
# ============================================================

thresholds = np.arange(
    0.10,
    0.991,
    0.01,
)

threshold_results10k = []

for threshold in thresholds:

    diag = correct_entity_f05(
        valid_s1,
        valid_predictions10k,
        FULL_GT_RID10K,
        threshold,
    )

    threshold_results10k.append(
        {
            "threshold": threshold,
            "macro_f05": diag["f05"].mean(),
            "total_tp": diag["tp"].sum(),
            "total_fp": diag["fp"].sum(),
            "total_fn": diag["fn"].sum(),
            "avg_predictions": diag["pred_count"].mean(),
        }
    )

threshold_results10k = (
    pd.DataFrame(threshold_results10k)
    .sort_values(
        "macro_f05",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("=== CORRECT THRESHOLD SWEEP ===")
display(
    threshold_results10k.head(15)
)

=== CORRECT THRESHOLD SWEEP ===


,threshold,macro_f05,total_tp,total_fp,total_fn,avg_predictions
0,0.60,0.986849,6856,51,149,3.4535
1,0.61,0.986835,6853,49,152,3.4510
2,0.59,0.986828,6858,52,147,3.4550
3,0.75,0.986817,6817,40,188,3.4285
4,0.74,0.986809,6819,41,186,3.4300
5,0.76,0.986806,6813,38,192,3.4255
6,0.62,0.986793,6850,48,155,3.4490
7,0.63,0.986772,6846,47,159,3.4465
8,0.65,0.986756,6841,45,164,3.4430
9,0.64,0.986755,6843,46,162,3.4445


In [79]:
# ============================================================
# 26A. CANDIDATE-GENERATION CEILING
# ============================================================

# target_union contains every candidate available to the
# feature/model stage.

# Build target-union candidate sets.
union_map10k = {
    int(s1_rid): set(
        g["cand_rid"].astype(int)
    )
    for s1_rid, g
    in target_union.groupby("s1_rid")
}

ceiling_rows10k = []

for s1_rid in valid_s1:

    s1_rid = int(s1_rid)

    true_set = FULL_GT_RID10K.get(
        s1_rid,
        set(),
    )

    available_set = union_map10k.get(
        s1_rid,
        set(),
    )

    # Perfect model:
    # predict exactly the true matches that survived blocking.
    predicted_set = true_set & available_set

    if len(true_set) == 0:
        f05 = 1.0 if len(predicted_set) == 0 else 0.0

        tp = 0
        fp = 0
        fn = 0

    else:

        tp = len(predicted_set)
        fp = 0
        fn = len(true_set - predicted_set)

        precision = 1.0
        recall = tp / len(true_set)

        f05 = (
            1.25 * precision * recall
            / (0.25 * precision + recall)
        )

    ceiling_rows10k.append(
        {
            "s1_rid": s1_rid,
            "true_count": len(true_set),
            "available_true_count": len(predicted_set),
            "blocked_misses": len(
                true_set - available_set
            ),
            "f05": f05,
        }
    )

ceiling10k = pd.DataFrame(
    ceiling_rows10k
)

print("=== BLOCKING CEILING ===")

print(
    "Macro F0.5:",
    ceiling10k["f05"].mean()
)

print(
    "Total blocked-out true pairs:",
    ceiling10k["blocked_misses"].sum()
)

print(
    "S1s with >=1 blocked-out true pair:",
    (
        ceiling10k["blocked_misses"] > 0
    ).sum()
)

print(
    "Perfect-within-candidates TP:",
    ceiling10k["available_true_count"].sum()
)

print()
display(
    ceiling10k.loc[
        ceiling10k["blocked_misses"] > 0
    ]
    .sort_values(
        "blocked_misses",
        ascending=False,
    )
    .head(20)
)

=== BLOCKING CEILING ===
Macro F0.5: 0.9973313985170603
Total blocked-out true pairs: 54
S1s with >=1 blocked-out true pair: 45
Perfect-within-candidates TP: 6951



,s1_rid,true_count,available_true_count,blocked_misses,f05
1121,4047,5,2,3,0.769231
456,3007,4,2,2,0.833333
1848,2955,7,5,2,0.925926
995,3921,5,3,2,0.882353
181,3100,7,5,2,0.925926
797,8946,4,2,2,0.833333
332,1617,3,1,2,0.714286
1214,2701,3,1,2,0.714286
1307,347,4,3,1,0.937500
1388,1458,5,4,1,0.952381


In [80]:
# ============================================================
# 26B. CORRECT MODEL A vs MODEL B
# ============================================================

# Model A predictions
valid_predictions_base10k = valid_df[
    [
        "s1_rid",
        "cand_rid",
    ]
].copy()

valid_predictions_base10k["prob"] = valid_prob_base


def sweep_correct_f05(
    valid_s1_ids,
    prediction_df,
    full_gt_map,
    thresholds,
):
    rows = []

    for threshold in thresholds:

        diag = correct_entity_f05(
            valid_s1_ids,
            prediction_df,
            full_gt_map,
            threshold,
        )

        rows.append(
            {
                "threshold": threshold,
                "macro_f05": diag["f05"].mean(),
                "tp": diag["tp"].sum(),
                "fp": diag["fp"].sum(),
                "fn": diag["fn"].sum(),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            "macro_f05",
            ascending=False,
        )
        .reset_index(drop=True)
    )


threshold_grid10k = np.arange(
    0.30,
    0.991,
    0.01,
)

base_correct_sweep10k = sweep_correct_f05(
    valid_s1,
    valid_predictions_base10k,
    FULL_GT_RID10K,
    threshold_grid10k,
)

all_correct_sweep10k = sweep_correct_f05(
    valid_s1,
    valid_predictions10k,
    FULL_GT_RID10K,
    threshold_grid10k,
)

best_base_correct = (
    base_correct_sweep10k.iloc[0]
)

best_all_correct = (
    all_correct_sweep10k.iloc[0]
)

comparison_correct10k = pd.DataFrame(
    [
        {
            "model": "Base",
            "features": len(BASE_FEATURES),
            "pair_auc": base_auc,
            "macro_f05": best_base_correct["macro_f05"],
            "threshold": best_base_correct["threshold"],
            "tp": best_base_correct["tp"],
            "fp": best_base_correct["fp"],
            "fn": best_base_correct["fn"],
        },
        {
            "model": "Base + TF-IDF",
            "features": len(ALL_FEATURES),
            "pair_auc": all_auc,
            "macro_f05": best_all_correct["macro_f05"],
            "threshold": best_all_correct["threshold"],
            "tp": best_all_correct["tp"],
            "fp": best_all_correct["fp"],
            "fn": best_all_correct["fn"],
        },
    ]
)

display(comparison_correct10k)


,model,features,pair_auc,macro_f05,threshold,tp,fp,fn
0,Base,31,0.999945,0.985043,0.65,6825.0,60.0,180.0
1,Base + TF-IDF,35,0.999956,0.986849,0.60,6856.0,51.0,149.0


In [81]:
# ============================================================
# 26C. INSPECT FALSE-POSITIVE SINGLETONS
# ============================================================

singleton_fp10k = correct_diag10k[
    (correct_diag10k["true_count"] == 0)
    &
    (correct_diag10k["pred_count"] > 0)
].copy()

print("False-positive singletons:")
display(singleton_fp10k)

for s1_rid in singleton_fp10k["s1_rid"]:

    print("\n" + "=" * 80)
    print("S1 RID:", s1_rid)

    s1_row = S1[
        S1["s1_rid"] == s1_rid
    ].iloc[0]

    print(
        "S1 entity_id:",
        s1_row["entity_id"]
    )

    print(
        "Name:",
        s1_row["name_native"]
    )

    print(
        "Address:",
        s1_row["addr_native"]
    )

    pred = valid_predictions10k[
        (valid_predictions10k["s1_rid"] == s1_rid)
        &
        (valid_predictions10k["prob"] >= BEST_THRESHOLD)
    ].sort_values(
        "prob",
        ascending=False,
    )

    display(
        pred[
            [
                "cand_rid",
                "prob",
            ]
        ]
    )

    for cand_rid in pred["cand_rid"]:

        c = CAND[
            CAND["cand_rid"] == int(cand_rid)
        ].iloc[0]

        print(
            "Candidate:",
            c["entity_id"],
            "|",
            c["name_native"],
            "|",
            c["addr_native"],
            "| country:",
            c["country"],
        )

False-positive singletons:


,s1_rid,true_count,pred_count,tp,fp,fn,precision,recall,f05
696,5467,0,1,0,1,0,0.0,1.0,0.0
699,1155,0,1,0,1,0,0.0,1.0,0.0
1048,1793,0,1,0,1,0,0.0,1.0,0.0
1522,6101,0,1,0,1,0,0.0,1.0,0.0
1752,1536,0,1,0,1,0,0.0,1.0,0.0



S1 RID: 5467
S1 entity_id: S1-483025271
Name: firefighters local 412
Address: 1439 markley street norristown borough pa


,cand_rid,prob
391695,7468,0.97815


Candidate: S3-586353403 | firefighters l0cal 720 |  | country: us

S1 RID: 1155
S1 entity_id: S1-451804668
Name: industrial textile consultants pc
Address: 1837 6th avenue az mesa


...

Candidate: S3-438580182 | main street hypnosis llc |  | country: us

S1 RID: 6101
S1 entity_id: S1-656235422
Name: urgent care physicians
Address: 326 meadowcreek drive wadsworth city oh


,cand_rid,prob
437069,45226,0.891184


Candidate: S2-651444562 | urgent care physicians co |  | country: us

S1 RID: 1536
S1 entity_id: S1-658200019
Name: ridge fellowship
Address: 620 cokesbury avenue baltimore md


,cand_rid,prob
109992,45750,0.862266


Candidate: S2-449612112 | ridge fellowship co |  | country: us


In [82]:
# ============================================================
# 26D. MODEL ERROR ANALYSIS
# ============================================================

# Model B at the currently best threshold.
ERR_THRESHOLD = 0.60

err = valid_df[
    [
        "s1_rid",
        "cand_rid",
        "label",
        "block_score",
    ]
].copy()

err["prob"] = valid_prob_all


# ------------------------------------------------------------
# False negatives
# ------------------------------------------------------------

false_neg10k = err[
    (err["label"] == 1)
    &
    (err["prob"] < ERR_THRESHOLD)
].copy()

# ------------------------------------------------------------
# False positives
# ------------------------------------------------------------

false_pos10k = err[
    (err["label"] == 0)
    &
    (err["prob"] >= ERR_THRESHOLD)
].copy()


print("=== MODEL ERRORS ===")
print("False negatives:", len(false_neg10k))
print("False positives:", len(false_pos10k))

print()
print("FN probability distribution:")
print(false_neg10k["prob"].describe())

print()
print("FP probability distribution:")
print(false_pos10k["prob"].describe())


# ------------------------------------------------------------
# Hardest false negatives
# ------------------------------------------------------------

print()
print("=== HIGHEST-CONFIDENCE FALSE NEGATIVES ===")

fn_show = (
    false_neg10k
    .sort_values("prob", ascending=False)
    .head(20)
)

display(
    fn_show.merge(
        features10k.drop(
            columns=["label"],
            errors="ignore",
        ),
        on=["s1_rid", "cand_rid"],
        how="left",
        suffixes=("", "_feature"),
    )
)


# ------------------------------------------------------------
# Highest-confidence false positives
# ------------------------------------------------------------

print()
print("=== HIGHEST-CONFIDENCE FALSE POSITIVES ===")

fp_show = (
    false_pos10k
    .sort_values("prob", ascending=False)
    .head(20)
)

display(
    fp_show.merge(
        features10k.drop(
            columns=["label"],
            errors="ignore",
        ),
        on=["s1_rid", "cand_rid"],
        how="left",
        suffixes=("", "_feature"),
    )
)

=== MODEL ERRORS ===
False negatives: 95
False positives: 51

FN probability distribution:
count    95.000000
mean      0.273055
std       0.196346
min       0.001772
25%       0.091895
50%       0.244259
75%       0.459709
max       0.597499
Name: prob, dtype: float64

FP probability distribution:
count    51.000000
mean      0.853039
std       0.120050
min       0.602911
25%       0.773305
50%       0.875950
75%       0.953120
max       0.999541
Name: prob, dtype: float64

=== HIGHEST-CONFIDENCE FALSE NEGATIVES ===


,s1_rid,cand_rid,label,block_score,prob,block_score_feature,name_ratio_native,name_ratio_ascii,name_token_sort,name_token_set,...,address_missing,candidate_address_missing,name_len_ratio,address_len_ratio,name_addr_ratio_mean,max_field_ratio,name_tfidf_char_cosine,name_tfidf_word_cosine,address_tfidf_char_cosine,address_tfidf_word_cosine
0,7415,30050,1,18.75,0.597499,18.75,0.742857,0.742857,0.742857,1.000000,...,0,1,0.590909,0.000000,0.371429,0.742857,0.763218,0.705874,0.000000,0.000000
1,6838,20990,1,15.00,0.596749,15.00,0.437500,0.437500,0.562500,0.608696,...,0,0,1.000000,0.965517,0.622259,0.807018,0.159798,0.075808,0.551552,0.249492
2,4216,19085,1,0.50,0.587549,0.50,0.125000,0.764706,0.705882,0.705882,...,0,0,0.888889,0.631579,0.592030,0.764706,0.293020,0.107429,0.258523,0.133220
3,8142,33934,1,1.50,0.579738,1.50,0.862069,0.862069,0.862069,1.000000,...,0,1,0.757576,0.000000,0.431034,0.862069,0.949943,0.960466,0.000000,0.000000
4,1142,8354,1,21.50,0.571956,21.50,0.730769,0.730769,0.730769,0.844444,...,0,0,1.000000,0.861111,0.589265,0.730769,0.722269,0.522761,0.100282,0.151095
5,1901,17896,1,21.00,0.559984,21.00,0.250000,0.250000,0.250000,0.250000,...,0,0,1.000000,1.000000,0.477273,0.704545,0.000000,0.000000,0.470225,0.232696
6,2138,19496,1,8.50,0.557128,8.50,0.622222,0.622222,0.800000,0.800000,...,0,1,0.956522,0.000000,0.311111,0.622222,0.501000,0.419482,0.000000,0.000000
7,7315,40934,1,20.00,0.552493,20.00,0.545455,0.545455,0.545455,0.545455,...,0,0,0.571429,0.893617,0.542390,0.545455,0.000000,0.000000,0.708506,0.535796
8,7653,37105,1,8.50,0.552370,8.50,0.755556,0.755556,0.755556,0.755556,...,0,0,0.730769,0.357143,0.627778,0.755556,0.258016,0.188350,0.302854,0.302475
9,9200,49039,1,19.00,0.545605,19.00,0.103448,0.393939,0.363636,0.363636,...,0,0,0.571429,0.946667,0.525737,0.657534,0.009135,0.000000,0.815548,0.746033



=== HIGHEST-CONFIDENCE FALSE POSITIVES ===


,s1_rid,cand_rid,label,block_score,prob,block_score_feature,name_ratio_native,name_ratio_ascii,name_token_sort,name_token_set,...,address_missing,candidate_address_missing,name_len_ratio,address_len_ratio,name_addr_ratio_mean,max_field_ratio,name_tfidf_char_cosine,name_tfidf_word_cosine,address_tfidf_char_cosine,address_tfidf_word_cosine
0,3309,14438,0,29.00,0.999541,29.00,0.920000,0.920000,0.920000,0.920000,...,0,0,1.000000,0.945946,0.918333,0.920000,0.885469,0.807776,0.701505,0.467670
1,3210,50982,0,21.00,0.998541,21.00,0.820513,0.820513,0.820513,1.000000,...,0,0,0.695652,0.965517,0.813765,0.820513,0.789408,0.752497,0.596202,0.168506
2,5538,16863,0,54.50,0.998307,54.50,0.629630,0.629630,0.629630,1.000000,...,0,0,0.459459,1.000000,0.738426,0.847222,0.758271,0.770862,0.877224,0.693424
3,9030,26697,0,16.50,0.994134,16.50,0.625000,0.625000,1.000000,1.000000,...,0,1,1.000000,0.000000,0.312500,0.625000,0.679792,0.405033,0.000000,0.000000
4,8158,51410,0,52.50,0.989609,52.50,0.227273,0.227273,0.227273,0.227273,...,0,0,0.257143,0.882353,0.527699,0.828125,0.004540,0.000000,0.767610,0.702016
5,4801,34828,0,20.75,0.986028,20.75,0.920000,0.920000,0.920000,1.000000,...,0,1,0.851852,0.000000,0.460000,0.920000,0.925823,0.919688,0.000000,0.000000
6,2474,45612,0,8.50,0.984516,8.50,0.166667,0.166667,0.166667,0.166667,...,0,0,0.200000,0.750000,0.511905,0.857143,0.000000,0.000000,0.747365,0.703163
7,5467,7468,0,0.50,0.978150,0.50,0.863636,0.863636,0.863636,0.863636,...,0,1,1.000000,0.000000,0.431818,0.863636,0.552427,0.180769,0.000000,0.000000
8,4993,42207,0,29.50,0.972937,29.50,0.242424,0.242424,0.242424,0.242424,...,0,0,0.941176,1.000000,0.621212,1.000000,0.000000,0.000000,1.000000,1.000000
9,6800,15209,0,24.50,0.970222,24.50,0.210526,0.210526,0.263158,0.285714,...,0,0,0.652174,0.833333,0.544657,0.878788,0.000000,0.000000,0.783750,0.425327


In [83]:
# ============================================================
# 26E. RAW RECORDS FOR HARD ERRORS
# ============================================================

def show_error_pairs(error_df, title, n=15):

    print("\n" + "=" * 100)
    print(title)

    for _, e in (
        error_df
        .sort_values("prob", ascending=False)
        .head(n)
        .iterrows()
    ):

        s1_rid = int(e["s1_rid"])
        cand_rid = int(e["cand_rid"])

        s = S1[
            S1["s1_rid"] == s1_rid
        ].iloc[0]

        c = CAND[
            CAND["cand_rid"] == cand_rid
        ].iloc[0]

        print("\n" + "-" * 100)

        print(
            "S1:",
            s["entity_id"],
            "| country:",
            s["country"],
        )

        print(
            "S1 name:",
            s["name_native"],
        )

        print(
            "S1 address:",
            s["addr_native"],
        )

        print()

        print(
            "Candidate:",
            c["entity_id"],
            "| source:",
            c["source"],
            "| country:",
            c["country"],
        )

        print(
            "Candidate name:",
            c["name_native"],
        )

        print(
            "Candidate address:",
            c["addr_native"],
        )

        print()

        print(
            "Probability:",
            round(float(e["prob"]), 6),
            "| block_score:",
            round(float(e["block_score"]), 3),
        )


show_error_pairs(
    false_neg10k,
    "HIGHEST-CONFIDENCE FALSE NEGATIVES",
    n=20,
)

show_error_pairs(
    false_pos10k,
    "HIGHEST-CONFIDENCE FALSE POSITIVES",
    n=20,
)


HIGHEST-CONFIDENCE FALSE NEGATIVES

----------------------------------------------------------------------------------------------------
S1: S1-503547265 | country: us
S1 name: torie hensley nova inc
S1 address: 200 14th court grimes ia

Candidate: S3-292827882 | source: S3 | country: us
Candidate name: torie hensley
Candidate address: 

Probability: 0.597499 | block_score: 18.75

----------------------------------------------------------------------------------------------------
S1: S1-398356006 | country: us
S1 name: i w tailwind llc
S1 address: 1803 9th street brownwood tx

Candidate: S3-43612720 | source: S3 | country: us
Candidate name: i w llc services
Candidate address: 1803 ninth st brownwood texas

Probability: 0.596749 | block_score: 15.0

----------------------------------------------------------------------------------------------------
S1: S1-435182020 | country: india
S1 name: golden shiv care private limited
S1 address: 52 ganga puram ghaziabad uttar pradesh

Candidate:

In [84]:
# ============================================================
# 26F. FIELD-AGREEMENT ERROR ANALYSIS
# ============================================================

analysis_cols = [
    "s1_rid",
    "cand_rid",
    "label",
    "name_ratio_ascii",
    "address_ratio_ascii",
    "name_tfidf_char_cosine",
    "name_tfidf_word_cosine",
    "address_tfidf_char_cosine",
    "address_tfidf_word_cosine",
]

field_analysis10k = (
    valid_df[
        analysis_cols
    ]
    .copy()
)

field_analysis10k["prob"] = valid_prob_all


# A deliberately simple descriptive definition.
NAME_STRONG = 0.80
ADDRESS_STRONG = 0.70

field_analysis10k["name_strong"] = (
    field_analysis10k["name_ratio_ascii"]
    >= NAME_STRONG
)

field_analysis10k["address_strong"] = (
    field_analysis10k["address_ratio_ascii"]
    >= ADDRESS_STRONG
)


def field_regime(row):

    if row["name_strong"] and row["address_strong"]:
        return "name+address strong"

    if row["name_strong"] and not row["address_strong"]:
        return "name strong / address weak"

    if not row["name_strong"] and row["address_strong"]:
        return "name weak / address strong"

    return "name+address weak"


field_analysis10k["regime"] = (
    field_analysis10k
    .apply(field_regime, axis=1)
)


# ------------------------------------------------------------
# Error rates by regime
# ------------------------------------------------------------

regime_rows = []

for regime, g in (
    field_analysis10k
    .groupby("regime")
):

    positives = (
        g["label"] == 1
    )

    negatives = (
        g["label"] == 0
    )

    missed_positive_rate = (
        (
            positives
            &
            (g["prob"] < ERR_THRESHOLD)
        ).sum()
        / positives.sum()
        if positives.sum() > 0
        else np.nan
    )

    false_positive_rate = (
        (
            negatives
            &
            (g["prob"] >= ERR_THRESHOLD)
        ).sum()
        / negatives.sum()
        if negatives.sum() > 0
        else np.nan
    )

    regime_rows.append(
        {
            "regime": regime,
            "rows": len(g),
            "positives": int(positives.sum()),
            "negatives": int(negatives.sum()),
            "missed_positive_rate": missed_positive_rate,
            "false_positive_rate": false_positive_rate,
        }
    )

regime_summary10k = (
    pd.DataFrame(regime_rows)
    .sort_values(
        "rows",
        ascending=False,
    )
)

display(regime_summary10k)

,regime,rows,positives,negatives,missed_positive_rate,false_positive_rate
3,name+address weak,133107,694,132413,0.089337,0.000083
0,name strong / address weak,4357,1312,3045,0.009909,0.007882
2,name+address strong,3332,3329,3,0.000000,0.666667
1,name weak / address strong,2262,1616,646,0.012376,0.021672


In [85]:
# ============================================================
# 27A. FIELD CONSISTENCY FEATURES
# ============================================================

def add_consistency_features(df):
    out = df.copy()

    nr = out["name_ratio_ascii"].to_numpy(dtype=np.float32)
    ar = out["address_ratio_ascii"].to_numpy(dtype=np.float32)

    nc = out["name_tfidf_char_cosine"].to_numpy(dtype=np.float32)
    nw = out["name_tfidf_word_cosine"].to_numpy(dtype=np.float32)

    ac = out["address_tfidf_char_cosine"].to_numpy(dtype=np.float32)
    aw = out["address_tfidf_word_cosine"].to_numpy(dtype=np.float32)

    name_strength = np.maximum.reduce([nr, nc, nw])
    address_strength = np.maximum.reduce([ar, ac, aw])

    out["name_strength"] = name_strength
    out["address_strength"] = address_strength

    out["field_strength_min"] = np.minimum(
        name_strength,
        address_strength,
    )

    out["field_strength_max"] = np.maximum(
        name_strength,
        address_strength,
    )

    out["field_strength_gap"] = np.abs(
        name_strength - address_strength
    )

    out["field_strength_product"] = (
        name_strength * address_strength
    )

    out["both_fields_present"] = (
        (
            out["address_missing"] == 0
        )
        &
        (
            out["candidate_address_missing"] == 0
        )
    ).astype(np.int8)

    out["candidate_address_missing_and_strong_name"] = (
        (
            out["candidate_address_missing"] == 1
        )
        &
        (
            name_strength >= 0.75
        )
    ).astype(np.int8)

    out["strong_address_weak_name"] = (
        (
            address_strength >= 0.70
        )
        &
        (
            name_strength < 0.50
        )
    ).astype(np.int8)

    out["strong_both_fields"] = (
        (
            name_strength >= 0.75
        )
        &
        (
            address_strength >= 0.70
        )
    ).astype(np.int8)

    out["weak_both_fields"] = (
        (
            name_strength < 0.50
        )
        &
        (
            address_strength < 0.50
        )
    ).astype(np.int8)

    out["name_address_agreement"] = (
        (
            name_strength >= 0.70
        )
        &
        (
            address_strength >= 0.60
        )
    ).astype(np.int8)

    return out


features_consistency10k = add_consistency_features(
    features10k
)

CONSISTENCY_FEATURES = [
    "name_strength",
    "address_strength",
    "field_strength_min",
    "field_strength_max",
    "field_strength_gap",
    "field_strength_product",
    "both_fields_present",
    "candidate_address_missing_and_strong_name",
    "strong_address_weak_name",
    "strong_both_fields",
    "weak_both_fields",
    "name_address_agreement",
]

print("Added:", len(CONSISTENCY_FEATURES), "features")

display(
    features_consistency10k[
        CONSISTENCY_FEATURES
    ].describe()
)

Added: 12 features


,name_strength,address_strength,field_strength_min,field_strength_max,field_strength_gap,field_strength_product,both_fields_present,candidate_address_missing_and_strong_name,strong_address_weak_name,strong_both_fields,weak_both_fields,name_address_agreement
count,715739.000000,715739.000000,715739.000000,715739.000000,715739.000000,715739.000000,715739.000000,715739.000000,715739.000000,715739.000000,715739.000000,715739.000000
mean,0.490191,0.366482,0.338033,0.518640,0.180607,0.185977,0.963999,0.003600,0.004799,0.030388,0.513898,0.036333
std,0.168996,0.148332,0.131737,0.156644,0.143496,0.138683,0.186292,0.059896,0.069110,0.171653,0.499807,0.187118
min,0.000000,0.000000,0.000000,0.117647,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.363636,0.301887,0.285714,0.400000,0.069398,0.118368,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.468085,0.347826,0.328767,0.488889,0.149914,0.159072,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,0.588235,0.400000,0.373333,0.600000,0.257908,0.209946,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [86]:
# ============================================================
# 27B. MODEL C — BASE + TF-IDF + CONSISTENCY
# ============================================================

CONSISTENCY_ALL_FEATURES = (
    ALL_FEATURES
    + CONSISTENCY_FEATURES
)

train_c = features_consistency10k[
    features_consistency10k["s1_rid"].isin(train_s1)
].copy()

valid_c = features_consistency10k[
    features_consistency10k["s1_rid"].isin(valid_s1)
].copy()

X_train_c = train_c[CONSISTENCY_ALL_FEATURES]
X_valid_c = valid_c[CONSISTENCY_ALL_FEATURES]

y_train_c = train_c["label"]
y_valid_c = valid_c["label"]

print("Training Model C...")

t0 = time.time()

model_c10k = CatBoostClassifier(
    iterations=700,
    depth=8,
    learning_rate=0.07,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False,
)

model_c10k.fit(
    X_train_c,
    y_train_c,
    eval_set=(X_valid_c, y_valid_c),
    early_stopping_rounds=75,
)

model_c_time = time.time() - t0

valid_prob_c = model_c10k.predict_proba(
    X_valid_c
)[:, 1]

c_pair_auc = roc_auc_score(
    y_valid_c,
    valid_prob_c,
)

print()
print("Model C pair AUC:", round(c_pair_auc, 6))
print("Training time:", round(model_c_time, 2), "sec")

Training Model C...
0:	test: 0.9982397	best: 0.9982397 (0)	total: 89.2ms	remaining: 1m 2s
100:	test: 0.9999287	best: 0.9999287 (100)	total: 5.07s	remaining: 30s
200:	test: 0.9999534	best: 0.9999534 (200)	total: 9.95s	remaining: 24.7s
300:	test: 0.9999583	best: 0.9999583 (300)	total: 15s	remaining: 19.9s
400:	test: 0.9999597	best: 0.9999598 (349)	total: 19.2s	remaining: 14.3s
Stopped by overfitting detector  (75 iterations wait)

bestTest = 0.9999597988
bestIteration = 349

Shrink model to first 350 iterations.

Model C pair AUC: 0.99996
Training time: 20.27 sec


In [87]:
# ============================================================
# 27C. MODEL C — CORRECT F0.5
# ============================================================

valid_predictions_c10k = valid_c[
    [
        "s1_rid",
        "cand_rid",
    ]
].copy()

valid_predictions_c10k["prob"] = valid_prob_c

c_sweep10k = sweep_correct_f05(
    valid_s1,
    valid_predictions_c10k,
    FULL_GT_RID10K,
    threshold_grid10k,
)

best_c10k = c_sweep10k.iloc[0]

comparison_models10k = pd.DataFrame(
    [
        {
            "model": "Base",
            "features": len(BASE_FEATURES),
            "pair_auc": base_auc,
            "macro_f05": best_base_correct["macro_f05"],
            "threshold": best_base_correct["threshold"],
            "tp": best_base_correct["tp"],
            "fp": best_base_correct["fp"],
            "fn": best_base_correct["fn"],
        },
        {
            "model": "Base + TF-IDF",
            "features": len(ALL_FEATURES),
            "pair_auc": all_auc,
            "macro_f05": best_all_correct["macro_f05"],
            "threshold": best_all_correct["threshold"],
            "tp": best_all_correct["tp"],
            "fp": best_all_correct["fp"],
            "fn": best_all_correct["fn"],
        },
        {
            "model": "Base + TF-IDF + Consistency",
            "features": len(CONSISTENCY_ALL_FEATURES),
            "pair_auc": c_pair_auc,
            "macro_f05": best_c10k["macro_f05"],
            "threshold": best_c10k["threshold"],
            "tp": best_c10k["tp"],
            "fp": best_c10k["fp"],
            "fn": best_c10k["fn"],
        },
    ]
)

display(comparison_models10k)

,model,features,pair_auc,macro_f05,threshold,tp,fp,fn
0,Base,31,0.999945,0.985043,0.65,6825.0,60.0,180.0
1,Base + TF-IDF,35,0.999956,0.986849,0.60,6856.0,51.0,149.0
2,Base + TF-IDF + Consistency,47,0.999960,0.986944,0.65,6846.0,49.0,159.0


In [88]:
# ============================================================
# 28A. FRESH HOLDOUT FROM THE CURRENT VALIDATION SET
# ============================================================

from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

tune_s1, holdout_s1 = train_test_split(
    valid_s1,
    test_size=0.50,
    random_state=2026,
)

tune_s1 = np.asarray(tune_s1)
holdout_s1 = np.asarray(holdout_s1)

print("=== FRESH HOLDOUT SPLIT ===")
print("Tune S1:", len(tune_s1))
print("Holdout S1:", len(holdout_s1))

print()
print(
    "Tune singleton count:",
    sum(
        len(FULL_GT_RID10K.get(int(x), set())) == 0
        for x in tune_s1
    )
)

print(
    "Holdout singleton count:",
    sum(
        len(FULL_GT_RID10K.get(int(x), set())) == 0
        for x in holdout_s1
    )
)

=== FRESH HOLDOUT SPLIT ===
Tune S1: 1000
Holdout S1: 1000

Tune singleton count: 56
Holdout singleton count: 56


In [89]:
# ============================================================
# 28B. PREDICTIONS FOR TUNE + HOLDOUT
# ============================================================

# Model B
pred_b10k = valid_df[
    ["s1_rid", "cand_rid"]
].copy()

pred_b10k["prob"] = valid_prob_all


# Model C
pred_c10k = valid_c[
    ["s1_rid", "cand_rid"]
].copy()

pred_c10k["prob"] = valid_prob_c


tune_b = pred_b10k[
    pred_b10k["s1_rid"].isin(tune_s1)
].copy()

holdout_b = pred_b10k[
    pred_b10k["s1_rid"].isin(holdout_s1)
].copy()


tune_c = pred_c10k[
    pred_c10k["s1_rid"].isin(tune_s1)
].copy()

holdout_c = pred_c10k[
    pred_c10k["s1_rid"].isin(holdout_s1)
].copy()


print("Model B tune rows:", len(tune_b))
print("Model B holdout rows:", len(holdout_b))

print("Model C tune rows:", len(tune_c))
print("Model C holdout rows:", len(holdout_c))

Model B tune rows: 71163
Model B holdout rows: 71895
Model C tune rows: 71163
Model C holdout rows: 71895


In [90]:
# ============================================================
# 28C. THRESHOLD TUNING — TUNE SET ONLY
# ============================================================

holdout_threshold_grid = np.arange(
    0.40,
    0.901,
    0.01,
)

b_tune_sweep = sweep_correct_f05(
    tune_s1,
    tune_b,
    FULL_GT_RID10K,
    holdout_threshold_grid,
)

c_tune_sweep = sweep_correct_f05(
    tune_s1,
    tune_c,
    FULL_GT_RID10K,
    holdout_threshold_grid,
)

best_b_tune = b_tune_sweep.iloc[0]
best_c_tune = c_tune_sweep.iloc[0]

print("=== THRESHOLDS SELECTED WITHOUT TOUCHING HOLDOUT ===")

print(
    "Model B:",
    round(float(best_b_tune["threshold"]), 3),
    "| tune F0.5:",
    round(float(best_b_tune["macro_f05"]), 6),
)

print(
    "Model C:",
    round(float(best_c_tune["threshold"]), 3),
    "| tune F0.5:",
    round(float(best_c_tune["macro_f05"]), 6),
)

=== THRESHOLDS SELECTED WITHOUT TOUCHING HOLDOUT ===
Model B: 0.57 | tune F0.5: 0.988067
Model C: 0.45 | tune F0.5: 0.988143


In [91]:
# ============================================================
# 28D. FINAL HOLDOUT EVALUATION
# ============================================================

b_holdout_diag = correct_entity_f05(
    holdout_s1,
    holdout_b,
    FULL_GT_RID10K,
    float(best_b_tune["threshold"]),
)

c_holdout_diag = correct_entity_f05(
    holdout_s1,
    holdout_c,
    FULL_GT_RID10K,
    float(best_c_tune["threshold"]),
)


holdout_comparison10k = pd.DataFrame(
    [
        {
            "model": "Model B",
            "features": len(ALL_FEATURES),
            "threshold": float(
                best_b_tune["threshold"]
            ),
            "macro_f05": b_holdout_diag["f05"].mean(),
            "tp": b_holdout_diag["tp"].sum(),
            "fp": b_holdout_diag["fp"].sum(),
            "fn": b_holdout_diag["fn"].sum(),
            "avg_predictions": b_holdout_diag[
                "pred_count"
            ].mean(),
        },
        {
            "model": "Model C",
            "features": len(CONSISTENCY_ALL_FEATURES),
            "threshold": float(
                best_c_tune["threshold"]
            ),
            "macro_f05": c_holdout_diag["f05"].mean(),
            "tp": c_holdout_diag["tp"].sum(),
            "fp": c_holdout_diag["fp"].sum(),
            "fn": c_holdout_diag["fn"].sum(),
            "avg_predictions": c_holdout_diag[
                "pred_count"
            ].mean(),
        },
    ]
)

print("=== UNTOUCHED HOLDOUT ===")
display(holdout_comparison10k)

=== UNTOUCHED HOLDOUT ===


,model,features,threshold,macro_f05,tp,fp,fn,avg_predictions
0,Model B,35,0.57,0.984325,3463,31,73,3.494
1,Model C,47,0.45,0.983541,3474,42,62,3.516


In [92]:
# ============================================================
# 29A. LEAKAGE-FREE TF-IDF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd
import time

# ------------------------------------------------------------
# Build row positions
# ------------------------------------------------------------

s1_pos = {
    int(rid): i
    for i, rid in enumerate(
        S1["s1_rid"].to_numpy()
    )
}

cand_pos = {
    int(rid): i
    for i, rid in enumerate(
        CAND["cand_rid"].to_numpy()
    )
}

pair_left_idx = np.array(
    [
        s1_pos[int(x)]
        for x in features10k["s1_rid"]
    ],
    dtype=np.int32,
)

pair_right_idx = np.array(
    [
        cand_pos[int(x)]
        for x in features10k["cand_rid"]
    ],
    dtype=np.int32,
)


# ------------------------------------------------------------
# TRAIN-ONLY S1 / CAND STRINGS
# ------------------------------------------------------------

train_s1_mask = S1["s1_rid"].isin(train_s1)

# Candidate records appearing in training pairs.
train_cand_ids = set(
    features10k.loc[
        features10k["s1_rid"].isin(train_s1),
        "cand_rid",
    ].astype(int)
)

train_cand_mask = CAND["cand_rid"].isin(
    train_cand_ids
)


def fit_transform_tfidf_train_only(
    s1_column,
    cand_column,
    analyzer,
    ngram_range,
    min_df,
):

    vec = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        min_df=min_df,
        lowercase=False,
        norm="l2",
        dtype=np.float32,
    )

    fit_text = pd.concat(
        [
            S1.loc[
                train_s1_mask,
                s1_column
            ].fillna(""),
            CAND.loc[
                train_cand_mask,
                cand_column
            ].fillna(""),
        ],
        ignore_index=True,
    )

    vec.fit(fit_text)

    S1_matrix = vec.transform(
        S1[s1_column].fillna("")
    )

    CAND_matrix = vec.transform(
        CAND[cand_column].fillna("")
    )

    cosine = pairwise_sparse_cosine(
        S1_matrix,
        CAND_matrix,
        pair_left_idx,
        pair_right_idx,
        batch_size=50_000,
    )

    return vec, cosine


# ------------------------------------------------------------
# Four leakage-free features
# ------------------------------------------------------------

t0 = time.time()

print("Name char...")
leakfree_name_char_vec, lf_name_char = (
    fit_transform_tfidf_train_only(
        "name_ascii",
        "name_ascii",
        "char",
        (2, 5),
        2,
    )
)

print("Name word...")
leakfree_name_word_vec, lf_name_word = (
    fit_transform_tfidf_train_only(
        "name_ascii",
        "name_ascii",
        "word",
        (1, 2),
        1,
    )
)

print("Address char...")
leakfree_addr_char_vec, lf_addr_char = (
    fit_transform_tfidf_train_only(
        "addr_ascii",
        "addr_ascii",
        "char",
        (2, 5),
        2,
    )
)

print("Address word...")
leakfree_addr_word_vec, lf_addr_word = (
    fit_transform_tfidf_train_only(
        "addr_ascii",
        "addr_ascii",
        "word",
        (1, 2),
        1,
    )
)

features_leakfree10k = features10k.copy()

features_leakfree10k[
    "name_tfidf_char_cosine"
] = lf_name_char

features_leakfree10k[
    "name_tfidf_word_cosine"
] = lf_name_word

features_leakfree10k[
    "address_tfidf_char_cosine"
] = lf_addr_char

features_leakfree10k[
    "address_tfidf_word_cosine"
] = lf_addr_word

print(
    "\nLeakage-free TF-IDF runtime:",
    round(time.time() - t0, 2),
    "sec"
)

Name char...
Name word...
Address char...
Address word...

Leakage-free TF-IDF runtime: 13.24 sec


In [93]:
# ============================================================
# 29B. LEAKAGE-FREE MODEL B
# ============================================================

train_lf = features_leakfree10k[
    features_leakfree10k["s1_rid"].isin(train_s1)
].copy()

tune_lf = features_leakfree10k[
    features_leakfree10k["s1_rid"].isin(tune_s1)
].copy()

holdout_lf = features_leakfree10k[
    features_leakfree10k["s1_rid"].isin(holdout_s1)
].copy()

X_train_lf = train_lf[ALL_FEATURES]
X_tune_lf = tune_lf[ALL_FEATURES]
X_holdout_lf = holdout_lf[ALL_FEATURES]

y_train_lf = train_lf["label"]
y_tune_lf = tune_lf["label"]

print("Training leakage-free Model B...")

t0 = time.time()

model_b_lf10k = CatBoostClassifier(
    iterations=600,
    depth=8,
    learning_rate=0.08,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False,
)

model_b_lf10k.fit(
    X_train_lf,
    y_train_lf,
    eval_set=(X_tune_lf, y_tune_lf),
    early_stopping_rounds=75,
)

print(
    "Training time:",
    round(time.time() - t0, 2),
    "sec"
)

# Predictions
tune_prob_lf = model_b_lf10k.predict_proba(
    X_tune_lf
)[:, 1]

holdout_prob_lf = model_b_lf10k.predict_proba(
    X_holdout_lf
)[:, 1]

# ------------------------------------------------------------
# Threshold tuning ONLY on tune S1
# ------------------------------------------------------------

tune_pred_lf = tune_lf[
    ["s1_rid", "cand_rid"]
].copy()

tune_pred_lf["prob"] = tune_prob_lf

holdout_pred_lf = holdout_lf[
    ["s1_rid", "cand_rid"]
].copy()

holdout_pred_lf["prob"] = holdout_prob_lf


lf_sweep = sweep_correct_f05(
    tune_s1,
    tune_pred_lf,
    FULL_GT_RID10K,
    np.arange(0.40, 0.901, 0.01),
)

best_lf = lf_sweep.iloc[0]

print()
print("=== LEAKAGE-FREE THRESHOLD ===")
print(
    "Threshold:",
    best_lf["threshold"]
)

print(
    "Tune F0.5:",
    best_lf["macro_f05"]
)


# ------------------------------------------------------------
# FINAL HOLDOUT
# ------------------------------------------------------------

lf_holdout_diag = correct_entity_f05(
    holdout_s1,
    holdout_pred_lf,
    FULL_GT_RID10K,
    float(best_lf["threshold"]),
)

print()
print("=== LEAKAGE-FREE HOLDOUT ===")
print(
    "Macro F0.5:",
    lf_holdout_diag["f05"].mean()
)

print(
    "TP:",
    lf_holdout_diag["tp"].sum()
)

print(
    "FP:",
    lf_holdout_diag["fp"].sum()
)

print(
    "FN:",
    lf_holdout_diag["fn"].sum()
)

Training leakage-free Model B...
0:	test: 0.9946398	best: 0.9946398 (0)	total: 50.3ms	remaining: 30.1s
100:	test: 0.9999466	best: 0.9999466 (96)	total: 4.79s	remaining: 23.7s
200:	test: 0.9999586	best: 0.9999586 (200)	total: 9.01s	remaining: 17.9s
300:	test: 0.9999631	best: 0.9999631 (299)	total: 13s	remaining: 12.9s
Stopped by overfitting detector  (75 iterations wait)

bestTest = 0.9999633832
bestIteration = 313

Shrink model to first 314 iterations.
Training time: 16.32 sec

=== LEAKAGE-FREE THRESHOLD ===
Threshold: 0.6800000000000003
Tune F0.5: 0.9880914753181744

=== LEAKAGE-FREE HOLDOUT ===
Macro F0.5: 0.9849001317589453
TP: 3459
FP: 26
FN: 77


In [94]:
# ============================================================
# 30A. EXACT PRODUCTION CANDIDATE POOLS
# ============================================================

PRODUCTION_KS = [50, 75, 100, 150]

exact_candidate_pools10k = {}

tu_ranked = target_union.sort_values(
    ["s1_rid", "block_score"],
    ascending=[True, False],
    kind="mergesort",
)

for k in PRODUCTION_KS:

    pool = (
        tu_ranked
        .groupby("s1_rid", sort=False)
        .head(k)
        [
            [
                "s1_rid",
                "cand_rid",
                "block_score",
            ]
        ]
        .drop_duplicates(
            ["s1_rid", "cand_rid"]
        )
        .reset_index(drop=True)
    )

    exact_candidate_pools10k[k] = pool

    print(
        f"K={k}:",
        len(pool),
        "candidate rows | avg/S1:",
        round(
            len(pool) / len(S1),
            3
        )
    )

K=50: 489897 candidate rows | avg/S1: 48.99
K=75: 715413 candidate rows | avg/S1: 71.541
K=100: 916742 candidate rows | avg/S1: 91.674
K=150: 1231311 candidate rows | avg/S1: 123.131


In [95]:
# ============================================================
# 30B. EXACT-CANDIDATE MODEL EVALUATION
# ============================================================

exact_results10k = []

for k, pool in exact_candidate_pools10k.items():

    print("\n" + "=" * 70)
    print(f"K = {k}")

    pool_keys = pd.MultiIndex.from_frame(
        pool[
            [
                "s1_rid",
                "cand_rid",
            ]
        ]
    )

    feature_keys = pd.MultiIndex.from_frame(
        features_leakfree10k[
            [
                "s1_rid",
                "cand_rid",
            ]
        ]
    )

    # Select feature rows belonging to the exact production pool.
    mask = feature_keys.isin(pool_keys)

    exact_features = (
        features_leakfree10k.loc[mask]
        .copy()
    )

    # Keep deterministic ordering.
    exact_features = (
        exact_features
        .merge(
            pool[
                [
                    "s1_rid",
                    "cand_rid",
                ]
            ],
            on=[
                "s1_rid",
                "cand_rid",
            ],
            how="inner",
        )
        .drop_duplicates(
            [
                "s1_rid",
                "cand_rid",
            ]
        )
    )

    print(
        "Feature rows:",
        len(exact_features)
    )

    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    X_exact = exact_features[
        ALL_FEATURES
    ]

    exact_prob = model_b_lf10k.predict_proba(
        X_exact
    )[:, 1]

    exact_predictions = exact_features[
        [
            "s1_rid",
            "cand_rid",
        ]
    ].copy()

    exact_predictions["prob"] = exact_prob

    # --------------------------------------------------------
    # Evaluate thresholds
    # --------------------------------------------------------

    # Use the already selected leakage-free threshold.
    threshold = float(
        best_lf["threshold"]
    )

    diag = correct_entity_f05(
        valid_s1,
        exact_predictions,
        FULL_GT_RID10K,
        threshold,
    )

    # --------------------------------------------------------
    # Candidate recall
    # --------------------------------------------------------

    candidate_recall_diag = []

    for s1_rid in valid_s1:

        true_set = FULL_GT_RID10K.get(
            int(s1_rid),
            set(),
        )

        available = set(
            exact_predictions.loc[
                exact_predictions["s1_rid"]
                == int(s1_rid),
                "cand_rid",
            ].astype(int)
        )

        if true_set:

            recall = len(
                true_set & available
            ) / len(true_set)

        else:

            recall = 1.0

        candidate_recall_diag.append(
            recall
        )

    exact_results10k.append(
        {
            "K": k,
            "candidate_rows": len(exact_predictions),
            "avg_candidates": (
                len(exact_predictions)
                / len(valid_s1)
            ),
            "candidate_recall": np.mean(
                candidate_recall_diag
            ),
            "macro_f05": diag["f05"].mean(),
            "tp": diag["tp"].sum(),
            "fp": diag["fp"].sum(),
            "fn": diag["fn"].sum(),
            "avg_predictions": diag[
                "pred_count"
            ].mean(),
        }
    )

exact_results10k = pd.DataFrame(
    exact_results10k
)

display(exact_results10k)


K = 50
Feature rows: 489897

K = 75
Feature rows: 715413

K = 100
Feature rows: 715518

K = 150
Feature rows: 715634


,K,candidate_rows,avg_candidates,candidate_recall,macro_f05,tp,fp,fn,avg_predictions
0,50,489897,244.9485,0.978825,0.982221,6765,49,240,3.4070
1,75,715413,357.7065,0.983543,0.983385,6794,49,211,3.4215
2,100,715518,357.7590,0.986389,0.984226,6810,49,195,3.4295
3,150,715634,357.8170,0.990593,0.985722,6836,49,169,3.4425


In [96]:
# ============================================================
# 31A. COMPLETE FEATURES FOR K100 / K150
# ============================================================

def compute_feature_matrix_for_pairs(
    pair_df,
    include_tfidf=True,
):
    """
    Compute the same Model-B 35-feature representation
    for an arbitrary candidate-pair DataFrame.
    """

    s1_name_native = S1.set_index("s1_rid")["name_native"].to_dict()
    s1_name_ascii = S1.set_index("s1_rid")["name_ascii"].to_dict()
    s1_addr_native = S1.set_index("s1_rid")["addr_native"].to_dict()
    s1_addr_ascii = S1.set_index("s1_rid")["addr_ascii"].to_dict()
    s1_country = S1.set_index("s1_rid")["country"].to_dict()

    s1_name_tok_native = S1.set_index("s1_rid")["name_tokens_native"].to_dict()
    s1_name_tok_ascii = S1.set_index("s1_rid")["name_tokens_ascii"].to_dict()
    s1_addr_tok_native = S1.set_index("s1_rid")["addr_tokens_native"].to_dict()
    s1_addr_tok_ascii = S1.set_index("s1_rid")["addr_tokens_ascii"].to_dict()
    s1_numbers = S1.set_index("s1_rid")["numbers_native"].to_dict()
    s1_postal = S1.set_index("s1_rid")["postal_native"].to_dict()

    cand_name_native = CAND.set_index("cand_rid")["name_native"].to_dict()
    cand_name_ascii = CAND.set_index("cand_rid")["name_ascii"].to_dict()
    cand_addr_native = CAND.set_index("cand_rid")["addr_native"].to_dict()
    cand_addr_ascii = CAND.set_index("cand_rid")["addr_ascii"].to_dict()
    cand_country = CAND.set_index("cand_rid")["country"].to_dict()

    cand_name_tok_native = CAND.set_index("cand_rid")["name_tokens_native"].to_dict()
    cand_name_tok_ascii = CAND.set_index("cand_rid")["name_tokens_ascii"].to_dict()
    cand_addr_tok_native = CAND.set_index("cand_rid")["addr_tokens_native"].to_dict()
    cand_addr_tok_ascii = CAND.set_index("cand_rid")["addr_tokens_ascii"].to_dict()
    cand_numbers = CAND.set_index("cand_rid")["numbers_native"].to_dict()
    cand_postal = CAND.set_index("cand_rid")["postal_native"].to_dict()

    rows = []

    t0 = time.time()

    for i, r in enumerate(
        pair_df.itertuples(index=False),
        start=1,
    ):
        s1_rid = int(r.s1_rid)
        cand_rid = int(r.cand_rid)

        nn = s1_name_native[s1_rid]
        cn = cand_name_native[cand_rid]

        na = s1_name_ascii[s1_rid]
        ca = cand_name_ascii[cand_rid]

        an = s1_addr_native[s1_rid]
        ac = cand_addr_native[cand_rid]

        aa = s1_addr_ascii[s1_rid]
        ad = cand_addr_ascii[cand_rid]

        name_ratio_native = safe_ratio(nn, cn)
        name_ratio_ascii = safe_ratio(na, ca)

        name_token_sort = safe_token_sort_ratio(na, ca)
        name_token_set = safe_token_set_ratio(na, ca)

        name_jaccard_native = safe_jaccard(
            s1_name_tok_native[s1_rid],
            cand_name_tok_native[cand_rid],
        )

        name_jaccard_ascii = safe_jaccard(
            s1_name_tok_ascii[s1_rid],
            cand_name_tok_ascii[cand_rid],
        )

        name_shared_tokens = overlap_count(
            s1_name_tok_ascii[s1_rid],
            cand_name_tok_ascii[cand_rid],
        )

        address_ratio_native = safe_ratio(an, ac)
        address_ratio_ascii = safe_ratio(aa, ad)

        address_token_sort = safe_token_sort_ratio(aa, ad)
        address_token_set = safe_token_set_ratio(aa, ad)

        address_jaccard_native = safe_jaccard(
            s1_addr_tok_native[s1_rid],
            cand_addr_tok_native[cand_rid],
        )

        address_jaccard_ascii = safe_jaccard(
            s1_addr_tok_ascii[s1_rid],
            cand_addr_tok_ascii[cand_rid],
        )

        address_shared_tokens = overlap_count(
            s1_addr_tok_ascii[s1_rid],
            cand_addr_tok_ascii[cand_rid],
        )

        number_overlap = safe_jaccard(
            s1_numbers[s1_rid],
            cand_numbers[cand_rid],
        )

        number_shared_count = overlap_count(
            s1_numbers[s1_rid],
            cand_numbers[cand_rid],
        )

        postal_match = int(
            bool(
                set(s1_postal[s1_rid] or [])
                &
                set(cand_postal[cand_rid] or [])
            )
        )

        name_exact_native = int(
            bool(nn and cn and nn == cn)
        )

        name_exact_ascii = int(
            bool(na and ca and na == ca)
        )

        address_exact_native = int(
            bool(an and ac and an == ac)
        )

        address_exact_ascii = int(
            bool(aa and ad and aa == ad)
        )

        country_match = int(
            bool(
                s1_country[s1_rid]
                and s1_country[s1_rid]
                == cand_country[cand_rid]
            )
        )

        name_missing = int(not na)
        candidate_name_missing = int(not ca)

        address_missing = int(not aa)
        candidate_address_missing = int(not ad)

        name_len_s1 = len(na)
        name_len_cand = len(ca)

        address_len_s1 = len(aa)
        address_len_cand = len(ad)

        name_len_ratio = (
            min(name_len_s1, name_len_cand)
            / max(name_len_s1, name_len_cand)
            if max(name_len_s1, name_len_cand) > 0
            else 0.0
        )

        address_len_ratio = (
            min(address_len_s1, address_len_cand)
            / max(address_len_s1, address_len_cand)
            if max(address_len_s1, address_len_cand) > 0
            else 0.0
        )

        name_addr_ratio_mean = (
            name_ratio_ascii
            + address_ratio_ascii
        ) / 2.0

        max_field_ratio = max(
            name_ratio_ascii,
            address_ratio_ascii,
        )

        rows.append(
            {
                "s1_rid": s1_rid,
                "cand_rid": cand_rid,
                "block_score": float(r.block_score),

                "name_ratio_native": name_ratio_native,
                "name_ratio_ascii": name_ratio_ascii,
                "name_token_sort": name_token_sort,
                "name_token_set": name_token_set,
                "name_jaccard_native": name_jaccard_native,
                "name_jaccard_ascii": name_jaccard_ascii,
                "name_shared_tokens": name_shared_tokens,
                "name_exact_native": name_exact_native,
                "name_exact_ascii": name_exact_ascii,

                "address_ratio_native": address_ratio_native,
                "address_ratio_ascii": address_ratio_ascii,
                "address_token_sort": address_token_sort,
                "address_token_set": address_token_set,
                "address_jaccard_native": address_jaccard_native,
                "address_jaccard_ascii": address_jaccard_ascii,
                "address_shared_tokens": address_shared_tokens,
                "address_exact_native": address_exact_native,
                "address_exact_ascii": address_exact_ascii,

                "number_overlap": number_overlap,
                "number_shared_count": number_shared_count,
                "postal_match": postal_match,

                "country_match": country_match,

                "name_missing": name_missing,
                "candidate_name_missing": candidate_name_missing,
                "address_missing": address_missing,
                "candidate_address_missing": candidate_address_missing,

                "name_len_ratio": name_len_ratio,
                "address_len_ratio": address_len_ratio,

                "name_addr_ratio_mean": name_addr_ratio_mean,
                "max_field_ratio": max_field_ratio,
            }
        )

        if i % 100_000 == 0:
            print(
                f"{i:,}/{len(pair_df):,} "
                f"pairs | {time.time() - t0:.1f}s"
            )

    out = pd.DataFrame(rows)

    if not include_tfidf:
        return out

    # --------------------------------------------------------
    # TF-IDF cosine
    # using the TRAIN-ONLY vectorizers
    # --------------------------------------------------------

    left_idx = np.array(
        [
            s1_pos[int(x)]
            for x in out["s1_rid"]
        ],
        dtype=np.int32,
    )

    right_idx = np.array(
        [
            cand_pos[int(x)]
            for x in out["cand_rid"]
        ],
        dtype=np.int32,
    )

    # Transform complete S1/CAND corpora with the
    # already train-fitted vectorizers.
    S1_name_char = leakfree_name_char_vec.transform(
        S1["name_ascii"].fillna("")
    )
    CAND_name_char = leakfree_name_char_vec.transform(
        CAND["name_ascii"].fillna("")
    )

    S1_name_word = leakfree_name_word_vec.transform(
        S1["name_ascii"].fillna("")
    )
    CAND_name_word = leakfree_name_word_vec.transform(
        CAND["name_ascii"].fillna("")
    )

    S1_addr_char = leakfree_addr_char_vec.transform(
        S1["addr_ascii"].fillna("")
    )
    CAND_addr_char = leakfree_addr_char_vec.transform(
        CAND["addr_ascii"].fillna("")
    )

    S1_addr_word = leakfree_addr_word_vec.transform(
        S1["addr_ascii"].fillna("")
    )
    CAND_addr_word = leakfree_addr_word_vec.transform(
        CAND["addr_ascii"].fillna("")
    )

    out["name_tfidf_char_cosine"] = pairwise_sparse_cosine(
        S1_name_char,
        CAND_name_char,
        left_idx,
        right_idx,
    )

    out["name_tfidf_word_cosine"] = pairwise_sparse_cosine(
        S1_name_word,
        CAND_name_word,
        left_idx,
        right_idx,
    )

    out["address_tfidf_char_cosine"] = pairwise_sparse_cosine(
        S1_addr_char,
        CAND_addr_char,
        left_idx,
        right_idx,
    )

    out["address_tfidf_word_cosine"] = pairwise_sparse_cosine(
        S1_addr_word,
        CAND_addr_word,
        left_idx,
        right_idx,
    )

    return out

In [97]:
# ============================================================
# 31B. GENERATE COMPLETE K100/K150 FEATURES
# ============================================================

exact_feature_pools10k = {}

for k in [100, 150]:

    print("\n" + "=" * 80)
    print(f"Generating COMPLETE features for K={k}")

    pool = exact_candidate_pools10k[k].copy()

    # Restrict to validation S1 only for this evaluation.
    pool = pool[
        pool["s1_rid"].isin(valid_s1)
    ].reset_index(drop=True)

    print("Pairs to process:", len(pool))

    exact_feature_pools10k[k] = (
        compute_feature_matrix_for_pairs(
            pool,
            include_tfidf=True,
        )
    )

    print(
        "Completed K=",
        k,
        "| rows:",
        len(exact_feature_pools10k[k])
    )


Generating COMPLETE features for K=100
Pairs to process: 183525
100,000/183,525 pairs | 1.8s
Completed K= 100 | rows: 183525

Generating COMPLETE features for K=150
Pairs to process: 247144
100,000/247,144 pairs | 1.6s
200,000/247,144 pairs | 3.2s
Completed K= 150 | rows: 247144


In [100]:
# ============================================================
# 31C. CORRECT END-TO-END K SWEEP
# ============================================================

K_RESULTS = []

THRESHOLD_PROD = float(best_lf["threshold"])


def evaluate_exact_k(
    k,
    feature_df,
    candidate_pool_df,
):
    """
    Evaluate an exact production candidate pool.

    feature_df:
        Complete feature rows for exactly those candidates.

    candidate_pool_df:
        Exact blocker output for the same S1s/candidates.
    """

    # --------------------------------------------------------
    # Safety check: feature rows must exactly match candidate
    # rows for this K.
    # --------------------------------------------------------

    feature_keys = set(
        zip(
            feature_df["s1_rid"].astype(int),
            feature_df["cand_rid"].astype(int),
        )
    )

    pool_keys = set(
        zip(
            candidate_pool_df["s1_rid"].astype(int),
            candidate_pool_df["cand_rid"].astype(int),
        )
    )

    missing_features = pool_keys - feature_keys
    extra_features = feature_keys - pool_keys

    if missing_features:
        raise ValueError(
            f"K={k}: {len(missing_features)} "
            "candidate pairs have no features."
        )

    if extra_features:
        raise ValueError(
            f"K={k}: {len(extra_features)} "
            "feature rows are not in candidate pool."
        )

    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    probs = model_b_lf10k.predict_proba(
        feature_df[ALL_FEATURES]
    )[:, 1]

    predictions = feature_df[
        ["s1_rid", "cand_rid"]
    ].copy()

    predictions["prob"] = probs

    # --------------------------------------------------------
    # Model evaluation
    # --------------------------------------------------------

    diag = correct_entity_f05(
        valid_s1,
        predictions,
        FULL_GT_RID10K,
        THRESHOLD_PROD,
    )

    # --------------------------------------------------------
    # Candidate recall
    # --------------------------------------------------------

    candidate_sets = {}

    for s1_rid, g in candidate_pool_df.groupby(
        "s1_rid"
    ):
        candidate_sets[int(s1_rid)] = set(
            g["cand_rid"].astype(int)
        )

    recall_values = []

    for s1_rid in valid_s1:

        s1_rid = int(s1_rid)

        truth = FULL_GT_RID10K.get(
            s1_rid,
            set(),
        )

        available = candidate_sets.get(
            s1_rid,
            set(),
        )

        if len(truth) == 0:
            # Correct singleton handling.
            recall = 1.0
        else:
            recall = len(
                truth & available
            ) / len(truth)

        recall_values.append(recall)

    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    return {
        "K": k,
        "candidate_rows": len(candidate_pool_df),
        "avg_candidates": (
            len(candidate_pool_df)
            / len(valid_s1)
        ),
        "candidate_recall": np.mean(
            recall_values
        ),
        "macro_f05": diag["f05"].mean(),
        "tp": int(diag["tp"].sum()),
        "fp": int(diag["fp"].sum()),
        "fn": int(diag["fn"].sum()),
        "avg_predictions": diag[
            "pred_count"
        ].mean(),
    }


# ============================================================
# K = 50 and 75
# ============================================================

for k in [50, 75]:

    print("\n" + "=" * 70)
    print(f"Evaluating K={k}")

    pool = exact_candidate_pools10k[k].copy()

    pool = pool[
        pool["s1_rid"].isin(valid_s1)
    ].copy()

    # The leakage-free feature matrix contains every
    # actual Top-75 candidate, so K50/K75 are already complete.
    feature_df = features_leakfree10k[
        features_leakfree10k["s1_rid"].isin(valid_s1)
    ].copy()

    pool_keys = set(
        zip(
            pool["s1_rid"].astype(int),
            pool["cand_rid"].astype(int),
        )
    )

    feature_keys = set(
        zip(
            feature_df["s1_rid"].astype(int),
            feature_df["cand_rid"].astype(int),
        )
    )

    selected_keys = pool_keys & feature_keys

    mask = [
        key in selected_keys
        for key in zip(
            feature_df["s1_rid"].astype(int),
            feature_df["cand_rid"].astype(int),
        )
    ]

    feature_df = feature_df.loc[
        mask
    ].copy()

    result = evaluate_exact_k(
        k,
        feature_df,
        pool,
    )

    K_RESULTS.append(result)

    print(
        f"K={k} | "
        f"rows={result['candidate_rows']:,} | "
        f"recall={result['candidate_recall']:.6f} | "
        f"F0.5={result['macro_f05']:.6f}"
    )


# ============================================================
# K = 100 and 150
# ============================================================

for k in [100, 150]:

    print("\n" + "=" * 70)
    print(f"Evaluating K={k}")

    pool = exact_candidate_pools10k[k].copy()

    pool = pool[
        pool["s1_rid"].isin(valid_s1)
    ].copy()

    feature_df = exact_feature_pools10k[k].copy()

    result = evaluate_exact_k(
        k,
        feature_df,
        pool,
    )

    K_RESULTS.append(result)

    print(
        f"K={k} | "
        f"rows={result['candidate_rows']:,} | "
        f"recall={result['candidate_recall']:.6f} | "
        f"F0.5={result['macro_f05']:.6f}"
    )


# ============================================================
# FINAL TABLE
# ============================================================

correct_k_results10k = (
    pd.DataFrame(K_RESULTS)
    .sort_values("K")
    .reset_index(drop=True)
)

print("\n=== CORRECT END-TO-END K SWEEP ===")

display(
    correct_k_results10k
)


Evaluating K=50
K=50 | rows=97,854 | recall=0.978825 | F0.5=0.982221

Evaluating K=75
K=75 | rows=142,993 | recall=0.983543 | F0.5=0.983385

Evaluating K=100
K=100 | rows=183,525 | recall=0.986389 | F0.5=0.984121

Evaluating K=150
K=150 | rows=247,144 | recall=0.990593 | F0.5=0.985343

=== CORRECT END-TO-END K SWEEP ===


,K,candidate_rows,avg_candidates,candidate_recall,macro_f05,tp,fp,fn,avg_predictions
0,50,97854,48.9270,0.978825,0.982221,6765,49,240,3.4070
1,75,142993,71.4965,0.983543,0.983385,6794,49,211,3.4215
2,100,183525,91.7625,0.986389,0.984121,6810,50,195,3.4300
3,150,247144,123.5720,0.990593,0.985343,6836,53,169,3.4445


In [101]:
# ============================================================
# 32A. EXACT K PREDICTIONS FOR TUNE + HOLDOUT
# ============================================================

exact_predictions_by_k10k = {}

for k in [50, 75, 100, 150]:

    print("\n" + "=" * 70)
    print(f"Scoring exact K={k}")

    # K <= 75: existing leakage-free complete feature matrix
    if k <= 75:

        pool = exact_candidate_pools10k[k].copy()

        pool = pool[
            pool["s1_rid"].isin(valid_s1)
        ].copy()

        pool_keys = set(
            zip(
                pool["s1_rid"].astype(int),
                pool["cand_rid"].astype(int),
            )
        )

        source = features_leakfree10k.copy()

        source_keys = zip(
            source["s1_rid"].astype(int),
            source["cand_rid"].astype(int),
        )

        mask = [
            key in pool_keys
            for key in source_keys
        ]

        feature_df = source.loc[
            mask
        ].copy()

    # K > 75: complete feature matrices generated above
    else:
        feature_df = exact_feature_pools10k[k].copy()

    # Safety check
    expected_pairs = set(
        zip(
            (
                exact_candidate_pools10k[k]
                .loc[
                    exact_candidate_pools10k[k]["s1_rid"]
                    .isin(valid_s1),
                    "s1_rid",
                ]
                .astype(int)
            ),
            (
                exact_candidate_pools10k[k]
                .loc[
                    exact_candidate_pools10k[k]["s1_rid"]
                    .isin(valid_s1),
                    "cand_rid",
                ]
                .astype(int)
            ),
        )
    )

    actual_pairs = set(
        zip(
            feature_df["s1_rid"].astype(int),
            feature_df["cand_rid"].astype(int),
        )
    )

    assert expected_pairs == actual_pairs, (
        f"K={k}: feature/candidate mismatch"
    )

    probabilities = model_b_lf10k.predict_proba(
        feature_df[ALL_FEATURES]
    )[:, 1]

    pred = feature_df[
        ["s1_rid", "cand_rid"]
    ].copy()

    pred["prob"] = probabilities

    exact_predictions_by_k10k[k] = pred

    print(
        f"K={k}: {len(pred):,} predictions"
    )


Scoring exact K=50
K=50: 97,854 predictions

Scoring exact K=75
K=75: 142,993 predictions

Scoring exact K=100
K=100: 183,525 predictions

Scoring exact K=150
K=150: 247,144 predictions


In [102]:
# ============================================================
# 32B. THRESHOLD TUNING PER CANDIDATE BUDGET
# ============================================================

K_THRESHOLD_RESULTS = []

threshold_grid_k = np.arange(
    0.30,
    0.951,
    0.01,
)

for k in [50, 75, 100, 150]:

    predictions = exact_predictions_by_k10k[k]

    tune_predictions = predictions[
        predictions["s1_rid"].isin(tune_s1)
    ].copy()

    tune_sweep = sweep_correct_f05(
        tune_s1,
        tune_predictions,
        FULL_GT_RID10K,
        threshold_grid_k,
    )

    best = tune_sweep.iloc[0]

    K_THRESHOLD_RESULTS.append(
        {
            "K": k,
            "best_threshold": float(
                best["threshold"]
            ),
            "tune_f05": float(
                best["macro_f05"]
            ),
            "tune_tp": int(best["tp"]),
            "tune_fp": int(best["fp"]),
            "tune_fn": int(best["fn"]),
        }
    )

k_threshold_results10k = (
    pd.DataFrame(K_THRESHOLD_RESULTS)
    .sort_values("K")
    .reset_index(drop=True)
)

print("=== THRESHOLD OPTIMIZATION BY K ===")
display(k_threshold_results10k)

=== THRESHOLD OPTIMIZATION BY K ===


,K,best_threshold,tune_f05,tune_tp,tune_fp,tune_fn
0,50,0.69,0.984504,3350,22,119
1,75,0.69,0.985965,3366,22,103
2,100,0.69,0.986491,3373,23,96
3,150,0.69,0.987567,3385,23,84


In [103]:
# ============================================================
# 32C. CLEAN K COMPARISON ON THE UNTOUCHED HOLDOUT
# ============================================================

K_HOLDOUT_RESULTS = []

for _, r in k_threshold_results10k.iterrows():

    k = int(r["K"])
    threshold = float(r["best_threshold"])

    predictions = exact_predictions_by_k10k[k]

    holdout_predictions = predictions[
        predictions["s1_rid"].isin(holdout_s1)
    ].copy()

    diag = correct_entity_f05(
        holdout_s1,
        holdout_predictions,
        FULL_GT_RID10K,
        threshold,
    )

    K_HOLDOUT_RESULTS.append(
        {
            "K": k,
            "threshold": threshold,
            "holdout_f05": diag["f05"].mean(),
            "tp": int(diag["tp"].sum()),
            "fp": int(diag["fp"].sum()),
            "fn": int(diag["fn"].sum()),
            "avg_predictions": diag[
                "pred_count"
            ].mean(),
        }
    )

k_holdout_results10k = (
    pd.DataFrame(K_HOLDOUT_RESULTS)
    .sort_values("K")
    .reset_index(drop=True)
)

print("=== CLEAN END-TO-END K HOLDOUT ===")
display(k_holdout_results10k)

=== CLEAN END-TO-END K HOLDOUT ===


,K,threshold,holdout_f05,tp,fp,fn,avg_predictions
0,50,0.69,0.979889,3414,26,122,3.440
1,75,0.69,0.980665,3426,26,110,3.452
2,100,0.69,0.981610,3435,26,101,3.461
3,150,0.69,0.982979,3449,29,87,3.478


In [105]:
# ============================================================
# 33A. WHO ACTUALLY BENEFITS FROM K150?
# ============================================================

ADAPTIVE_THRESHOLD = 0.69

p100 = exact_predictions_by_k10k[100].copy()
p150 = exact_predictions_by_k10k[150].copy()

p100["selected"] = (
    p100["prob"] >= ADAPTIVE_THRESHOLD
)

p150["selected"] = (
    p150["prob"] >= ADAPTIVE_THRESHOLD
)

adaptive_diag_rows = []

for s1_rid in valid_s1:

    s1_rid = int(s1_rid)

    truth = FULL_GT_RID10K.get(
        s1_rid,
        set(),
    )

    g100 = p100[
        p100["s1_rid"] == s1_rid
    ]

    g150 = p150[
        p150["s1_rid"] == s1_rid
    ]

    pred100 = set(
        g100.loc[
            g100["selected"],
            "cand_rid"
        ].astype(int)
    )

    pred150 = set(
        g150.loc[
            g150["selected"],
            "cand_rid"
        ].astype(int)
    )

    extra_truth_101_150 = (
        truth & set(g150["cand_rid"].astype(int))
    ) - set(g100["cand_rid"].astype(int))

    extra_tp = len(
        extra_truth_101_150
    )

    extra_predictions = (
        pred150 - pred100
    )

    extra_fp = len(
        extra_predictions - truth
    )

    probs100 = g100["prob"].to_numpy()

    predicted_probs100 = g100.loc[
        g100["selected"],
        "prob"
    ].to_numpy()

    adaptive_diag_rows.append(
        {
            "s1_rid": s1_rid,

            "true_count": len(truth),

            "pred100_count": len(pred100),
            "pred150_count": len(pred150),

            "top1_prob100": (
                float(probs100.max())
                if len(probs100)
                else 0.0
            ),

            "min_selected_prob100": (
                float(predicted_probs100.min())
                if len(predicted_probs100)
                else 0.0
            ),

            "extra_tp_101_150": extra_tp,
            "extra_fp_101_150": extra_fp,

            "benefits_from_k150": (
                extra_tp > 0
            ),
        }
    )

adaptive_diag10k = pd.DataFrame(
    adaptive_diag_rows
)

print("=== K150 BENEFIT ANALYSIS ===")

print(
    "S1s benefiting from K150:",
    adaptive_diag10k[
        "benefits_from_k150"
    ].sum()
)

print(
    "Total additional TP:",
    adaptive_diag10k[
        "extra_tp_101_150"
    ].sum()
)

print(
    "Total additional FP:",
    adaptive_diag10k[
        "extra_fp_101_150"
    ].sum()
)

print()

display(
    adaptive_diag10k[
        [
            "top1_prob100",
            "pred100_count",
            "min_selected_prob100",
            "extra_tp_101_150",
            "extra_fp_101_150",
        ]
    ].describe()
)

print()
print("S1s benefiting from K150:")

display(
    adaptive_diag10k.loc[
        adaptive_diag10k[
            "benefits_from_k150"
        ]
    ]
    .sort_values(
        "extra_tp_101_150",
        ascending=False,
    )
)

=== K150 BENEFIT ANALYSIS ===
S1s benefiting from K150: 25
Total additional TP: 30
Total additional FP: 3



,top1_prob100,pred100_count,min_selected_prob100,extra_tp_101_150,extra_fp_101_150
count,2000.000000,2000.000000,2000.000000,2000.0000,2000.00000
mean,0.946988,3.428500,0.924325,0.0150,0.00150
std,0.217057,1.704212,0.228863,0.1476,0.04999
min,0.000023,0.000000,0.000000,0.0000,0.00000
25%,0.999905,2.000000,0.980529,0.0000,0.00000
50%,0.999982,3.000000,0.999000,0.0000,0.00000
75%,0.999995,5.000000,0.999831,0.0000,0.00000
max,1.000000,10.000000,0.999998,3.0000,2.00000



S1s benefiting from K150:


,s1_rid,true_count,pred100_count,pred150_count,top1_prob100,min_selected_prob100,extra_tp_101_150,extra_fp_101_150,benefits_from_k150
719,9587,7,4,6,0.999985,0.913133,3,0,True
1336,7537,3,0,3,0.000401,0.000000,3,0,True
531,7684,7,5,7,0.999858,0.999348,2,0,True
81,952,4,3,4,0.999952,0.999140,1,0,True
1122,5731,3,2,3,0.999880,0.999655,1,0,True
1902,7691,4,3,4,0.999994,0.999878,1,0,True
1832,3520,2,1,2,0.999994,0.999994,1,0,True
1549,5447,5,4,5,0.999994,0.999899,1,0,True
1459,8605,7,6,7,0.999895,0.976508,1,0,True
1417,4549,5,3,4,0.998615,0.997956,1,0,True


In [106]:
# ============================================================
# 33B. ADAPTIVE K100 -> K150 RULE SWEEP
# ============================================================

# Create lookup tables.
p100_by_s1 = {
    int(s1): g.copy()
    for s1, g in p100.groupby("s1_rid")
}

p150_by_s1 = {
    int(s1): g.copy()
    for s1, g in p150.groupby("s1_rid")
}


def evaluate_adaptive_rule(
    predictions100,
    predictions150,
    valid_s1_ids,
    condition,
    threshold=0.69,
):
    rows = []

    total_candidates = 0

    for s1_rid in valid_s1_ids:

        s1_rid = int(s1_rid)

        g100 = predictions100.loc[
            predictions100["s1_rid"] == s1_rid
        ]

        g150 = predictions150.loc[
            predictions150["s1_rid"] == s1_rid
        ]

        use150 = condition(g100)

        selected_source = (
            g150
            if use150
            else g100
        )

        pred = set(
            selected_source.loc[
                selected_source["prob"]
                >= threshold,
                "cand_rid",
            ].astype(int)
        )

        truth = FULL_GT_RID10K.get(
            s1_rid,
            set(),
        )

        tp = len(pred & truth)
        fp = len(pred - truth)
        fn = len(truth - pred)

        if len(truth) == 0:

            f05 = (
                1.0
                if len(pred) == 0
                else 0.0
            )

        else:

            precision = (
                tp / (tp + fp)
                if tp + fp > 0
                else 0.0
            )

            recall = (
                tp / len(truth)
            )

            f05 = (
                1.25
                * precision
                * recall
                / (
                    0.25 * precision
                    + recall
                )
                if precision > 0
                else 0.0
            )

        total_candidates += (
            150 if use150 else 100
        )

        rows.append(
            {
                "s1_rid": s1_rid,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "f05": f05,
                "used_k150": use150,
            }
        )

    result = pd.DataFrame(rows)

    return {
        "macro_f05": result["f05"].mean(),
        "tp": int(result["tp"].sum()),
        "fp": int(result["fp"].sum()),
        "fn": int(result["fn"].sum()),
        "avg_candidates": (
            total_candidates
            / len(valid_s1_ids)
        ),
        "pct_s1_using_k150": (
            result["used_k150"].mean()
        ),
    }


adaptive_results = []


# ------------------------------------------------------------
# Rule 1: expand when top1 probability is low
# ------------------------------------------------------------

for t in [
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]:

    def rule(g, t=t):
        return (
            g["prob"].max() < t
        )

    # Tune
    tune_score = evaluate_adaptive_rule(
        p100,
        p150,
        tune_s1,
        rule,
        ADAPTIVE_THRESHOLD,
    )

    # Holdout
    holdout_score = evaluate_adaptive_rule(
        p100,
        p150,
        holdout_s1,
        rule,
        ADAPTIVE_THRESHOLD,
    )

    adaptive_results.append(
        {
            "rule": "top1_prob < threshold",
            "parameter": t,
            "tune_f05": tune_score[
                "macro_f05"
            ],
            "holdout_f05": holdout_score[
                "macro_f05"
            ],
            "holdout_tp": holdout_score["tp"],
            "holdout_fp": holdout_score["fp"],
            "holdout_fn": holdout_score["fn"],
            "avg_candidates": holdout_score[
                "avg_candidates"
            ],
            "pct_using_k150": holdout_score[
                "pct_s1_using_k150"
            ],
        }
    )


# ------------------------------------------------------------
# Rule 2: expand when K100 finds few matches
# ------------------------------------------------------------

for n in [
    0,
    1,
    2,
    3,
]:

    def rule(g, n=n):
        return (
            (g["prob"] >= ADAPTIVE_THRESHOLD).sum()
            <= n
        )

    tune_score = evaluate_adaptive_rule(
        p100,
        p150,
        tune_s1,
        rule,
        ADAPTIVE_THRESHOLD,
    )

    holdout_score = evaluate_adaptive_rule(
        p100,
        p150,
        holdout_s1,
        rule,
        ADAPTIVE_THRESHOLD,
    )

    adaptive_results.append(
        {
            "rule": "predicted_count <= n",
            "parameter": n,
            "tune_f05": tune_score[
                "macro_f05"
            ],
            "holdout_f05": holdout_score[
                "macro_f05"
            ],
            "holdout_tp": holdout_score["tp"],
            "holdout_fp": holdout_score["fp"],
            "holdout_fn": holdout_score["fn"],
            "avg_candidates": holdout_score[
                "avg_candidates"
            ],
            "pct_using_k150": holdout_score[
                "pct_s1_using_k150"
            ],
        }
    )


adaptive_results10k = (
    pd.DataFrame(adaptive_results)
    .sort_values(
        "holdout_f05",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(adaptive_results10k)

,rule,parameter,tune_f05,holdout_f05,holdout_tp,holdout_fp,holdout_fn,avg_candidates,pct_using_k150
0,predicted_count <= n,1.00,0.986825,0.982972,3440,26,96,106.10,0.122
1,predicted_count <= n,3.00,0.987487,0.982910,3445,28,91,125.85,0.517
2,predicted_count <= n,2.00,0.987292,0.982722,3442,28,94,114.05,0.281
3,top1_prob < threshold,0.70,0.986491,0.982610,3438,26,98,102.70,0.054
4,top1_prob < threshold,0.75,0.986491,0.982610,3438,26,98,102.75,0.055
5,top1_prob < threshold,0.80,0.986491,0.982610,3438,26,98,102.90,0.058
6,top1_prob < threshold,0.85,0.986491,0.982610,3438,26,98,103.00,0.060
7,top1_prob < threshold,0.90,0.986491,0.982610,3438,26,98,103.00,0.060
8,predicted_count <= n,0.00,0.986491,0.982610,3438,26,98,102.70,0.054


In [107]:
# ============================================================
# 34. JOINT THRESHOLD + ADAPTIVE RULE TEST
# ============================================================

JOINT_THRESHOLDS = np.arange(
    0.55,
    0.801,
    0.01,
)

JOINT_RESULTS = []


def evaluate_strategy(
    predictions100,
    predictions150,
    s1_ids,
    threshold,
    rule_name,
):
    rows = []
    total_candidates = 0

    for s1_rid in s1_ids:

        s1_rid = int(s1_rid)

        g100 = predictions100[
            predictions100["s1_rid"] == s1_rid
        ]

        g150 = predictions150[
            predictions150["s1_rid"] == s1_rid
        ]

        pred100 = set(
            g100.loc[
                g100["prob"] >= threshold,
                "cand_rid",
            ].astype(int)
        )

        # ----------------------------------------------------
        # Decide whether to expand
        # ----------------------------------------------------

        if rule_name == "K100":
            use150 = False

        elif rule_name == "K150":
            use150 = True

        elif rule_name == "adaptive_count_le_1":
            use150 = len(pred100) <= 1

        else:
            raise ValueError(rule_name)

        source = g150 if use150 else g100

        pred = set(
            source.loc[
                source["prob"] >= threshold,
                "cand_rid",
            ].astype(int)
        )

        truth = FULL_GT_RID10K.get(
            s1_rid,
            set(),
        )

        tp = len(pred & truth)
        fp = len(pred - truth)
        fn = len(truth - pred)

        if len(truth) == 0:

            f05 = (
                1.0
                if len(pred) == 0
                else 0.0
            )

        else:

            precision = (
                tp / (tp + fp)
                if tp + fp > 0
                else 0.0
            )

            recall = (
                tp / len(truth)
            )

            f05 = (
                1.25 * precision * recall
                / (
                    0.25 * precision
                    + recall
                )
                if precision > 0
                else 0.0
            )

        total_candidates += (
            150 if use150 else 100
        )

        rows.append(
            {
                "s1_rid": s1_rid,
                "f05": f05,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "used_k150": use150,
            }
        )

    result = pd.DataFrame(rows)

    return {
        "macro_f05": result["f05"].mean(),
        "tp": int(result["tp"].sum()),
        "fp": int(result["fp"].sum()),
        "fn": int(result["fn"].sum()),
        "avg_candidates": (
            total_candidates
            / len(s1_ids)
        ),
        "pct_k150": result[
            "used_k150"
        ].mean(),
    }


# ------------------------------------------------------------
# Tune strategies on TUNE set
# ------------------------------------------------------------

for strategy in [
    "K100",
    "K150",
    "adaptive_count_le_1",
]:

    for threshold in JOINT_THRESHOLDS:

        score = evaluate_strategy(
            p100,
            p150,
            tune_s1,
            threshold,
            strategy,
        )

        JOINT_RESULTS.append(
            {
                "strategy": strategy,
                "threshold": threshold,
                "tune_f05": score["macro_f05"],
                "tune_tp": score["tp"],
                "tune_fp": score["fp"],
                "tune_fn": score["fn"],
                "tune_avg_candidates": score[
                    "avg_candidates"
                ],
            }
        )


joint_tune10k = (
    pd.DataFrame(JOINT_RESULTS)
    .sort_values(
        "tune_f05",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("=== JOINT TUNE RESULTS ===")

display(
    joint_tune10k.head(15)
)


# ------------------------------------------------------------
# Evaluate ONLY the best configuration of each strategy
# ------------------------------------------------------------

best_configs = (
    joint_tune10k
    .groupby("strategy", as_index=False)
    .first()
)

JOINT_HOLDOUT = []

for _, cfg in best_configs.iterrows():

    strategy = cfg["strategy"]
    threshold = float(cfg["threshold"])

    score = evaluate_strategy(
        p100,
        p150,
        holdout_s1,
        threshold,
        strategy,
    )

    JOINT_HOLDOUT.append(
        {
            "strategy": strategy,
            "threshold": threshold,
            "holdout_f05": score["macro_f05"],
            "holdout_tp": score["tp"],
            "holdout_fp": score["fp"],
            "holdout_fn": score["fn"],
            "avg_candidates": score[
                "avg_candidates"
            ],
            "pct_k150": score["pct_k150"],
        }
    )


joint_holdout10k = (
    pd.DataFrame(JOINT_HOLDOUT)
    .sort_values(
        "holdout_f05",
        ascending=False,
    )
    .reset_index(drop=True)
)

print()
print("=== JOINT HOLDOUT RESULTS ===")

display(joint_holdout10k)

=== JOINT TUNE RESULTS ===


,strategy,threshold,tune_f05,tune_tp,tune_fp,tune_fn,tune_avg_candidates
0,K150,0.69,0.987567,3385,23,84,150.0
1,K150,0.68,0.987540,3386,24,83,150.0
2,K150,0.70,0.987353,3382,23,87,150.0
3,K150,0.71,0.987306,3381,23,88,150.0
4,K150,0.58,0.987250,3395,30,74,150.0
5,K150,0.72,0.987167,3379,23,90,150.0
6,K150,0.67,0.987166,3387,26,82,150.0
7,K150,0.79,0.987147,3363,19,106,150.0
8,K150,0.57,0.987116,3397,31,72,150.0
9,K150,0.73,0.987105,3378,23,91,150.0



=== JOINT HOLDOUT RESULTS ===


,strategy,threshold,holdout_f05,holdout_tp,holdout_fp,holdout_fn,avg_candidates,pct_k150
0,K150,0.69,0.982979,3449,29,87,150.0,1.000
1,adaptive_count_le_1,0.69,0.982972,3440,26,96,106.1,0.122
2,K100,0.69,0.981610,3435,26,101,100.0,0.000


In [108]:
# ============================================================
# 35A. PRODUCTION-DISTRIBUTION TRAINING POOL
# ============================================================

TRAIN_K = 150

train_candidate_pool10k = (
    tu_ranked[
        tu_ranked["s1_rid"].isin(train_s1)
    ]
    .groupby("s1_rid", sort=False)
    .head(TRAIN_K)
    [
        [
            "s1_rid",
            "cand_rid",
            "block_score",
        ]
    ]
    .drop_duplicates(
        ["s1_rid", "cand_rid"]
    )
    .reset_index(drop=True)
)

print("=== PRODUCTION TRAINING POOL ===")
print(
    "Rows:",
    len(train_candidate_pool10k)
)

print(
    "S1:",
    train_candidate_pool10k["s1_rid"].nunique()
)

print(
    "Avg candidates/S1:",
    train_candidate_pool10k.groupby(
        "s1_rid"
    ).size().mean()
)

=== PRODUCTION TRAINING POOL ===
Rows: 984167
S1: 8000
Avg candidates/S1: 123.020875


In [109]:
# ============================================================
# 35B. LABEL REAL TRAINING CANDIDATES
# ============================================================

train_candidate_pool10k["label"] = 0

train_candidate_pool10k["pair_key"] = list(
    zip(
        train_candidate_pool10k["s1_rid"].astype(int),
        train_candidate_pool10k["cand_rid"].astype(int),
    )
)

train_positive_keys = set(
    zip(
        positive_in_union10k[
            "s1_rid"
        ].astype(int),
        positive_in_union10k[
            "cand_rid"
        ].astype(int),
    )
)

train_candidate_pool10k["label"] = (
    train_candidate_pool10k["pair_key"]
    .isin(train_positive_keys)
    .astype(np.int8)
)

train_candidate_pool10k.drop(
    columns=["pair_key"],
    inplace=True,
)

print("=== TRAINING LABELS ===")
print(
    "Positive:",
    int(train_candidate_pool10k["label"].sum())
)

print(
    "Negative:",
    int(
        (train_candidate_pool10k["label"] == 0)
        .sum()
    )
)

print(
    "Positive rate:",
    train_candidate_pool10k["label"].mean()
)

=== TRAINING LABELS ===
Positive: 27467
Negative: 956700
Positive rate: 0.027908881317906413


In [110]:
# ============================================================
# 35C. COMPLETE MODEL-B FEATURES FOR REAL TRAINING POOL
# ============================================================

production_train_features10k = (
    compute_feature_matrix_for_pairs(
        train_candidate_pool10k[
            [
                "s1_rid",
                "cand_rid",
                "block_score",
            ]
        ],
        include_tfidf=True,
    )
)

# Add the actual labels.
production_train_features10k = (
    production_train_features10k
    .merge(
        train_candidate_pool10k[
            [
                "s1_rid",
                "cand_rid",
                "label",
            ]
        ],
        on=[
            "s1_rid",
            "cand_rid",
        ],
        how="left",
        validate="one_to_one",
    )
)

print("=== PRODUCTION TRAIN FEATURES ===")
print(
    "Rows:",
    len(production_train_features10k)
)

print(
    "Positive:",
    int(
        production_train_features10k[
            "label"
        ].sum()
    )
)

print(
    "Negative:",
    int(
        (
            production_train_features10k[
                "label"
            ] == 0
        ).sum()
    )
)

100,000/984,167 pairs | 1.7s
200,000/984,167 pairs | 3.4s
300,000/984,167 pairs | 5.1s
400,000/984,167 pairs | 6.8s
500,000/984,167 pairs | 8.3s
600,000/984,167 pairs | 9.8s
700,000/984,167 pairs | 11.3s
800,000/984,167 pairs | 12.8s
900,000/984,167 pairs | 14.3s
=== PRODUCTION TRAIN FEATURES ===
Rows: 984167
Positive: 27467
Negative: 956700


In [111]:
# ============================================================
# 35D. FINAL MODEL-B TRAINING
# ============================================================

X_prod_train = (
    production_train_features10k[
        ALL_FEATURES
    ]
)

y_prod_train = (
    production_train_features10k[
        "label"
    ]
)

print("Training production-distribution Model B...")

t0 = time.time()

final_model10k = CatBoostClassifier(
    iterations=600,
    depth=8,
    learning_rate=0.08,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False,
)

final_model10k.fit(
    X_prod_train,
    y_prod_train,
)

final_model_time = time.time() - t0

print()
print(
    "Training time:",
    round(final_model_time, 2),
    "sec"
)

print(
    "Final training rows:",
    len(X_prod_train)
)

Training production-distribution Model B...
0:	total: 95.6ms	remaining: 57.3s
100:	total: 6.53s	remaining: 32.3s
200:	total: 12.4s	remaining: 24.7s
300:	total: 18.3s	remaining: 18.2s
400:	total: 23.9s	remaining: 11.9s
500:	total: 28.9s	remaining: 5.71s
599:	total: 33.9s	remaining: 0us

Training time: 34.26 sec
Final training rows: 984167


In [112]:
# ============================================================
# 36A. SCORE K100 / K150 WITH FINAL PRODUCTION MODEL
# ============================================================

final_predictions_by_k10k = {}

for k in [100, 150]:

    print("\n" + "=" * 70)
    print(f"Scoring K={k} with FINAL MODEL")

    feature_df = exact_feature_pools10k[k].copy()

    probs = final_model10k.predict_proba(
        feature_df[ALL_FEATURES]
    )[:, 1]

    pred = feature_df[
        ["s1_rid", "cand_rid"]
    ].copy()

    pred["prob"] = probs

    final_predictions_by_k10k[k] = pred

    print(
        f"K={k}: {len(pred):,} scored pairs"
    )


Scoring K=100 with FINAL MODEL
K=100: 183,525 scored pairs

Scoring K=150 with FINAL MODEL
K=150: 247,144 scored pairs


In [113]:
# ============================================================
# 36B. FINAL MODEL — JOINT THRESHOLD TUNING
# ============================================================

FINAL_THRESHOLDS = np.arange(
    0.40,
    0.901,
    0.01,
)

FINAL_TUNE_RESULTS = []


def evaluate_final_strategy(
    pred100,
    pred150,
    s1_ids,
    threshold,
    strategy,
):
    rows = []
    total_candidates = 0

    for s1_rid in s1_ids:

        s1_rid = int(s1_rid)

        g100 = pred100[
            pred100["s1_rid"] == s1_rid
        ]

        g150 = pred150[
            pred150["s1_rid"] == s1_rid
        ]

        # ----------------------------------------------------
        # Decide candidate budget
        # ----------------------------------------------------

        if strategy == "K100":

            source = g100
            used_k150 = False

        elif strategy == "K150":

            source = g150
            used_k150 = True

        elif strategy == "adaptive_count_le_1":

            count100 = (
                g100["prob"] >= threshold
            ).sum()

            if count100 <= 1:
                source = g150
                used_k150 = True
            else:
                source = g100
                used_k150 = False

        else:
            raise ValueError(
                f"Unknown strategy: {strategy}"
            )

        # ----------------------------------------------------
        # Final predictions
        # ----------------------------------------------------

        pred = set(
            source.loc[
                source["prob"] >= threshold,
                "cand_rid",
            ].astype(int)
        )

        truth = FULL_GT_RID10K.get(
            s1_rid,
            set(),
        )

        tp = len(pred & truth)
        fp = len(pred - truth)
        fn = len(truth - pred)

        # ----------------------------------------------------
        # Correct singleton scoring
        # ----------------------------------------------------

        if len(truth) == 0:

            f05 = (
                1.0
                if len(pred) == 0
                else 0.0
            )

        else:

            precision = (
                tp / (tp + fp)
                if tp + fp > 0
                else 0.0
            )

            recall = (
                tp / len(truth)
            )

            f05 = (
                1.25 * precision * recall
                / (
                    0.25 * precision
                    + recall
                )
                if precision > 0
                else 0.0
            )

        total_candidates += (
            150 if used_k150 else 100
        )

        rows.append(
            {
                "s1_rid": s1_rid,
                "f05": f05,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "used_k150": used_k150,
            }
        )

    result = pd.DataFrame(rows)

    return {
        "macro_f05": result["f05"].mean(),
        "tp": int(result["tp"].sum()),
        "fp": int(result["fp"].sum()),
        "fn": int(result["fn"].sum()),
        "avg_candidates": (
            total_candidates
            / len(s1_ids)
        ),
        "pct_k150": result[
            "used_k150"
        ].mean(),
    }


pred100_final = final_predictions_by_k10k[100]
pred150_final = final_predictions_by_k10k[150]

for strategy in [
    "K100",
    "K150",
    "adaptive_count_le_1",
]:

    for threshold in FINAL_THRESHOLDS:

        result = evaluate_final_strategy(
            pred100_final,
            pred150_final,
            tune_s1,
            threshold,
            strategy,
        )

        FINAL_TUNE_RESULTS.append(
            {
                "strategy": strategy,
                "threshold": threshold,
                "tune_f05": result["macro_f05"],
                "tune_tp": result["tp"],
                "tune_fp": result["fp"],
                "tune_fn": result["fn"],
                "tune_avg_candidates": result[
                    "avg_candidates"
                ],
                "tune_pct_k150": result[
                    "pct_k150"
                ],
            }
        )

final_tune10k = (
    pd.DataFrame(FINAL_TUNE_RESULTS)
    .sort_values(
        "tune_f05",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("=== FINAL MODEL — TUNING ===")
display(
    final_tune10k.head(15)
)

=== FINAL MODEL — TUNING ===


,strategy,threshold,tune_f05,tune_tp,tune_fp,tune_fn,tune_avg_candidates,tune_pct_k150
0,K150,0.65,0.987974,3385,22,84,150.0,1.0
1,K150,0.75,0.987902,3372,17,97,150.0,1.0
2,K150,0.63,0.987894,3387,23,82,150.0,1.0
3,K150,0.67,0.987880,3382,21,87,150.0,1.0
4,K150,0.64,0.987846,3386,23,83,150.0,1.0
5,K150,0.66,0.987836,3383,22,86,150.0,1.0
6,K150,0.74,0.987760,3376,19,93,150.0,1.0
7,K150,0.72,0.987734,3378,20,91,150.0,1.0
8,K150,0.69,0.987726,3380,21,89,150.0,1.0
9,K150,0.68,0.987726,3380,21,89,150.0,1.0


In [114]:
# ============================================================
# 36C. FINAL 10K HOLDOUT
# ============================================================

best_final_configs = (
    final_tune10k
    .groupby("strategy", as_index=False)
    .first()
)

FINAL_HOLDOUT_RESULTS = []

for _, cfg in best_final_configs.iterrows():

    strategy = cfg["strategy"]
    threshold = float(
        cfg["threshold"]
    )

    result = evaluate_final_strategy(
        pred100_final,
        pred150_final,
        holdout_s1,
        threshold,
        strategy,
    )

    FINAL_HOLDOUT_RESULTS.append(
        {
            "strategy": strategy,
            "threshold": threshold,
            "holdout_f05": result[
                "macro_f05"
            ],
            "tp": result["tp"],
            "fp": result["fp"],
            "fn": result["fn"],
            "avg_candidates": result[
                "avg_candidates"
            ],
            "pct_k150": result[
                "pct_k150"
            ],
        }
    )

final_holdout10k = (
    pd.DataFrame(FINAL_HOLDOUT_RESULTS)
    .sort_values(
        "holdout_f05",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("=== FINAL PRODUCTION-MODEL HOLDOUT ===")
display(final_holdout10k)

=== FINAL PRODUCTION-MODEL HOLDOUT ===


,strategy,threshold,holdout_f05,tp,fp,fn,avg_candidates,pct_k150
0,K150,0.65,0.983183,3436,28,100,150.0,1.000
1,adaptive_count_le_1,0.65,0.983171,3429,26,107,106.1,0.122
2,K100,0.65,0.981809,3424,26,112,100.0,0.000


In [115]:
# ============================================================
# SAVE 10K EXPERIMENT CHECKPOINT
# ============================================================

from pathlib import Path
import json
import joblib

ARTIFACT_DIR = Path("amazon_ml_10k_artifacts")
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. CatBoost model
# ------------------------------------------------------------

final_model10k.save_model(
    ARTIFACT_DIR / "checkpoint_model_8k.cbm"
)

# ------------------------------------------------------------
# 2. TF-IDF vectorizers
# ------------------------------------------------------------

joblib.dump(
    leakfree_name_char_vec,
    ARTIFACT_DIR / "name_char_tfidf.joblib"
)

joblib.dump(
    leakfree_name_word_vec,
    ARTIFACT_DIR / "name_word_tfidf.joblib"
)

joblib.dump(
    leakfree_addr_char_vec,
    ARTIFACT_DIR / "address_char_tfidf.joblib"
)

joblib.dump(
    leakfree_addr_word_vec,
    ARTIFACT_DIR / "address_word_tfidf.joblib"
)

# ------------------------------------------------------------
# 3. Feature configuration
# ------------------------------------------------------------

with open(
    ARTIFACT_DIR / "feature_columns.json",
    "w"
) as f:
    json.dump(
        ALL_FEATURES,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 4. Model / candidate configuration
# ------------------------------------------------------------

config = {
    "model": "CatBoostClassifier",
    "model_checkpoint": "checkpoint_model_8k.cbm",
    "feature_count": len(ALL_FEATURES),

    "candidate_strategy": {
        "base_k": 100,
        "expansion_k": 150,
        "expansion_rule": "predicted_count <= 1",
    },

    "current_threshold": 0.65,

    "training_s1_count": len(train_s1),
    "tune_s1_count": len(tune_s1),
    "holdout_s1_count": len(holdout_s1),

    "note": (
        "Checkpoint only. "
        "Model was trained on 8,000 S1 production-distribution "
        "candidate pairs; planned 9,000-S1 retraining remains."
    ),
}

with open(
    ARTIFACT_DIR / "model_config.json",
    "w"
) as f:
    json.dump(
        config,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 5. Preserve exact S1 splits
# ------------------------------------------------------------

np.savetxt(
    ARTIFACT_DIR / "train_s1_ids.txt",
    np.asarray(train_s1),
    fmt="%d"
)

np.savetxt(
    ARTIFACT_DIR / "tune_s1_ids.txt",
    np.asarray(tune_s1),
    fmt="%d"
)

np.savetxt(
    ARTIFACT_DIR / "holdout_s1_ids.txt",
    np.asarray(holdout_s1),
    fmt="%d"
)

# ------------------------------------------------------------
# 6. Save benchmark tables
# ------------------------------------------------------------

correct_k_results10k.to_csv(
    ARTIFACT_DIR / "k_sweep_results.csv",
    index=False
)

final_holdout10k.to_csv(
    ARTIFACT_DIR / "final_holdout_results.csv",
    index=False
)

comparison_models10k.to_csv(
    ARTIFACT_DIR / "model_comparison.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Save the notebook itself
# ------------------------------------------------------------

print("Artifacts saved to:")
print(ARTIFACT_DIR.resolve())

print("\nFiles:")
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(
        f"{p.name:40s}",
        f"{p.stat().st_size / 1024**2:.2f} MB"
    )

Artifacts saved to:
/Users/atharv/Documents/AMAZON_ML/amazon_ml_10k_artifacts

Files:
address_char_tfidf.joblib                3.29 MB
address_word_tfidf.joblib                3.81 MB
checkpoint_model_8k.cbm                  2.42 MB
feature_columns.json                     0.00 MB
final_holdout_results.csv                0.00 MB
holdout_s1_ids.txt                       0.00 MB
k_sweep_results.csv                      0.00 MB
model_comparison.csv                     0.00 MB
model_config.json                        0.00 MB
name_char_tfidf.joblib                   1.62 MB
name_word_tfidf.joblib                   2.05 MB
train_s1_ids.txt                         0.04 MB
tune_s1_ids.txt                          0.00 MB
